In [2]:
#1
import os

base_paths = [
    "/kaggle/input/datasets/jasiajabeen1/nepal-dataset-solar",
    "/kaggle/input/datasets/jasiajabeen1/pakistan-dataset-forsolar",
    "/kaggle/input/datasets/jasiajabeen1/zambi-dataset"
]

for base_path in base_paths:
    print("\n" + "="*100)
    print(f"DATASET: {base_path}")
    print("="*100)
    
    if not os.path.exists(base_path):
        print("❌ Path does not exist")
        continue
    
    for root, dirs, files in os.walk(base_path):
        for file in files:
            full_path = os.path.join(root, file)
            print(full_path)


DATASET: /kaggle/input/datasets/jasiajabeen1/nepal-dataset-solar
/kaggle/input/datasets/jasiajabeen1/nepal-dataset-solar/solar-measurements_nepal_dharan_wb-esmap_header.xlsx
/kaggle/input/datasets/jasiajabeen1/nepal-dataset-solar/solar-measurements_nepal_nepalgunj_wb-esmap_qc.csv
/kaggle/input/datasets/jasiajabeen1/nepal-dataset-solar/solar-measurements_nepal_kathmandu_wb-esmap_header.xlsx
/kaggle/input/datasets/jasiajabeen1/nepal-dataset-solar/solar-measurements_nepal_jumla_wb-esmap_header.xlsx
/kaggle/input/datasets/jasiajabeen1/nepal-dataset-solar/solar-measurements_nepal_nepalgunj_wb-esmap_header.xlsx
/kaggle/input/datasets/jasiajabeen1/nepal-dataset-solar/solar-measurements_nepal_dharan_wb-esmap_qc.csv
/kaggle/input/datasets/jasiajabeen1/nepal-dataset-solar/solar-measurements_nepal_jumla_wb-esmap_qc.csv
/kaggle/input/datasets/jasiajabeen1/nepal-dataset-solar/solar-measurements_nepal_kathmandu_wb-esmap_qc.csv
/kaggle/input/datasets/jasiajabeen1/nepal-dataset-solar/solar-measuremen

In [3]:
# 2
# ============================================================================================
#  SolarQC-Net  |  PHASE 1  v2   -  REPLACES the earlier Phase 1 cell entirely
#
#  What changed vs v1:
#    1. parse_time()      - shape-aware parser. Files are ISO8601; dayfirst=True was destroying
#                           4,000,464 rows across 7 stations. All 12.78M rows now survive.
#    2. header metadata   - latitude, longitude, altitude, TIME ZONE, station tier and equipment
#                           are read from the _Header files instead of being detected/guessed.
#    3. Bahawalpur        - coordinates taken from the authors' published station table.
#    4. QC flags          - the "ghi_pyr_flag:8;..." strings in `comments` are parsed into
#                           structured per-channel columns, with a circularity audit.
#    5. feature set       - wind gust and rain dropped (not present in all three countries).
#                           30 features per timestep, unchanged in count.
#
#  Output: /kaggle/working/features/*.parquet, /events/*.parquet, manifest.json
#  CPU only. ~25-40 min for 12.78M rows.
# ============================================================================================

import os, re, gc, json, glob, math, hashlib, warnings, textwrap
from collections import defaultdict, Counter
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 90)
pd.set_option("display.max_rows", 250); pd.set_option("display.max_colwidth", 70)

# ============================================================================== CONFIG
SEARCH_ROOTS  = ["/kaggle/input"]
OUT_DIR       = "/kaggle/working"
COUNTRIES     = ["pakistan", "nepal", "zambia"]
TARGET_STEP   = "10min"
MAX_FFILL_MIN = 60
MIN_SEG_LEN   = 36
DAY_ELEV      = 10.0
WINDOW_L      = 12
SPLIT         = dict(train=0.60, earlystop=0.12, calib=0.18, test=0.10)
SENTINELS     = [-9999.0, -999.0, 9999.0]      # 999 is a valid value, never blanked
SOLAR_CONST   = 1361.0
SEED          = 42
PRINT_HEADERS = True      # set False once you have read them

# Bahawalpur has no _Header file. Coordinates from the authors' own published station table
# (Jabeen et al., Sci Rep 2026, Table 4) -> cite that table in the manuscript.
MANUAL_META = {
    ("pakistan", "bahawalpur"): dict(lat=29.32, lon=71.81, alt=120.0,
                                     tz=5.0, equip="unknown", src="published table"),
}

os.makedirs(f"{OUT_DIR}/features", exist_ok=True)
os.makedirs(f"{OUT_DIR}/events", exist_ok=True)
try:
    import pyarrow; _FMT = "parquet"
except Exception:
    _FMT = "pkl"
def save_df(df, p): (df.to_parquet(f"{p}.{_FMT}") if _FMT == "parquet" else df.to_pickle(f"{p}.{_FMT}"))
def load_df(p):     return (pd.read_parquet(f"{p}.{_FMT}") if _FMT == "parquet" else pd.read_pickle(f"{p}.{_FMT}"))

RULE = "=" * 112
def H1(t): print("\n" + RULE + "\n" + t + "\n" + RULE)
def H2(t): print("\n" + t + "\n" + "-" * 112)

# ============================================================================================
#  THE PARSER  (fix 01)
# ============================================================================================
_FMT_MAP = {
    "DD/DD/DDDD DD:DD":    "%d/%m/%Y %H:%M",   "DD/DD/DDDD DD:DD:DD": "%d/%m/%Y %H:%M:%S",
    "DDDD-DD-DD DD:DD":    "%Y-%m-%d %H:%M",   "DDDD-DD-DD DD:DD:DD": "%Y-%m-%d %H:%M:%S",
    "DDDD-DD-DDADD:DD:DD": "%Y-%m-%dT%H:%M:%S","DD.DD.DDDD DD:DD":    "%d.%m.%Y %H:%M",
    "DD.DD.DDDD DD:DD:DD": "%d.%m.%Y %H:%M:%S","DD-DD-DDDD DD:DD":    "%d-%m-%Y %H:%M",
    "DD-DD-DDDD DD:DD:DD": "%d-%m-%Y %H:%M:%S",
}
def _shape(v):
    # letters FIRST, then digits. Doing it the other way round turns every "D" back into
    # an "A" (D is itself a letter), the shape never matches _FMT_MAP, and the fallback
    # parser silently swaps day and month on ISO dates whose day is <= 12.
    return re.sub(r"\d", "D", re.sub(r"[A-Za-z]", "A", str(v).strip()))
def parse_time(s):
    """Each structural timestamp shape gets its own explicit format. pandas>=2.0 otherwise
       infers ONE format for the column and silently drops every row that does not match."""
    sh = s.map(_shape)
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    unknown = []
    for k in sh.unique():
        m = sh == k
        if k in _FMT_MAP:
            out[m] = pd.to_datetime(s[m], format=_FMT_MAP[k], errors="coerce")
        else:
            unknown.append((k, int(m.sum())))
            out[m] = pd.to_datetime(s[m], errors="coerce", dayfirst=True, format="mixed")
    if unknown:
        print("   !! timestamp shapes with no explicit format (fallback used, VERIFY):", unknown)
    return out

def check_time(t, name):
    """ESMAP QC files are written in chronological order. If the parsed series is not
       monotonic, day and month were swapped somewhere - stop rather than train on it."""
    tv = t.dropna()
    mono = float((tv.diff().dt.total_seconds().dropna() >= 0).mean()) if len(tv) > 1 else 1.0
    if mono < 0.999:
        print(f"   !! {name}: only {100*mono:.2f}% of timestamps increase -> DAY/MONTH SWAP. "
              f"Check _FMT_MAP before using this station.")
    return mono

# ============================================================================================
# PART 1  -  DISCOVERY
# ============================================================================================
H1("PART 1  |  FILE DISCOVERY")
def norm(s): return re.sub(r"[^a-z0-9.]", "", s.lower())

recs = []
for root in SEARCH_ROOTS:
    if not os.path.isdir(root): continue
    for dp, _, fns in os.walk(root):
        for fn in fns:
            nf, npth = norm(fn), norm(dp + "/" + fn)
            c = next((x for x in COUNTRIES if x in npth), None)
            if c is None: continue
            role = ("HEADER" if "header" in nf else
                    "SAT"    if ("satellite" in nf or "tmy" in nf) else
                    "RAW"    if re.search(r"raw\.", nf) else
                    "QC"     if re.search(r"qc\.(csv|txt)$", nf) else "OTHER")
            m = re.search(r"solarmeasurements?(?:%s)(.+?)wb[-]?esmap" % "|".join(COUNTRIES), nf)
            recs.append(dict(country=c, station=m.group(1) if m else nf[:18], role=role,
                             mb=round(os.path.getsize(os.path.join(dp, fn)) / 1e6, 2),
                             file=fn, path=os.path.join(dp, fn)))
INV = pd.DataFrame(recs)
DATA = INV[INV.role == "QC"].sort_values(["country", "station"]).reset_index(drop=True)
HDR  = {(r.country, r.station): r.path for _, r in INV[INV.role == "HEADER"].iterrows()}
print(DATA.groupby("country").station.nunique().to_string())
print(f"QC files: {len(DATA)}   Header files: {len(HDR)}   Raw size: {DATA.mb.sum()/1000:.2f} GB")
print("Missing header:", [ (r.country, r.station) for _, r in DATA.iterrows()
                           if (r.country, r.station) not in HDR ])

# ============================================================================================
# PART 2  -  HEADER METADATA  (coordinates, TIME ZONE, station tier, equipment)
# ============================================================================================
H1("PART 2  |  HEADER METADATA")

def read_grid(p):
    try:
        if p.lower().endswith((".xlsx", ".xls")):
            g = pd.read_excel(p, sheet_name=0, header=None, dtype=str, nrows=150)
        else:
            g = pd.read_csv(p, header=None, dtype=str, nrows=150, sep=None, engine="python",
                            encoding="latin-1", encoding_errors="replace", on_bad_lines="skip")
        return g.fillna("").astype(str)
    except Exception as e:
        print("  !! header unreadable:", os.path.basename(p), e); return pd.DataFrame()

def num_in(cells, lo, hi):
    for x in cells:
        m = re.search(r"-?\d+(?:\.\d+)?", str(x))
        if m:
            v = float(m.group(0))
            if lo <= v <= hi: return v
    return None

def parse_tz(cells):
    """'UTC+5' -> 5.0 | 'UTC+2:00' -> 2.0 | 'UTC+5:45' -> 5.75 | 'UTC+5.75' -> 5.75
       The separator decides: ':' means MINUTES, '.' means a DECIMAL fraction of an hour.
       Treating '.75' as 75 minutes gives 6.25 and silently shifts the sun by 30 minutes."""
    for x in cells:
        m = re.search(r"(?:utc|gmt)\s*([+-])\s*(\d{1,2})(?:([:.])(\d{1,2}))?", str(x), re.I)
        if m:
            sign = 1 if m.group(1) == "+" else -1
            h = float(m.group(2)); sep, val = m.group(3), m.group(4)
            if val is None:      frac = 0.0
            elif sep == ":":     frac = float(val) / 60.0          # minutes
            else:                frac = float("0." + val)          # decimal hours
            return sign * (h + frac)
    return None

META, HEADER_TEXT = {}, {}
for (c, st), p in HDR.items():
    g = read_grid(p)
    if g.empty: continue
    HEADER_TEXT[(c, st)] = g
    d = {}
    for _, row in g.iterrows():
        cells = row.tolist(); lbl = " ".join(cells).lower()
        rest = [x for x in cells if not re.search(r"lat|lon|alt|elev|zone|equip", str(x), re.I)]
        if "lat" in lbl and "lat" not in d:
            v = num_in(rest, -90, 90);  d["lat"] = v if v not in (None, 0) else d.get("lat")
        if "lon" in lbl and "lon" not in d:
            v = num_in(rest, -180, 180); d["lon"] = v if v not in (None, 0) else d.get("lon")
        if re.search(r"elevation|altitude", lbl) and "alt" not in d:
            v = num_in(rest, -500, 9000)
            if v is not None: d["alt"] = v
        if "time zone" in lbl and "tz" not in d:
            v = parse_tz(cells)
            if v is not None: d["tz"] = v
        if "equipment" in lbl and "equip" not in d:
            d["equip"] = " ".join(x for x in rest if x.strip())[:70]
    if d.get("lat") is not None and d.get("lon") is not None:
        META[(c, st)] = dict(lat=d["lat"], lon=d["lon"], alt=d.get("alt", 0.0),
                             tz=d.get("tz"), equip=d.get("equip", ""), src="header")
META.update(MANUAL_META)

MD = pd.DataFrame([dict(country=k[0], station=k[1], **v) for k, v in META.items()])
MD["tier"] = MD.equip.fillna("").str.extract(r"(?i)(tier\s*\d)")[0].str.upper().str.replace(" ", "")
MD["rsi"]  = MD.equip.fillna("").str.contains("shadowband|rsi|rsr", case=False).astype(int)
MD["tier"] = MD["tier"].fillna("UNKNOWN")
print(MD.sort_values(["country", "station"])[
    ["country", "station", "lat", "lon", "alt", "tz", "tier", "rsi", "src", "equip"]
].to_string(index=False))
missing = [(r.country, r.station) for _, r in DATA.iterrows() if (r.country, r.station) not in META]
print("\nStill missing metadata:", missing if missing else "none")
if MD.tz.isna().any():
    print("!! Time zone not parsed for:", MD[MD.tz.isna()].station.tolist(),
          "-> will fall back to detection.")

if PRINT_HEADERS:
    H2("FULL HEADER CONTENT  (one per country - look here for the QC FLAG LEGEND)")
    seen = set()
    for (c, st), g in HEADER_TEXT.items():
        if c in seen: continue
        seen.add(c)
        print(f"\n---------- {c}/{st} ----------")
        print(g.to_string(max_rows=60))

# ============================================================================================
# PART 3  -  SOLAR GEOMETRY
# ============================================================================================
def solar_geometry(idx_utc, lat, lon, alt=0.0):
    doy = idx_utc.dayofyear.values.astype("float64")
    hr  = idx_utc.hour.values + idx_utc.minute.values / 60.0 + idx_utc.second.values / 3600.0
    g = 2 * np.pi / 365.0 * (doy - 1 + (hr - 12) / 24.0)
    eqt = 229.18 * (0.000075 + 0.001868*np.cos(g) - 0.032077*np.sin(g)
                    - 0.014615*np.cos(2*g) - 0.040849*np.sin(2*g))
    decl = (0.006918 - 0.399912*np.cos(g) + 0.070257*np.sin(g) - 0.006758*np.cos(2*g)
            + 0.000907*np.sin(2*g) - 0.002697*np.cos(3*g) + 0.00148*np.sin(3*g))
    tst = (hr*60.0 + eqt + 4.0*lon) % 1440.0
    ha  = np.radians(tst/4.0 - 180.0); la = np.radians(lat)
    cosz = np.clip(np.sin(la)*np.sin(decl) + np.cos(la)*np.cos(decl)*np.cos(ha), -1, 1)
    elev = np.degrees(np.arcsin(cosz))
    sinz = np.sqrt(np.clip(1 - cosz**2, 0, 1))
    cosaz = np.divide(np.sin(decl)*np.cos(la) - np.cos(decl)*np.sin(la)*np.cos(ha),
                      np.where(sinz > 1e-6, sinz, np.nan))
    cosaz = np.clip(np.nan_to_num(cosaz, nan=0.0), -1, 1)
    az = np.where(ha > 0, 2*np.pi - np.arccos(cosaz), np.arccos(cosaz))
    E0 = SOLAR_CONST * (1 + 0.033*np.cos(2*np.pi*doy/365.0))
    czp = np.where(cosz > 0.01, cosz, np.nan)
    am = 1.0/(czp + 0.50572*np.power(np.degrees(np.arcsin(czp)) + 6.07995, -1.6364))
    am = np.clip(np.nan_to_num(am, nan=40.0, posinf=40.0)*np.exp(-alt/8435.0), 0, 40)
    ghi_cs = np.nan_to_num(1098.0*czp*np.exp(-0.059/czp), nan=0.0)
    dni_cs = np.where(cosz > 0.01, 0.9751*SOLAR_CONST*np.exp(-0.09*am), 0.0)
    return dict(cosz=cosz, elev=elev, az=az, decl=decl, E0=E0, am=am,
                ghi_cs=ghi_cs, dni_cs=np.nan_to_num(dni_cs))

def detect_offset(idx_local, ghi, lat, lon, max_n=150_000):
    m = np.isfinite(ghi)
    if m.sum() < 5000: return 0.0
    ix, gv = idx_local[m][:max_n], np.asarray(ghi, "float64")[m][:max_n]
    best, br = 0.0, -2.0
    for off in np.arange(-12, 14.01, 0.25):
        cz = np.clip(solar_geometry(pd.DatetimeIndex(ix - pd.Timedelta(hours=float(off))),
                                    lat, lon)["cosz"], 0, None)
        if cz.std() == 0: continue
        r = float(np.corrcoef(gv, cz)[0, 1])
        if r > br: br, best = r, float(off)
    return best

# ============================================================================================
# PART 4  -  COLUMNS, QC, EVENTS
# ============================================================================================
CANON = {
    "time":  ["time", "timestamp", "datetime", "date_time", "local_time", "utc", "date"],
    "ghi":   ["ghi_pyr", "ghi", "ghi_rsi"],              # prefer thermopile pyranometer
    "dni":   ["dni", "dni_pyr", "dni_rsi"],              # prefer pyrheliometer over RSI
    "dhi":   ["dhi", "dhi_pyr", "dhi_rsi"],
    "temp":  ["air_temperature", "temperature", "air_temp"],
    "rh":    ["relative_humidity", "humidity"],
    "wspd":  ["wind_speed", "windspeed"],
    "wdir":  ["wind_from_direction", "wind_direction"],
    "pres":  ["barometric_pressure", "pressure", "air_pressure"],
    "clean": ["sensor_cleaning", "cleaning"],
    "cmt":   ["comments", "comment", "remarks", "notes"],
}
MEAS = ["ghi", "dni", "dhi", "temp", "rh", "wspd", "wdir", "pres"]   # gust & rain dropped

def resolve(cols):
    low = {c.lower().strip(): c for c in cols}; out = {}
    for k, alts in CANON.items():
        for a in alts:
            if a in low: out[k] = low[a]; break
        if k not in out:
            for lc, orig in low.items():
                if "st_dev" in lc or "stdev" in lc or lc.endswith("_flag"): continue
                if any(a in lc for a in alts): out[k] = orig; break
    return out

# ---- Provider QC flag legend, taken verbatim from the Zambia header files:
#   0 sun below horizon | 1 passed | 2 failed physical limit LOW | 3 failed physical limit HIGH
#   5 failed CONSISTENCY between DNI, GHI and DIF | 8 failed visual check (partial)
#   9 failed visual check (full day) | -1 not flagged, other problem
QC_FLAG_MEANING = {0: "night", 1: "passed", 2: "limit_low", 3: "limit_high",
                   5: "consistency", 8: "visual_partial", 9: "visual_full", -1: "other"}
QC_BAD   = {2, 3, 5, 8, 9}
QC_CLASS = {5: ("B7_qc_consistency", "anomaly"),   # the provider's own closure test
            2: ("B6_qc_limit", "anomaly"), 3: ("B6_qc_limit", "anomaly"),
            8: ("B8_qc_visual", "anomaly"), 9: ("B8_qc_visual", "anomaly"),
            -1: ("B9_qc_other", "mask")}
QC_PRIORITY = [5, 3, 2, 9, 8, -1]      # consistency first: it is our direct comparator

EVENT_RULES = [
    (r"dew on|frost|soil|dirty|not cleaned|contaminat",           "B1_sensor_fault", "anomaly"),
    (r"power (supply )?fail|outage|no power|battery",             "B1_sensor_fault", "anomaly"),
    (r"modeled|modelled|filled with data|substitut|"
     r"parallel me[as]surement|other time period|erbs",           "B2_substituted",  "anomaly"),
    (r"maintenance|datalogger reset|reset|calibrat|visit|"
     r"sensor cleaning|cleaning|installation",                    "B3_context",      "mask"),
    (r"misalign|not operational|tracking device|tracker",         "B4_tracker",      "special"),
]
def classify(t):
    t = str(t).lower()
    for pat, cls, sc in EVENT_RULES:
        if re.search(pat, t): return cls, sc
    return "B5_other", "mask"

FLAG_RE = re.compile(r"([a-z_]+?)_flag\s*:\s*(-?\d+)", re.I)
def split_comments(s):
    """Separate machine QC flags from human free text.
       Returns (flags DataFrame of ints, free-text Series)."""
    txt = s.astype("string")
    blank = txt.isna() | txt.str.strip().str.lower().isin(
        ["", "nan", "none", "0", "0.0", "false", "-"])
    txt = txt.where(~blank)
    has_flag = txt.fillna("").str.contains("_flag", case=False)
    flags = pd.DataFrame(index=s.index, dtype="float32")
    if has_flag.any():
        for m in FLAG_RE.finditer(str(txt[has_flag].iloc[0])):
            flags[m.group(1).lower() + "_flag"] = np.nan
        sub = txt[has_flag]
        for col in flags.columns:
            pat = col.replace("_flag", "") + r"_flag\s*:\s*(-?\d+)"
            flags.loc[has_flag, col] = pd.to_numeric(
                sub.str.extract(pat, flags=re.I)[0], errors="coerce").values
    free = txt.where(~has_flag)
    # a comment can hold both: strip the flag part and keep any remaining prose
    return flags, free

# ============================================================================================
# PART 5  -  MAIN LOOP
# ============================================================================================
H1("PART 5  |  BUILDING THE FEATURE LAYER")

FEATURE_ORDER = [
    "ghi", "dni", "dhi", "temp", "rh", "wspd", "pres", "wdir_sin", "wdir_cos",       # 9
    "cosz", "sin_az", "cos_az", "airmass", "decl", "ghi_cs", "dni_cs",               # 7
    "kt", "kt_dni", "diffuse_frac", "r_closure", "closure_valid",
    "r_closure_6h", "kt_std_1h", "ghi_ramp",                                          # 8
    "block_completeness", "imputed_flag", "gap_log",                                  # 3
    "hour_sin", "hour_cos", "daylight",                                               # 3
]

summary, closT, flagT, evAll, tzT = [], [], [], [], []

for i, R in DATA.iterrows():
    c, st = R.country, R.station
    tag = f"{c}__{re.sub(r'[^a-z0-9]+','_',st.lower())}"
    md = META.get((c, st))
    if md is None:
        print(f"  SKIP {c}/{st}: no metadata"); continue

    head = pd.read_csv(R.path, nrows=3); cm = resolve(head.columns)
    if not {"time", "ghi", "dni", "dhi"} <= set(cm):
        print(f"  SKIP {c}/{st}: core columns missing -> {list(head.columns)}"); continue
    df = pd.read_csv(R.path, usecols=[cm[k] for k in cm], low_memory=False)
    df = df.rename(columns={v: k for k, v in cm.items()})

    # ---- TIME (fixed parser)
    t = parse_time(df["time"]); n_bad = int(t.isna().sum())
    mono = check_time(t, f"{c}/{st}")
    df = df.loc[t.notna()].copy(); t = t[t.notna()]
    df.index = pd.DatetimeIndex(t.values); df = df.sort_index()
    n_dup = int(df.index.duplicated().sum()); df = df[~df.index.duplicated(keep="first")]
    df = df.drop(columns=["time"])
    for k in MEAS:
        if k in df: df[k] = pd.to_numeric(df[k], errors="coerce").astype("float64")

    # ---- COMMENTS -> machine flags + human events (labels only, never features)
    FLAGS = pd.DataFrame(index=df.index); FREE = pd.Series(pd.NA, index=df.index, dtype="string")
    if "cmt" in df.columns:
        FLAGS, FREE = split_comments(df["cmt"])
    if "clean" in df.columns:
        cl = pd.to_numeric(df["clean"], errors="coerce")
        FREE = FREE.fillna(pd.Series(np.where(cl == 1, "sensor cleaning", pd.NA),
                                     index=df.index, dtype="string"))
    df = df.drop(columns=[x for x in ("cmt", "clean") if x in df.columns])

    if len(FLAGS.columns):
        for col in FLAGS.columns:
            vc = FLAGS[col].value_counts(dropna=True)
            base = col.replace("_flag", "")
            tgt = ("ghi" if base.startswith("ghi") else "dni" if base.startswith("dni")
                   else "dhi" if base.startswith("dhi") else None)
            for val, n in vc.items():
                mrow = FLAGS[col] == val
                nan_share = (float(df[tgt][mrow].isna().mean()) if tgt in df else np.nan)
                flagT.append(dict(station=tag, flag_col=col, value=int(val), rows=int(n),
                                  data_missing_pct=round(100*nan_share, 2)
                                  if nan_share == nan_share else np.nan))
    ev = FREE.dropna()
    if len(ev):
        for v, n in ev.value_counts().items():
            cls, sc = classify(v)
            evAll.append(dict(country=c, station=tag, raw=str(v)[:95],
                              event_class=cls, scored_as=sc, rows=int(n)))

    # ---- TIME ZONE: header first, detection only as fallback
    tz = md.get("tz")
    tz_src = "header"
    if tz is None or not np.isfinite(tz):
        tz = detect_offset(df.index, df["ghi"].values, md["lat"], md["lon"]); tz_src = "detected"
    tzT.append(dict(station=tag, tz=tz, source=tz_src))
    df.index = df.index - pd.Timedelta(hours=float(tz))
    FLAGS.index = df.index if len(FLAGS) == len(df) else FLAGS.index
    FREE = pd.Series(FREE.values, index=df.index) if len(FREE) == len(df) else FREE

    # ---- PHYSICAL QC at native resolution
    geo_n = solar_geometry(df.index, md["lat"], md["lon"], md["alt"])
    cz_n, E0 = np.clip(geo_n["cosz"], 0, None), geo_n["E0"]
    mu = np.power(cz_n, 1.2)
    LIM = {"ghi": (-4.0, 1.5*E0*mu + 100), "dni": (-4.0, E0), "dhi": (-4.0, 0.95*E0*mu + 50),
           "temp": (-45.0, 60.0), "rh": (0.0, 105.0), "wspd": (0.0, 75.0),
           "wdir": (0.0, 360.0), "pres": (400.0, 1100.0)}
    for k, (lo, hi) in LIM.items():
        if k not in df: continue
        v = df[k].values
        bad = np.isin(v, SENTINELS) | (v < lo) | (v > hi)
        df[k] = np.where(bad, np.nan, v)
    if "pres" in df:      # altitude-aware: reject anything far from this station's own level
        p = df["pres"]; med = p.median()
        df["pres"] = p.where((p - med).abs() < 60)
    if "rh" in df: df["rh"] = df["rh"].clip(upper=100.0)
    night = geo_n["elev"] < -2
    for k in ("ghi", "dni", "dhi"):
        if k in df: df.loc[night & df[k].notna() & (df[k].abs() < 20), k] = 0.0
    del geo_n; gc.collect()

    # ---- RESAMPLE to the common grid
    agg = {k: "mean" for k in MEAS if k in df and k != "wdir"}
    if "wdir" in df:
        df["_ws"] = np.sin(np.radians(df["wdir"])); df["_wc"] = np.cos(np.radians(df["wdir"]))
        agg["_ws"] = "mean"; agg["_wc"] = "mean"
    rs = df.resample(TARGET_STEP, label="left", closed="left").agg(agg)
    step_native = float(pd.Series(df.index).diff().dt.total_seconds().mode().iloc[0])
    per_block = max(1, int(round(600 / max(step_native, 1))))
    rs["block_completeness"] = (df["ghi"].resample(TARGET_STEP, label="left", closed="left")
                                .count() / per_block).clip(0, 1).astype("float32")
    n_native = len(df)

    lab = pd.DataFrame(index=rs.index); lab["event_class"] = "none"; lab["scored_as"] = "none"
    if len(FREE.dropna()):
        cls = FREE.dropna().map(lambda v: classify(v))
        tmp = pd.DataFrame({"event_class": [a for a, _ in cls], "scored_as": [b for _, b in cls]},
                           index=FREE.dropna().index)
        pr = {"anomaly": 3, "special": 2, "mask": 1}
        tmp["_p"] = tmp.scored_as.map(pr)
        red = tmp.resample(TARGET_STEP, label="left", closed="left")["_p"].max()
        pick = tmp.resample(TARGET_STEP, label="left", closed="left").agg(
            {"event_class": "last", "scored_as": "last"})
        ix = lab.index.intersection(pick.dropna(subset=["scored_as"]).index)
        lab.loc[ix, ["event_class", "scored_as"]] = pick.loc[ix, ["event_class", "scored_as"]].values
    if len(FLAGS.columns):
        # keep only the irradiance channels; a block is "bad" if any sample in it was flagged bad
        irr = [c for c in FLAGS.columns if c.split("_")[0] in ("ghi", "dni", "dhi")]
        worst = pd.DataFrame(index=FLAGS.index)
        for code in QC_PRIORITY:
            worst[code] = FLAGS[irr].eq(code).any(axis=1).astype("int8")
        w10 = worst.resample(TARGET_STEP, label="left", closed="left").max()
        for col in FLAGS.columns:
            lab[col] = FLAGS[col].resample(TARGET_STEP, label="left",
                                           closed="left").max().reindex(lab.index).values
        qc_cls = pd.Series("none", index=lab.index, dtype=object)
        qc_sc  = pd.Series("none", index=lab.index, dtype=object)
        for code in QC_PRIORITY:                      # lowest priority written first
            hit = w10[code].reindex(lab.index).fillna(0).astype(bool).values
            cls, sc = QC_CLASS[code]
            qc_cls[hit] = cls; qc_sc[hit] = sc
        lab["qc_class"] = qc_cls.values; lab["qc_scored"] = qc_sc.values
        # free text wins where it exists (it is more specific); QC flags fill the rest
        fill = (lab["scored_as"] == "none") & (lab["qc_scored"] != "none")
        lab.loc[fill, "event_class"] = lab.loc[fill, "qc_class"]
        lab.loc[fill, "scored_as"]   = lab.loc[fill, "qc_scored"]
    del df; gc.collect()

    # ---- GEOMETRY + FEATURES
    geo = solar_geometry(rs.index, md["lat"], md["lon"], md["alt"])
    cz = np.clip(geo["cosz"], 0, None)
    lim = MAX_FFILL_MIN // 10
    cols = [k for k in MEAS if k in rs]
    pre = rs[cols].isna(); rs[cols] = rs[cols].ffill(limit=lim); post = rs[cols].isna()
    if "_ws" in rs: rs[["_ws", "_wc"]] = rs[["_ws", "_wc"]].ffill(limit=lim)
    imputed = (pre & ~post).any(axis=1).astype("float32")
    ok = rs["ghi"].notna()
    gap = ok.groupby((ok != ok.shift()).cumsum()).cumcount()

    F = pd.DataFrame(index=rs.index)
    for k in ["ghi", "dni", "dhi", "temp", "rh", "wspd", "pres"]:
        F[k] = rs[k].astype("float32") if k in rs else np.float32(0)
    F["wdir_sin"] = rs["_ws"].astype("float32") if "_ws" in rs else np.float32(0)
    F["wdir_cos"] = rs["_wc"].astype("float32") if "_wc" in rs else np.float32(0)
    F["cosz"] = cz.astype("float32");            F["sin_az"] = np.sin(geo["az"]).astype("float32")
    F["cos_az"] = np.cos(geo["az"]).astype("float32"); F["airmass"] = (geo["am"]/40).astype("float32")
    F["decl"] = geo["decl"].astype("float32")
    F["ghi_cs"] = geo["ghi_cs"].astype("float32"); F["dni_cs"] = geo["dni_cs"].astype("float32")

    cs = np.maximum(geo["ghi_cs"], 1.0)
    F["kt"]  = np.clip(F["ghi"]/cs, 0, 1.3).astype("float32")
    F["kt_dni"] = np.clip(F["dni"]/np.maximum(geo["dni_cs"], 1.0), 0, 1.3).astype("float32")
    F["diffuse_frac"] = np.clip(F["dhi"]/np.maximum(F["ghi"], 1.0), 0, 1.5).astype("float32")
    rc = (F["ghi"] - F["dhi"] - F["dni"]*cz) / cs
    valid = ((geo["elev"] > DAY_ELEV) & F["ghi"].notna() & F["dni"].notna() & F["dhi"].notna()
             & (geo["ghi_cs"] > 50))
    F["r_closure"] = np.where(valid, np.clip(rc, -1.5, 1.5), 0.0).astype("float32")
    F["closure_valid"] = valid.astype("float32")
    F["r_closure_6h"] = pd.Series(F["r_closure"], index=F.index).rolling(36, min_periods=6)\
                          .mean().fillna(0).astype("float32")
    F["kt_std_1h"] = pd.Series(F["kt"], index=F.index).rolling(6, min_periods=3)\
                       .std().fillna(0).astype("float32")
    F["ghi_ramp"] = (pd.Series(F["ghi"], index=F.index).diff().fillna(0)/cs).astype("float32")
    F["block_completeness"] = rs["block_completeness"].fillna(0).values
    F["imputed_flag"] = imputed.values
    F["gap_log"] = np.log1p(np.where(ok, 0, gap*10)).astype("float32")
    hr = F.index.hour.values + F.index.minute.values/60
    F["hour_sin"] = np.sin(2*np.pi*hr/24).astype("float32")
    F["hour_cos"] = np.cos(2*np.pi*hr/24).astype("float32")
    F["daylight"] = (geo["elev"] > 0).astype("float32")

    if valid.sum() > 1000:
        r = rc[valid]; ktv = F["kt"].values[valid]
        clear = ktv > 0.7
        closT.append(dict(station=tag, tier=str(MD.loc[(MD.country == c) & (MD.station == st),
                                                       "tier"].squeeze()),
                          n=int(valid.sum()), usable=round(100*float(valid.sum()/max((geo["elev"]>DAY_ELEV).sum(),1)), 1),
                          mean=round(float(np.nanmean(r)), 4), std=round(float(np.nanstd(r)), 4),
                          p01=round(float(np.nanpercentile(r, 1)), 4),
                          p99=round(float(np.nanpercentile(r, 99)), 4),
                          gt08=round(100*float(np.nanmean(np.abs(r) > 0.08)), 2),
                          gt20=round(100*float(np.nanmean(np.abs(r) > 0.20)), 2),
                          near0_clear=round(100*float(np.nanmean(np.abs(r[clear]) < 0.005)), 2)
                          if clear.sum() > 100 else np.nan))

    # ---- SEGMENTS / SPLIT / LABELS
    usable = F["ghi"].notna().values & (rs["block_completeness"].fillna(0).values > 0.2)
    sid = np.cumsum(~usable); F["segment"] = np.where(usable, sid, -1).astype("int32")
    sz = pd.Series(F.loc[F.segment >= 0, "segment"]).value_counts()
    F["segment"] = np.where(F["segment"].isin(set(sz[sz >= MIN_SEG_LEN].index)),
                            F["segment"], -1).astype("int32")
    F = F.fillna(0.0)
    n = len(F); b = (np.cumsum([SPLIT["train"], SPLIT["earlystop"], SPLIT["calib"]])*n).astype(int)
    part = np.full(n, "test", dtype=object)
    part[:b[0]] = "train"; part[b[0]:b[1]] = "earlystop"; part[b[1]:b[2]] = "calib"
    F["partition"] = part; F["country"] = c; F["station"] = tag
    F["tier"] = str(MD.loc[(MD.country == c) & (MD.station == st), "tier"].squeeze())
    F["rsi"]  = int(MD.loc[(MD.country == c) & (MD.station == st), "rsi"].squeeze())
    for col in lab.columns: F[col] = lab[col].reindex(F.index).values
    F.loc[F.partition.isin(["train", "calib"]) & (F.scored_as != "none"), "partition"] = "excluded"

    save_df(F, f"{OUT_DIR}/features/{tag}")
    save_df(lab, f"{OUT_DIR}/events/{tag}")
    summary.append(dict(country=c, station=tag, native=n_native, rows=n, bad_time=n_bad, dup=n_dup,
                        start=F.index.min(), end=F.index.max(), tz=tz, tz_src=tz_src,
                        mono=round(100*mono, 2),
                        tier=F["tier"].iloc[0], rsi=F["rsi"].iloc[0],
                        seg=int(F.loc[F.segment >= 0, "segment"].nunique()),
                        train=int((F.partition == "train").sum()),
                        es=int((F.partition == "earlystop").sum()),
                        calib=int((F.partition == "calib").sum()),
                        test=int((F.partition == "test").sum()),
                        excl=int((F.partition == "excluded").sum()),
                        anom=int((F.scored_as == "anomaly").sum()),
                        anom_txt=int(F.event_class.isin(
                            ["B1_sensor_fault", "B2_substituted"]).sum()),
                        anom_qc=int(F.event_class.isin(
                            ["B6_qc_limit", "B7_qc_consistency", "B8_qc_visual"]).sum())))
    print(f"  [{i+1:>2}/{len(DATA)}] {tag:<24} {n_native:>9,} -> {n:>7,} | tz {tz:+.2f} ({tz_src})"
          f" | bad_time {n_bad} | seg {summary[-1]['seg']:>4}")
    del F, rs, geo, lab, FLAGS, FREE; gc.collect()

SUM = pd.DataFrame(summary)

# ============================================================================================
# PART 6  -  REPORTS
# ============================================================================================
H2("A. Result per station")
print(SUM[["country", "station", "native", "rows", "bad_time", "mono", "dup", "tz", "tz_src",
           "tier", "rsi", "seg", "start", "end"]].to_string(index=False))
print("\nmono = % of timestamps that increase. Anything below 100 means day/month were swapped.")
print(f"\nTotal native rows kept: {int(SUM.native.sum()):,}   (expected 12,782,668)")
print(f"Timestamps lost       : {int(SUM.bad_time.sum()):,}   (was 4,000,464 with the old parser)")

H2("B. Time zones used")
print(pd.DataFrame(tzT).to_string(index=False))

H2("C. CLOSURE RESIDUAL by station and tier   ***  Novelty 1  ***")
if closT:
    CL = pd.DataFrame(closT).sort_values("std")
    print(CL.to_string(index=False))
    print("\n  near0_clear = |residual| < 0.005 under CLEAR sky only (kt > 0.7).")
    print("  Real measurements always carry noise, so a high value here is the fingerprint")
    print("  of DNI/DHI that were modelled rather than measured.")
    print("\n  Tier 1 stations use a pyrheliometer on a tracker; Tier 2 stations use a rotating")
    print("  shadowband irradiometer, which derives DNI and DHI from one sensor. Expect a")
    print("  tighter closure at Tier 2 for that reason - condition the threshold on tier.")

H2("D. QC FLAG AUDIT  (are the provider flags usable as labels, or just missingness?)")
if flagT:
    FT = pd.DataFrame(flagT)
    piv = FT.groupby(["flag_col", "value"]).agg(
        rows=("rows", "sum"), stations=("station", "nunique"),
        data_missing_pct=("data_missing_pct", "mean")).reset_index()
    print(piv.sort_values(["flag_col", "value"]).to_string(index=False))
    print("\n  HOW TO READ THIS - the circularity test:")
    print("   data_missing_pct ~ 100  -> the flag only marks data that is already absent.")
    print("                              Useless as a label; it adds nothing beyond missingness.")
    print("   data_missing_pct ~ 0    -> the measurement IS STILL THERE but the provider's own")
    print("                              QC system judged it bad. That is a REAL labelled fault,")
    print("                              and it gives you the operational-QC baseline for free.")
else:
    print("No machine QC flags found in the comments column.")

H2("E. Free-text events (Track B)")
if evAll:
    EV = pd.DataFrame(evAll).groupby(["event_class", "scored_as", "raw"], as_index=False).rows.sum()
    print(EV.sort_values(["event_class", "rows"], ascending=[True, False]).to_string(index=False))
    print("\nTotals:"); print(EV.groupby(["event_class", "scored_as"]).rows.sum().to_string())

H2("F. Partitions")
print(SUM[["country", "station", "train", "es", "calib", "test", "excl",
           "anom", "anom_txt", "anom_qc"]].to_string(index=False))
print("\nanom_txt = from operator free text (Pakistan, Nepal)")
print("anom_qc  = from the provider's own QC flags (Zambia)")
print("Track B therefore has a different label source per country - state this in the paper.")

# ============================================================================================
# PART 7  -  FOLDS  (scaler fitted per fold, on TRAIN rows only)
# ============================================================================================
H1("PART 7  |  FOLD BUILDER")
FEATS = list(FEATURE_ORDER)
def load_station(tag): return load_df(f"{OUT_DIR}/features/{tag}")

def build_fold(train_stations, test_stations, feats=FEATS):
    from sklearn.preprocessing import RobustScaler
    tr = pd.concat([load_station(s) for s in train_stations])
    sc = RobustScaler(quantile_range=(5, 95)).fit(
        tr[(tr.partition == "train") & (tr.segment >= 0)][feats].values)
    def take(st, parts):
        d = pd.concat([load_station(s) for s in st])
        d = d[d.partition.isin(parts) & (d.segment >= 0)]
        return np.clip(sc.transform(d[feats].values).astype("float32"), -10, 10), d
    cross = bool(set(test_stations) - set(train_stations))
    Xtr, dtr = take(train_stations, ["train"])
    Xes, des = take(train_stations, ["earlystop"])
    Xca, dca = take(train_stations, ["calib"])
    Xte, dte = take(test_stations,
                    ["train", "earlystop", "calib", "test", "excluded"] if cross else ["test"])
    return dict(scaler=sc, X_train=Xtr, X_earlystop=Xes, X_calib=Xca, X_test=Xte,
                meta_train=dtr, meta_earlystop=des, meta_calib=dca, meta_test=dte, feats=feats)

def make_windows(X, meta, L=WINDOW_L, stride=1):
    keys = meta["station"].astype(str).values + "|" + meta["segment"].astype(str).values
    starts, s0 = [], 0
    for i in range(1, len(keys) + 1):
        if i == len(keys) or keys[i] != keys[s0]:
            starts.extend(range(s0, i - L + 1, stride)); s0 = i
    starts = np.asarray(starts, np.int64)
    W = (np.stack([X[s:s+L] for s in starts]).astype("float32") if len(starts)
         else np.empty((0, L, X.shape[1]), "float32"))
    return W, starts

ALL = sorted(SUM.station)
PK = [s for s in ALL if s.startswith("pakistan__")]
NP_ = [s for s in ALL if s.startswith("nepal__")]
ZM = [s for s in ALL if s.startswith("zambia__")]
FOLDS = {**{f"T1_{s}": dict(train=[x for x in PK if x != s], test=[s]) for s in PK},
         "T2_nepal": dict(train=PK, test=NP_), "T3_zambia": dict(train=PK, test=ZM)}

man = dict(seed=SEED, step=TARGET_STEP, window=WINDOW_L, split=SPLIT, sentinels=SENTINELS,
           features=FEATS, n_features=len(FEATS), folds=FOLDS,
           stations=SUM.drop(columns=["start", "end"]).to_dict("records"))
man["sha256"] = hashlib.sha256(json.dumps(man, default=str, sort_keys=True).encode()).hexdigest()[:16]
json.dump(man, open(f"{OUT_DIR}/manifest.json", "w"), indent=2, default=str)

print(f"Features/timestep : {len(FEATS)}      Input tensor: (N, {WINDOW_L}, {len(FEATS)})")
print(f"Stations          : PK={len(PK)}  NP={len(NP_)}  ZM={len(ZM)}")
print(f"Folds             : {len(FOLDS)}   Benchmark hash: {man['sha256']}")
groups = [("measured", FEATS[:9]), ("solar geometry", FEATS[9:16]),
          ("physics", FEATS[16:24]), ("data quality", FEATS[24:27]), ("time", FEATS[27:30])]
for g, cs_ in groups: print(f"  {g:<16} ({len(cs_):>2}) : {', '.join(cs_)}")
print("\n  wind gust and rain are absent from at least one country, so they are excluded.")
print("  doy_sin/doy_cos replaced by solar declination -> hemisphere-safe for Zambia.")
print("  tier / rsi are stored as metadata for stratification, NOT fed to the model,")
print("  because a station-constant input would let it memorise the station.")

H1("PHASE 1 v2 COMPLETE  -  still CPU only")
print(f"rows at 10-min : {int(SUM.rows.sum()):,}")
print(f"anomaly-labelled: {int(SUM.anom.sum()):,}")
print("\nNext: the closure figure (cloudy day vs a real sensor fault). GPU stays off.")


PART 1  |  FILE DISCOVERY
country
nepal       5
pakistan    9
zambia      6
QC files: 20   Header files: 19   Raw size: 1.34 GB
Missing header: [('pakistan', 'bahawalpur')]

PART 2  |  HEADER METADATA
 country    station       lat      lon    alt   tz    tier  rsi             src                                                                equip
   nepal     dharan  26.79290 87.29260  310.0 5.75   TIER1    0          header  Tier1 station with solar tracker and thermopile irradiation sensors
   nepal      jumla  29.27240 82.19350 2383.0 5.75   TIER1    0          header  Tier1 station with solar tracker and thermopile irradiation sensors
   nepal  kathmandu  27.68160 85.31870 1320.0 5.75   TIER1    0          header  Tier1 station with solar tracker and thermopile irradiation sensors
   nepal      lumle  28.29660 83.81790 1740.0 5.75   TIER1    0          header  Tier1 station with solar tracker and thermopile irradiation sensors
   nepal  nepalgunj  28.11300 81.58900  150.0 5.75   

In [4]:
# 3
# ============================================================================================
#  SolarQC-Net  |  PHASE 2  -  SYNTHETIC ANOMALY INJECTION + THE CLOSURE FIGURE
#
#  Why this step exists
#  --------------------
#  Training is unsupervised: the model only ever sees normal data. Labels are needed ONLY to
#  answer "is the detector any good?". Real operator-logged faults exist at 14 of 20 stations,
#  but 6 Pakistan stations have none, so a real-label-only evaluation would be unbalanced and
#  would silently exclude a third of the network. Controlled synthetic anomalies give every
#  station a balanced, reproducible evaluation set; the real events then validate externally.
#
#  Leakage rules enforced here
#  ---------------------------
#    * injection happens AFTER the split, and ONLY into the TEST partition
#    * the CALIBRATION partition is deliberately left clean, because the conformal threshold
#      must be fitted on uncontaminated normal data or its false-alarm guarantee is void
#    * severity parameters are drawn from TRAIN-partition statistics only
#    * injection is applied in physical units, then every derived physics feature
#      (kt, closure residual, ...) is RECOMPUTED so the corruption propagates properly
#
#  Input : /kaggle/working/features/*      (from Phase 1 v3)
#  Output: /kaggle/working/features_inj/*  (use these from now on)
#          /kaggle/working/figures/fig2_closure.png
#  CPU only, ~5-10 min.
# ============================================================================================

import os, re, gc, json, glob, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_rows", 200)

OUT_DIR   = "/kaggle/working"
SEED      = 42
TARGET_CONTAM = 0.08        # fraction of TEST rows that become anomalous
MAX_EVENT_FRAC = 0.15       # one event may never exceed this share of the test partition
DAY_ELEV_KT   = 0.05        # kt is only meaningful when clear-sky GHI is non-trivial

os.makedirs(f"{OUT_DIR}/features_inj", exist_ok=True)
os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)
try:
    import pyarrow; _FMT = "parquet"
except Exception:
    _FMT = "pkl"
def save_df(d, p): (d.to_parquet(f"{p}.{_FMT}") if _FMT == "parquet" else d.to_pickle(f"{p}.{_FMT}"))
def load_df(p):    return (pd.read_parquet(f"{p}.{_FMT}") if _FMT == "parquet" else pd.read_pickle(f"{p}.{_FMT}"))

RULE = "=" * 104
def H1(t): print("\n" + RULE + "\n" + t + "\n" + RULE)
def H2(t): print("\n" + t + "\n" + "-" * 104)

STATIONS = sorted(os.path.basename(p).rsplit(".", 1)[0]
                  for p in glob.glob(f"{OUT_DIR}/features/*.{_FMT}"))
if not STATIONS:
    raise SystemExit("No Phase 1 output found - run Phase 1 v3 first.")
H1(f"PHASE 2  |  {len(STATIONS)} stations")

# ============================================================================================
# ANOMALY TAXONOMY
#   Each type mirrors a documented failure mode of a radiometric station. Durations come from
#   the stuck-run and event statistics measured in the EDA, not from invented numbers.
# ============================================================================================
TYPES = {
    # name            duration (10-min steps)   physical analogue
    "spike":        dict(dur=(1, 3),        desc="datalogger glitch / electrical transient"),
    "freeze":       dict(dur=(6, 72),       desc="datalogger hang, comms loss (1-12 h)"),
    "drift":        dict(dur=(144, 1008),   desc="calibration drift (1-7 days)"),
    "bias":         dict(dur=(36, 288),     desc="zero-offset / thermal offset (6 h - 2 d)"),
    "soiling":      dict(dur=(288, 2016),   desc="dome soiling, shading (2-14 days)"),
    "decoupling":   dict(dur=(36, 432),     desc="tracker failure: DNI collapses alone"),
}
CHANNELS = ["ghi", "dni", "dhi"]

def inject_station(F, rng):
    """Corrupt GHI/DNI/DHI inside the TEST partition only, then recompute derived features."""
    F = F.copy()
    F["inj_label"] = 0
    F["inj_type"]  = "none"

    tr = F[(F.partition == "train") & (F.segment >= 0)]
    te_mask = (F.partition == "test") & (F.segment >= 0) & (F.scored_as == "none")
    idx_all = np.where(te_mask.values)[0]
    if len(idx_all) < 500:
        return F, pd.DataFrame()

    # severity scales measured on the TRAIN partition only
    day = tr["cosz"].values > 0.2
    sd = {c: float(np.nanstd(tr[c].values[day])) if day.sum() > 100 else 1.0 for c in CHANNELS}
    ghi = F["ghi"].values.astype("float64").copy()
    dni = F["dni"].values.astype("float64").copy()
    dhi = F["dhi"].values.astype("float64").copy()

    budget = int(TARGET_CONTAM * len(idx_all))
    per_type = max(6, budget // len(TYPES))
    cap = max(3, int(MAX_EVENT_FRAC * len(idx_all)))
    lo, hi = idx_all.min(), idx_all.max()
    taken = np.zeros(len(F), bool)
    log = []

    for name, spec in TYPES.items():
        placed = 0
        guard = 0
        while placed < per_type and guard < 4000:
            guard += 1
            # never let one event blow the budget: durations are truncated, not invented
            rem = per_type - placed
            if rem < 1: break
            hi_d = max(1, min(spec["dur"][1], rem, cap))
            lo_d = max(1, min(spec["dur"][0], hi_d))
            d = int(rng.integers(lo_d, hi_d + 1))
            s = int(rng.integers(lo, max(lo + 1, hi - d)))
            sl = slice(s, s + d)
            # the whole window must sit inside test, in one segment, and be untouched
            if not te_mask.values[sl].all():      continue
            if taken[sl].any():                   continue
            if F["segment"].values[s] != F["segment"].values[s + d - 1]: continue

            # An injected event that cannot change the data is an UNLEARNABLE label: it says
            # "anomaly" while the samples are untouched, so every detector is forced to miss it
            # and the benchmark is depressed for everyone. Reject such placements.
            cz_w = F["cosz"].values[sl]
            dayl = cz_w > 0.2
            if name in ("drift", "bias", "soiling", "decoupling"):
                if dayl.sum() < max(3, d // 10):  continue
                if name == "decoupling" and np.nanmedian(dni[sl][dayl]) < 50:  continue
                if name == "soiling"    and np.nanmedian(ghi[sl][dayl]) < 50:  continue
                if name == "drift"      and np.nanmedian(ghi[sl][dayl]) < 30:  continue
            before = np.stack([ghi[sl].copy(), dni[sl].copy(), dhi[sl].copy()])

            if name == "spike":
                ch = CHANNELS[int(rng.integers(0, 3))]
                k = float(rng.uniform(4, 10)) * (1 if rng.random() < 0.5 else -1)
                arr = {"ghi": ghi, "dni": dni, "dhi": dhi}[ch]
                arr[sl] = np.clip(arr[sl] + k * sd[ch], 0, None)

            elif name == "freeze":
                for arr in (ghi, dni, dhi):
                    arr[sl] = arr[max(s - 1, 0)]

            elif name == "drift":
                ch = "ghi" if rng.random() < 0.6 else "dni"
                arr = {"ghi": ghi, "dni": dni}[ch]
                mag = float(rng.uniform(0.08, 0.30)) * (1 if rng.random() < 0.5 else -1)
                ramp = np.linspace(0, mag, d)
                arr[sl] = np.clip(arr[sl] * (1 + ramp), 0, None)

            elif name == "bias":
                ch = CHANNELS[int(rng.integers(0, 3))]
                arr = {"ghi": ghi, "dni": dni, "dhi": dhi}[ch]
                off = float(rng.uniform(1.5, 5.0)) * sd[ch] * (1 if rng.random() < 0.5 else -1)
                arr[sl] = np.clip(arr[sl] + off, 0, None)

            elif name == "soiling":
                att = float(rng.uniform(0.10, 0.40))
                ramp = np.linspace(0, att, d)
                ghi[sl] = ghi[sl] * (1 - ramp)          # only the GHI dome is dirty

            elif name == "decoupling":
                att = float(rng.uniform(0.6, 0.95))
                dni[sl] = dni[sl] * (1 - att)           # DNI alone collapses

            after = np.stack([ghi[sl], dni[sl], dhi[sl]])
            scale = max(float(np.nanmedian(F["ghi_cs"].values[sl][dayl])) if dayl.any() else 0.0,
                        50.0)
            effect = float(np.nanmax(np.abs(after - before)) / scale)
            if effect < 0.02:                       # no measurable change -> undo, try again
                ghi[sl], dni[sl], dhi[sl] = before[0], before[1], before[2]
                continue

            taken[sl] = True
            F.iloc[s:s + d, F.columns.get_loc("inj_label")] = 1
            F.iloc[s:s + d, F.columns.get_loc("inj_type")]  = name
            log.append(dict(type=name, start=str(F.index[s]), dur_steps=d,
                            effect=round(effect, 4)))
            placed += d

    F["ghi"] = ghi.astype("float32")
    F["dni"] = dni.astype("float32")
    F["dhi"] = dhi.astype("float32")

    # ---- recompute every feature derived from the irradiance channels
    cz  = F["cosz"].values.astype("float64")
    cs  = np.maximum(F["ghi_cs"].values.astype("float64"), 1.0)
    dcs = np.maximum(F["dni_cs"].values.astype("float64"), 1.0)
    F["kt"]  = np.clip(F["ghi"].values / cs, 0, 1.3).astype("float32")
    F["kt_dni"] = np.clip(F["dni"].values / dcs, 0, 1.3).astype("float32")
    F["diffuse_frac"] = np.clip(F["dhi"].values / np.maximum(F["ghi"].values, 1.0),
                                0, 1.5).astype("float32")
    rc = (F["ghi"].values - F["dhi"].values - F["dni"].values * cz) / cs
    valid = F["closure_valid"].values > 0.5
    F["r_closure"] = np.where(valid, np.clip(rc, -1.5, 1.5), 0.0).astype("float32")
    F["r_closure_6h"] = pd.Series(F["r_closure"].values, index=F.index)\
                          .rolling(36, min_periods=6).mean().fillna(0).astype("float32").values
    F["kt_std_1h"] = pd.Series(F["kt"].values, index=F.index)\
                       .rolling(6, min_periods=3).std().fillna(0).astype("float32").values
    F["ghi_ramp"] = (pd.Series(F["ghi"].values, index=F.index).diff().fillna(0).values
                     / cs).astype("float32")
    return F, pd.DataFrame(log)

# ============================================================================================
# RUN
# ============================================================================================
H2("Injecting (TEST partition only; CALIB deliberately left clean)")
rows, logs = [], []
for i, tag in enumerate(STATIONS):
    F = load_df(f"{OUT_DIR}/features/{tag}")
    rng = np.random.default_rng(SEED + i)
    Fi, lg = inject_station(F, rng)
    save_df(Fi, f"{OUT_DIR}/features_inj/{tag}")
    te = Fi[Fi.partition == "test"]
    rows.append(dict(station=tag,
                     test_rows=len(te),
                     injected=int(te.inj_label.sum()),
                     contam_pct=round(100 * float(te.inj_label.mean()), 2) if len(te) else 0,
                     real_anom=int((te.scored_as == "anomaly").sum()),
                     events=len(lg)))
    if len(lg): logs.append(lg.assign(station=tag))
    print(f"  [{i+1:>2}/{len(STATIONS)}] {tag:<24} test {len(te):>7,} | "
          f"injected {int(te.inj_label.sum()):>6,} ({rows[-1]['contam_pct']:>5.2f}%) | "
          f"real {rows[-1]['real_anom']:>6,}")
    del F, Fi; gc.collect()

S = pd.DataFrame(rows)
H2("A. Evaluation sets per station")
print(S.to_string(index=False))
print(f"\nTotal test rows      : {int(S.test_rows.sum()):,}")
print(f"Synthetic anomalies  : {int(S.injected.sum()):,}   (Track A - all {len(S)} stations)")
print(f"Real logged anomalies: {int(S.real_anom.sum()):,}   "
      f"(Track B - {int((S.real_anom > 0).sum())} stations)")

if logs:
    L = pd.concat(logs)
    H2("B. Injected events by type")
    print(L.groupby("type").agg(events=("dur_steps", "size"),
                                rows=("dur_steps", "sum"),
                                median_dur_h=("dur_steps", lambda x: round(x.median()/6, 1)))
           .to_string())
    L.to_csv(f"{OUT_DIR}/injection_log.csv", index=False)
    print(f"\nFull log -> {OUT_DIR}/injection_log.csv  (reproducible from SEED={SEED})")

# ============================================================================================
# THE CLOSURE FIGURE  -  cloud vs sensor fault
# ============================================================================================
H1("FIGURE 2  |  the closure residual separates weather from failure")

def day_frame(F, day):
    d = F.loc[str(day)]
    return d[d.cosz > 0.05]

def pick_cloudy_day(F):
    """The most variable clouded day whose ENTIRE 24 h is free of logged events and of
       injected anomalies. Selecting on clean rows alone is not enough - the day must be
       clean end to end, or the 'weather' panel would quietly contain a fault."""
    clean_day = F.groupby(F.index.date).apply(
        lambda g: bool(((g.scored_as == "none") & (g.inj_label == 0)).all()))
    d = F[(F.cosz > 0.2) & (F.closure_valid > 0.5)]
    if not len(d): return None
    g = d.groupby(d.index.date).agg(var=("kt_std_1h", "mean"), kt=("kt", "mean"),
                                    n=("kt", "size"))
    g = g[g.n > 30]
    g = g[[clean_day.get(ix, False) for ix in g.index]]     # clean end to end
    if not len(g): return None
    cloudy = g[g.kt < g.kt.quantile(0.7)]                   # relative to this station
    g = cloudy if len(cloudy) else g
    return g.sort_values("var", ascending=False).index[0]

def pick_fault_day(F, cls=("B1_sensor_fault",)):
    d = F[F.event_class.isin(cls) & (F.cosz > 0.2)]
    if not len(d): return None
    g = d.groupby(d.index.date).size()
    return g.sort_values(ascending=False).index[0] if len(g) else None

cloudy = fault = None
for tag in STATIONS:
    F = load_df(f"{OUT_DIR}/features_inj/{tag}")
    fd = pick_fault_day(F)
    if fd is not None and fault is None:
        cd = pick_cloudy_day(F)
        if cd is not None:
            cloudy = (tag, cd, day_frame(F, cd))
            fault  = (tag, fd, day_frame(F, fd))
            break
    del F; gc.collect()

if fault is None:     # fall back to an injected decoupling event
    for tag in STATIONS:
        F = load_df(f"{OUT_DIR}/features_inj/{tag}")
        d = F[(F.inj_type == "decoupling") & (F.cosz > 0.2)]
        if len(d):
            fd = d.groupby(d.index.date).size().sort_values(ascending=False).index[0]
            cd = pick_cloudy_day(F)
            if cd is not None:
                cloudy = (tag, cd, day_frame(F, cd))
                fault  = (tag, fd, day_frame(F, fd))
                break
        del F; gc.collect()

if fault is not None:
    fig, ax = plt.subplots(2, 2, figsize=(12.5, 6.8), sharex="col")
    panels = [("Cloudy day - atmospheric variability", cloudy, "#1f77b4"),
              ("Sensor fault day", fault, "#c0392b")]
    for j, (title, (tag, day, d), col) in enumerate(panels):
        t = d.index
        ax[0, j].plot(t, d["ghi"], lw=1.4, color="#e08214", label="GHI")
        ax[0, j].plot(t, d["dni"], lw=1.2, color="#c0392b", label="DNI")
        ax[0, j].plot(t, d["dhi"], lw=1.2, color="#2c7fb8", label="DHI")
        ax[0, j].plot(t, d["ghi_cs"], lw=1.0, ls="--", color="grey", label="clear-sky GHI")
        ax[0, j].set_title(f"{title}\n{tag}  {day}", fontsize=10)
        ax[0, j].set_ylabel("W/m$^2$"); ax[0, j].legend(fontsize=7, ncol=2)
        ax[0, j].grid(alpha=.25)

        ax[1, j].plot(t, d["r_closure"], lw=1.5, color=col)
        ax[1, j].axhline(0, color="k", lw=.8)
        ax[1, j].axhspan(-0.08, 0.08, color="green", alpha=.10,
                         label="normal band (|r| < 0.08)")
        ax[1, j].set_ylabel("closure residual $r_c$")
        ax[1, j].set_ylim(-0.6, 0.6); ax[1, j].grid(alpha=.25)
        ax[1, j].legend(fontsize=7)
        m = float(np.nanmean(np.abs(d["r_closure"])))
        ax[1, j].text(.02, .92, f"mean |$r_c$| = {m:.3f}", transform=ax[1, j].transAxes,
                      fontsize=9, weight="bold")
        ax[1, j].xaxis.set_major_formatter(matplotlib.dates.DateFormatter("%H:%M"))
        ax[1, j].set_xlabel("time (UTC)")
        for lb in ax[1, j].get_xticklabels(): lb.set_rotation(0)
    fig.suptitle("GHI = DHI + DNI$\\cdot\\cos\\theta_z$ holds through cloud, "
                 "but breaks when a sensor fails", fontsize=12, y=1.0)
    fig.tight_layout()
    fig.savefig(f"{OUT_DIR}/figures/fig2_closure.png", dpi=170, bbox_inches="tight")
    print(f"saved -> {OUT_DIR}/figures/fig2_closure.png")
    plt.show()
    print("\nRead the bottom row. Left: the three components move together through the cloud,")
    print("so the residual stays inside the band. Right: one channel moves alone, the physical")
    print("identity cannot hold, and the residual leaves the band. A detector that sees only")
    print("GHI cannot tell these two days apart. That difference is the contribution.")
else:
    print("Could not locate both a clean cloudy day and a fault day - check event_class values.")

# ============================================================================================
# SEPARABILITY CHECK  -  does the closure channel actually carry signal?
# ============================================================================================
H1("SEPARABILITY  |  |r_closure| on normal vs anomalous test rows")
res = []
for tag in STATIONS:
    F = load_df(f"{OUT_DIR}/features_inj/{tag}")
    te = F[(F.partition == "test") & (F.closure_valid > 0.5)]
    if len(te) < 500: continue
    a = te["r_closure"].abs()
    norm = a[(te.inj_label == 0) & (te.scored_as == "none")]
    row = dict(station=tag, normal_med=round(float(norm.median()), 4))
    for ty in TYPES:
        sub = a[te.inj_type == ty]
        row[ty] = round(float(sub.median()), 4) if len(sub) > 30 else np.nan
    for cls in ["B1_sensor_fault", "B2_substituted", "B6_qc_limit",
                "B7_qc_consistency", "B8_qc_visual"]:
        sub = a[te.event_class == cls]
        row[cls.split("_", 1)[1]] = round(float(sub.median()), 4) if len(sub) > 30 else np.nan
    res.append(row); del F; gc.collect()

R = pd.DataFrame(res)
print(R.to_string(index=False))
print("\nThe last columns are REAL labels, split by class. The closure channel is a")
print("CONSISTENCY test, so it should light up on qc_consistency and on tracker/dew faults,")
print("and NOT on qc_visual or qc_limit, which flag entirely different problems. Reporting")
print("this breakdown is what keeps the Novelty 1 claim honest.")
print("\nEach column is the MEDIAN |closure residual| for that anomaly type, against the")
print("normal_med baseline. A column much larger than normal_med means the physics channel")
print("alone already separates that fault type - that is Novelty 1 working before any training.")
print("Types close to normal_med (typically soiling early on, and pure-GHI bias) are the ones")
print("the learned reconstruction and prediction heads have to catch instead.")

H1("PHASE 2 COMPLETE")
print(f"Use /features_inj/ from now on (not /features/).")
print(f"Stations {len(STATIONS)} | test rows {int(S.test_rows.sum()):,} | "
      f"Track A {int(S.injected.sum()):,} | Track B {int(S.real_anom.sum()):,}")
print("\nNext: Phase 3 - baselines + SolarQC-Net. TURN THE GPU ON for that one.")


PHASE 2  |  20 stations

Injecting (TEST partition only; CALIB deliberately left clean)
--------------------------------------------------------------------------------------------------------
  [ 1/20] nepal__dharan            test  10,486 | injected    828 ( 7.90%) | real      0
  [ 2/20] nepal__jumla             test  10,527 | injected    834 ( 7.92%) | real      0
  [ 3/20] nepal__kathmandu         test  10,527 | injected    834 ( 7.92%) | real     11
  [ 4/20] nepal__lumle             test  11,420 | injected    906 ( 7.93%) | real      0
  [ 5/20] nepal__nepalgunj         test  10,527 | injected    832 ( 7.90%) | real     10
  [ 6/20] pakistan__bahawalpur     test  13,343 | injected  1,054 ( 7.90%) | real      0
  [ 7/20] pakistan__hyderabad      test  10,660 | injected    850 ( 7.97%) | real      0
  [ 8/20] pakistan__islamabad      test  13,226 | injected  1,041 ( 7.87%) | real      0
  [ 9/20] pakistan__karachi        test  10,648 | injected    846 ( 7.95%) | real      0
  [10

In [5]:
# 4
# ============================================================================================
#  SolarQC-Net  |  PHASE 6  -  TUNING + FINAL TRAINING + ABLATION, ONE CELL   *** GPU ON ***
#
#  Replaces Phases 4b / 5 / 5b / 5c. Three stages, all resumable:
#
#    STAGE 1  HYPERPARAMETER SEARCH   -> paper Table 2
#             6 configurations for EVERY model, on the development fold only (Quetta).
#             Equal budget matters: giving the proposed model 20 trials and the baselines 2
#             is the easiest way to lose a reviewer on fairness.
#
#    STAGE 2  FINAL TRAINING          -> paper Tables 3-6
#             Every model at its chosen configuration, on all 11 folds, with SEEDS seeds.
#             Per-timestep scores are saved, so later analysis never needs retraining.
#             Ablation variants inherit the tuned SQ_full configuration, so the ablation
#             isolates the component instead of an incidental change in capacity.
#
#    STAGE 3  ANALYSIS                -> all tables + significance
#             Score aggregation, component weights and the operating point are all frozen on
#             the development fold; Quetta is then excluded from every confirmatory number.
#             Paired bootstrap CIs and exact sign-flip permutation tests over fold x seed.
#
#  Settings carried over from Phases 5/5b/5c: aggregation 'last', target-domain calibration,
#  alpha chosen on the dev fold for detection, alpha = 0.01 for the guarantee analysis.
#
#  ~2.5 h on GPU for 3 seeds. Set SEEDS=[42] first if you want a 50-minute dry run.
# ============================================================================================

import os, re, gc, json, glob, time, math, warnings, itertools
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as Fn
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score, roc_auc_score

warnings.filterwarnings("ignore")
pd.set_option("display.width", 240); pd.set_option("display.max_rows", 500)

# ============================================================================== CONFIG
OUT        = "/kaggle/working"
L          = 12
STRIDE     = 3
BATCH      = 512
EPOCHS     = 12
PATIENCE   = 3
LR         = 1e-3
DEV_FOLD   = "T1_pakistan__quetta"
SEEDS      = [42, 1337, 2024]          # -> [42] for a quick dry run
AGG        = "last"                    # label sits on the window's final timestep
ALPHA_FAR  = 0.01                      # nominal rate for the guarantee analysis
ALPHAS     = [0.02, 0.04, 0.06, 0.08, 0.10, 0.12]
MAX_TRAIN_WIN   = 250_000
SEARCH_TRAIN_WIN = 150_000
BASE       = ["IF", "MLPAE", "LSTMAE", "ATRAN", "TRANAD"]
VARIANTS   = {"SQ_full":      dict(closure=True,  pred=True,  multi=True),
              "SQ_noclosure": dict(closure=False, pred=True,  multi=True),
              "SQ_nopred":    dict(closure=True,  pred=False, multi=True),
              "SQ_nomulti":   dict(closure=True,  pred=True,  multi=False)}
MODELS     = BASE + list(VARIANTS)
WGRID      = [(1,1,1), (1,1,0), (1,0,0), (0,1,0), (0,0,1), (1,1,2), (1,1,4),
              (0,1,1), (0,1,2), (0,1,4), (1,0,2), (1,2,4), (2,1,4)]
DO_STAGE1  = True
DO_STAGE2  = True

GRID = {
 "IF":      [dict(n_estimators=50), dict(n_estimators=100), dict(n_estimators=200),
             dict(n_estimators=300), dict(n_estimators=100, max_samples=256),
             dict(n_estimators=200, max_samples=1024)],
 "MLPAE":   [dict(lat=16, hid=128), dict(lat=32, hid=128), dict(lat=32, hid=256),
             dict(lat=64, hid=256), dict(lat=32, hid=256, lr=3e-4), dict(lat=64, hid=512)],
 "LSTMAE":  [dict(h=32), dict(h=64), dict(h=128), dict(h=64, lr=3e-4),
             dict(h=128, lr=3e-4), dict(h=256)],
 "ATRAN":   [dict(d=32, heads=2), dict(d=64, heads=4), dict(d=128, heads=4),
             dict(d=64, heads=8), dict(d=64, heads=4, lr=3e-4), dict(d=128, heads=8)],
 "TRANAD":  [dict(d=32, heads=2), dict(d=64, heads=4), dict(d=128, heads=4),
             dict(d=64, heads=8), dict(d=64, heads=4, lr=3e-4), dict(d=128, heads=8)],
 "SQ_full": [dict(d=48, heads=2, lam=0.3), dict(d=96, heads=4, lam=0.3),
             dict(d=144, heads=4, lam=0.3), dict(d=96, heads=4, lam=0.1),
             dict(d=96, heads=4, lam=1.0), dict(d=96, heads=4, lam=0.3, lr=3e-4)],
}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
for d in ("hpsearch", "scores6", "figs"): os.makedirs(f"{OUT}/{d}", exist_ok=True)
try:
    import pyarrow; _FMT = "parquet"
except Exception:
    _FMT = "pkl"
def load_df(p): return pd.read_parquet(f"{p}.{_FMT}") if _FMT == "parquet" else pd.read_pickle(f"{p}.{_FMT}")
RULE = "=" * 116
def H1(t): print("\n" + RULE + "\n" + t + "\n" + RULE)
def H2(t): print("\n" + t + "\n" + "-" * 116)

MAN = json.load(open(f"{OUT}/manifest.json")); FEATS = MAN["features"]
NF = len(FEATS); IDX = {f: i for i, f in enumerate(FEATS)}
STATIONS = sorted(os.path.basename(p).rsplit(".", 1)[0]
                  for p in glob.glob(f"{OUT}/features_inj/*.{_FMT}"))
PK = [s for s in STATIONS if s.startswith("pakistan__")]
NP_= [s for s in STATIONS if s.startswith("nepal__")]
ZM = [s for s in STATIONS if s.startswith("zambia__")]
FOLDS = {**{f"T1_{s}": (sorted(set(PK)-{s}), [s]) for s in PK},
         "T2_nepal": (PK, NP_), "T3_zambia": (PK, ZM)}
H1(f"PHASE 6  |  {DEVICE}  |  {len(FOLDS)} folds x {len(MODELS)} models x {len(SEEDS)} seeds")

# ============================================================================================
#  DATA
# ============================================================================================
_c = {}
def st_df(t):
    if t not in _c: _c[t] = load_df(f"{OUT}/features_inj/{t}")
    return _c[t]
def assemble(tags, parts):
    d = pd.concat([st_df(t) for t in tags]); return d[d.partition.isin(parts) & (d.segment >= 0)]
def starts_of(m):
    k = (m["station"].astype(str) + "|" + m["segment"].astype(str)).values
    o, s0 = [], 0
    for i in range(1, len(k)+1):
        if i == len(k) or k[i] != k[s0]: o.extend(range(s0, i-L+1)); s0 = i
    return np.asarray(o, np.int64)
class WinDS(torch.utils.data.Dataset):
    def __init__(s, X, st): s.X, s.s = X, st
    def __len__(s): return len(s.s)
    def __getitem__(s, i): return torch.from_numpy(s.X[s.s[i]:s.s[i]+L])
def loader(X, st, sh): return torch.utils.data.DataLoader(WinDS(X, st), batch_size=BATCH,
                                                          shuffle=sh, num_workers=0)
def prepare(trt, tet, cap, seed):
    rng = np.random.default_rng(seed)
    tr = assemble(trt, ["train"]); sc = RobustScaler(quantile_range=(5, 95)).fit(tr[FEATS].values)
    def pack(tags, parts):
        d = assemble(tags, parts)
        return np.clip(sc.transform(d[FEATS].values), -10, 10).astype("float32"), d
    Xtr, dtr = pack(trt, ["train"]); Xes, des = pack(trt, ["earlystop"])
    Xca, dca = pack(trt, ["calib"]); Xte, dte = pack(tet, ["test"])
    s_tr = starts_of(dtr)[::STRIDE]
    if len(s_tr) > cap: s_tr = np.sort(rng.choice(s_tr, cap, replace=False))
    return sc, (Xtr, s_tr), (Xes, starts_of(des)[::STRIDE]), (Xca, starts_of(dca), dca), \
           (Xte, starts_of(dte), dte), rng

# ============================================================================================
#  MODELS   -   score() returns (batch, L, components)
# ============================================================================================
class MLPAE(nn.Module):
    def __init__(s, lat=32, hid=256, **k):
        super().__init__()
        s.e = nn.Sequential(nn.Linear(L*NF, hid), nn.ReLU(), nn.Linear(hid, lat))
        s.d = nn.Sequential(nn.Linear(lat, hid), nn.ReLU(), nn.Linear(hid, L*NF))
    def forward(s, x): b = x.size(0); return s.d(s.e(x.reshape(b, -1))).reshape(b, L, NF)
    def loss(s, x, ep=1): return Fn.mse_loss(s(x), x)
    def score(s, x): return ((s(x)-x)**2).mean(-1, keepdim=True)

class LSTMAE(nn.Module):
    def __init__(s, h=64, **k):
        super().__init__(); s.en = nn.LSTM(NF, h, batch_first=True)
        s.de = nn.LSTM(h, h, batch_first=True); s.o = nn.Linear(h, NF)
    def forward(s, x):
        _, (hn, _) = s.en(x); y, _ = s.de(hn[-1].unsqueeze(1).repeat(1, L, 1)); return s.o(y)
    def loss(s, x, ep=1): return Fn.mse_loss(s(x), x)
    def score(s, x): return ((s(x)-x)**2).mean(-1, keepdim=True)

class ATRAN(nn.Module):
    def __init__(s, d=64, heads=4, **k):
        super().__init__(); s.inp = nn.Linear(NF, d); s.d = d
        s.q = nn.Linear(d, d); s.k = nn.Linear(d, d); s.v = nn.Linear(d, d)
        s.sig = nn.Linear(d, heads)
        s.ff = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, d*2), nn.GELU(), nn.Linear(d*2, d))
        s.out = nn.Linear(d, NF)
        s.register_buffer("dist", (torch.arange(L).view(-1,1)-torch.arange(L).view(1,-1)).float().abs())
    def assoc(s, x):
        z = s.inp(x); q, kk, v = s.q(z), s.k(z), s.v(z)
        att = torch.softmax(q @ kk.transpose(1,2)/math.sqrt(s.d), -1)
        sg = torch.sigmoid(s.sig(z)).mean(-1).unsqueeze(-1)*5 + 1e-2
        pr = torch.exp(-(s.dist.unsqueeze(0)**2)/(2*sg**2)); pr = pr/(pr.sum(-1, keepdim=True)+1e-8)
        return s.out(s.ff(att @ v)+z), att, pr
    def kl(s, a, b): return (a*(torch.log(a+1e-8)-torch.log(b+1e-8))).sum(-1)
    def loss(s, x, ep=1):
        r, a, p = s.assoc(x)
        return Fn.mse_loss(r, x) - 1e-2*(s.kl(p, a.detach())+s.kl(a.detach(), p)).mean()
    def score(s, x):
        r, a, p = s.assoc(x)
        w = torch.softmax(-(s.kl(p, a)+s.kl(a, p)), 1)*L
        return (w*((r-x)**2).mean(-1)).unsqueeze(-1)

class TRANAD(nn.Module):
    def __init__(s, d=64, heads=4, **k):
        super().__init__(); s.inp = nn.Linear(NF*2, d)
        s.enc = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d, heads, d*2, batch_first=True, dropout=0.0), 1)
        s.d1 = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Linear(d, NF))
        s.d2 = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Linear(d, NF))
    def two(s, x):
        o1 = s.d1(s.enc(s.inp(torch.cat([x, torch.zeros_like(x)], -1))))
        return o1, s.d2(s.enc(s.inp(torch.cat([x, (o1-x)**2], -1))))
    def loss(s, x, ep=1):
        o1, o2 = s.two(x); n = 1.0/max(ep, 1)
        return n*Fn.mse_loss(o1, x) + (1-n)*Fn.mse_loss(o2, x)
    def score(s, x):
        o1, o2 = s.two(x)
        return (0.5*(((o1-x)**2).mean(-1) + ((o2-x)**2).mean(-1))).unsqueeze(-1)

class SOLARQC(nn.Module):
    def __init__(s, center, scale, closure=True, pred=True, multi=True,
                 d=96, heads=4, lam=0.3, **k):
        super().__init__(); s.use_cl, s.use_pr, s.lam = closure, pred, lam
        rates = (1, 2, 4) if multi else (1, 1, 1)
        s.convs = nn.ModuleList([nn.Conv1d(NF, d//3, 3, padding=r, dilation=r) for r in rates])
        dm = (d//3)*3
        s.enc = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(dm, heads, d*2, batch_first=True, dropout=0.0), 2)
        s.rec = nn.Linear(dm, NF); s.pr = nn.Linear(dm, NF)
        s.register_buffer("c", torch.tensor(center, dtype=torch.float32))
        s.register_buffer("sc", torch.tensor(scale, dtype=torch.float32))
    def forward(s, x):
        h = s.enc(torch.cat([c(x.transpose(1,2)) for c in s.convs], 1).transpose(1,2))
        return s.rec(h), s.pr(h)
    def un(s, y, j): return y[..., j]*s.sc[j] + s.c[j]
    def cl_pen(s, rec, x):
        g, dn, dh = s.un(rec, IDX["ghi"]), s.un(rec, IDX["dni"]), s.un(rec, IDX["dhi"])
        cz = s.un(x, IDX["cosz"]).clamp(min=0); cs = s.un(x, IDX["ghi_cs"]).clamp(min=1.0)
        v = s.un(x, IDX["closure_valid"]) > 0.5
        return (((g-dh-dn*cz)/cs).abs()*v).sum()/(v.sum()+1.0)
    def loss(s, x, ep=1):
        rec, pr = s(x); l = Fn.mse_loss(rec, x)
        if s.use_pr: l = l + Fn.mse_loss(pr[:, :-1], x[:, 1:])
        if s.use_cl: l = l + s.lam*s.cl_pen(rec, x)
        return l
    def score(s, x):
        rec, pr = s(x)
        e_r = ((rec-x)**2).mean(-1)
        e_p = torch.zeros_like(e_r)
        if s.use_pr: e_p[:, 1:] = ((pr[:, :-1]-x[:, 1:])**2).mean(-1)
        e_c = s.un(x, IDX["r_closure"]).abs() if s.use_cl else torch.zeros_like(e_r)
        return torch.stack([e_r, e_p, e_c], -1)

CLS = dict(MLPAE=MLPAE, LSTMAE=LSTMAE, ATRAN=ATRAN, TRANAD=TRANAD)
def n_par(m): return sum(p.numel() for p in m.parameters())

def make(name, sc, cfg):
    cfg = {k: v for k, v in cfg.items() if k != "lr"}
    if name in VARIANTS: return SOLARQC(sc.center_, sc.scale_, **{**VARIANTS[name], **cfg})
    return CLS[name](**cfg)

def fit(name, sc, cfg, tr, es, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    m = make(name, sc, cfg).to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=cfg.get("lr", LR))
    tl, el = loader(*tr, True), loader(*es, False)
    best, bad, bs, used = np.inf, 0, None, 0
    for ep in range(1, EPOCHS+1):
        used = ep; m.train()
        for xb in tl:
            xb = xb.to(DEVICE); opt.zero_grad(); l = m.loss(xb, ep)
            l.backward(); nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
        m.eval(); t = n = 0.0
        with torch.no_grad():
            for xb in el:
                xb = xb.to(DEVICE); t += float(m.loss(xb, ep))*len(xb); n += len(xb)
        v = t/max(n, 1)
        if v < best-1e-6: best, bad = v, 0; bs = {k: q.detach().clone() for k, q in m.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE: break
    if bs: m.load_state_dict(bs)
    m.eval(); return m, n_par(m), used

@torch.no_grad()
def seq(m, X, st):
    o = [m.score(xb.to(DEVICE)).float().cpu().numpy() for xb in loader(X, st, False)]
    return np.concatenate(o) if o else np.empty((0, L, 1), "float32")

def if_seq(tr, others, rng, cfg):
    cfg = dict(cfg); cfg.setdefault("n_estimators", 100); cfg.setdefault("max_samples", "auto")
    Xtr, s_tr = tr
    A = np.stack([Xtr[i:i+L].ravel() for i in
                  (s_tr if len(s_tr) <= 50000 else rng.choice(s_tr, 50000, replace=False))])
    clf = IsolationForest(random_state=SEEDS[0], n_jobs=-1, **cfg).fit(A)
    out = []
    for X, st in others:
        r = []
        for i in range(0, len(st), 20000):
            r.append(-clf.score_samples(np.stack([X[j:j+L].ravel() for j in st[i:i+20000]])))
        v = np.concatenate(r) if r else np.empty((0,))
        out.append(np.repeat(v[:, None, None], L, 1).astype("float32"))
    return out

# ============================================================================================
#  SCORING UTILITIES
# ============================================================================================
def agg(S): return S[:, -1, :] if AGG == "last" else (S.max(1) if AGG == "max" else S.mean(1))
def fuse(Acal, Ates, w=None):
    if Acal.shape[1] == 1: return Acal[:, 0], Ates[:, 0]
    q = np.quantile(np.abs(Acal), 0.99, 0) + 1e-12
    w = np.ones(Acal.shape[1]) if w is None else np.asarray(w, float)
    return (Acal/q) @ w, (Ates/q) @ w
def sc_of(z, w=None):
    use = w if (z["cal"].ndim == 3 and z["cal"].shape[2] == 3) else None
    return fuse(agg(z["cal"]), agg(z["tes"]), use)
def thr_target(t, a, stn):
    o = np.empty(len(t))
    for s_ in np.unique(stn):
        m = stn == s_
        o[m] = np.quantile(t[m], 1-a) if m.sum() >= 500 else np.quantile(t, 1-a)
    return o
def thr_source(c, t, a): return np.full(len(t), np.quantile(c, 1-a))
def regimes(cz, kt): return np.digitize(cz, [0.05, 0.3, 0.6])*3 + np.digitize(kt, [0.35, 0.75])
def thr_regime(c, t, a, rc, rt):
    o = np.full(len(t), np.quantile(c, 1-a))
    for r in np.unique(rt):
        m = rc == r
        if m.sum() >= 200: o[rt == r] = np.quantile(c[m], 1-a)
    return o

def dilate(y, k):
    if k <= 0: return y
    z = y.copy()
    for s_ in range(1, k+1): z[s_:] |= y[:-s_]; z[:-s_] |= y[s_:]
    return z
def vus(y, s, roc=False, bufs=(0,1,2,4,6,9,12)):
    f = roc_auc_score if roc else average_precision_score
    v = [f(dilate(y, b), s) for b in bufs if 0 < dilate(y, b).sum() < len(y)]
    return float(np.mean(v)) if v else np.nan
def events(y):
    e, cur = [], None
    for i, v in enumerate(y):
        if v and cur is None: cur = i
        if (not v or i == len(y)-1) and cur is not None:
            e.append((cur, i if not v else i+1)); cur = None
    return e
def full_metrics(y, s, pred):
    y = y.astype(bool)
    tp = int((pred & y).sum()); fp = int((pred & ~y).sum())
    fn = int((~pred & y).sum()); tn = int((~pred & ~y).sum())
    p = tp/(tp+fp) if tp+fp else 0.0; r = tp/(tp+fn) if tp+fn else 0.0
    ev = events(y); hit = [any(pred[a:b]) for a, b in ev]
    rr = float(np.mean(hit)) if ev else 0.0
    # point-adjusted F1: reported ONLY for comparability with older papers that use it
    pa = pred.copy()
    for (a, b), h in zip(ev, hit):
        if h: pa[a:b] = True
    pt = int((pa & y).sum()); pf = int((pa & ~y).sum()); pn = int((~pa & y).sum())
    pp = pt/(pt+pf) if pt+pf else 0.0; prr = pt/(pt+pn) if pt+pn else 0.0
    # affiliation-style: mean normalised distance from each predicted point to the nearest event
    aff = np.nan
    if ev and pred.any():
        cent = np.array([(a+b)/2 for a, b in ev]); span = max(len(y), 1)
        d = np.abs(np.where(pred)[0][:, None] - cent[None, :]).min(1)/span
        aff = float(1 - d.mean())
    # detection delay in steps, averaged over detected events
    dly = [next((i-a for i in range(a, b) if pred[i]), None) for a, b in ev]
    dly = [d for d in dly if d is not None]
    return dict(precision=p, recall=r, f1=2*p*r/(p+r) if p+r else 0.0,
                range_recall=rr, range_f1=2*p*rr/(p+rr) if p+rr else 0.0,
                event_f1=2*pp*prr/(pp+prr) if pp+prr else 0.0,
                pa_f1=2*pp*prr/(pp+prr) if pp+prr else 0.0,
                affiliation=aff, delay_steps=float(np.mean(dly)) if dly else np.nan,
                far=fp/(fp+tn) if fp+tn else 0.0, tp=tp, fp=fp, fn=fn, tn=tn, n_events=len(ev))

# ============================================================================================
#  STAGE 1  -  HYPERPARAMETER SEARCH  (development fold only, equal budget)
# ============================================================================================
HP_CSV = f"{OUT}/hpsearch/table2.csv"
if DO_STAGE1:
    H1("STAGE 1  |  hyperparameter search on " + DEV_FOLD + "   (6 configs per model)")
    trt, tet = FOLDS[DEV_FOLD]
    sc, tr, es, (Xca, s_ca, dca), (Xte, s_te, dte), rng = prepare(
        trt, tet, SEARCH_TRAIN_WIN, SEEDS[0])
    Ydev = dte["inj_label"].values[s_te+L-1].astype(bool)
    stn_dev = dte["station"].values[s_te+L-1].astype(str)
    print(f"train {len(tr[1]):,} | calib {len(s_ca):,} | test {len(s_te):,} "
          f"(positives {int(Ydev.sum()):,})")
    rows = pd.read_csv(HP_CSV).to_dict("records") if os.path.exists(HP_CSV) else []
    seen = {(r["model"], r["config"]) for r in rows}
    for name, cfgs in GRID.items():
        H2(f"{name}")
        for i, cfg in enumerate(cfgs, 1):
            tag = json.dumps(cfg, sort_keys=True)
            if (name, tag) in seen: print(f"  [{i}/6] skip {tag}"); continue
            t1 = time.time()
            if name == "IF":
                c_, t_ = if_seq(tr, [(Xca, s_ca), (Xte, s_te)], rng, cfg); npar, ep = 0, 0
            else:
                m, npar, ep = fit(name, sc, cfg, tr, es, SEEDS[0])
                c_, t_ = seq(m, Xca, s_ca), seq(m, Xte, s_te)
                del m
                if DEVICE == "cuda": torch.cuda.empty_cache()
            cal, tes = fuse(agg(c_), agg(t_))
            v = vus(Ydev, tes); a = float(average_precision_score(Ydev, tes))
            f1 = max(full_metrics(Ydev, tes, tes > thr_target(tes, al, stn_dev))["f1"]
                     for al in ALPHAS)
            rows.append(dict(model=name, config=tag, vus_pr=round(v, 4), auc_pr=round(a, 4),
                             best_f1=round(f1, 4), params=npar, epochs=ep,
                             seconds=round(time.time()-t1, 1)))
            pd.DataFrame(rows).to_csv(HP_CSV, index=False)
            print(f"  [{i}/6] {tag:<50} VUS-PR {v:.4f}  AUC-PR {a:.4f}  "
                  f"F1 {f1:.4f}  params {npar:,}  {time.time()-t1:.0f}s")
    del Xca, Xte, dca, dte, tr, es; _c.clear(); gc.collect()

HP = pd.read_csv(HP_CSV) if os.path.exists(HP_CSV) else pd.DataFrame()
if len(HP):
    H2("TABLE 2  -  hyperparameter search (development fold only)")
    for n_ in GRID:
        s_ = HP[HP.model == n_].sort_values("vus_pr", ascending=False)
        if len(s_): print(f"\n{n_}"); print(s_[["config","vus_pr","auc_pr","best_f1",
                                                "params","epochs","seconds"]].to_string(index=False))
    BEST = {r.model: json.loads(r.config)
            for r in HP.sort_values("vus_pr", ascending=False).groupby("model").head(1).itertuples()}
    H2("Selected configuration, and how much the choice mattered")
    sen = HP.groupby("model").agg(best=("vus_pr","max"), worst=("vus_pr","min"),
                                  mean=("vus_pr","mean")).round(4)
    sen["spread"] = (sen.best-sen.worst).round(4)
    sen["chosen"] = [json.dumps(BEST.get(m, {})) for m in sen.index]
    print(sen.to_string())
    print("\n  A small spread means the ranking in the main table is not an artefact of tuning.")
else:
    BEST = {}
    print("no search results - falling back to built-in defaults")
def cfg_for(n): return dict(BEST.get("SQ_full" if n in VARIANTS else n, {}))
json.dump(BEST, open(f"{OUT}/best_hparams6.json", "w"), indent=1)

# ============================================================================================
#  STAGE 2  -  FINAL TRAINING: all folds, all models, all seeds
# ============================================================================================
if DO_STAGE2:
    H1(f"STAGE 2  |  final training   {len(FOLDS)} folds x {len(MODELS)} models x {len(SEEDS)} seeds")
    for fname, (trt, tet) in FOLDS.items():
        todo = [(m, sd) for m in MODELS for sd in SEEDS
                if not os.path.exists(f"{OUT}/scores6/{fname}__{m}__s{sd}.npz")]
        if not todo: print(f"[skip] {fname}"); continue
        t0 = time.time()
        sc, tr, es, (Xca, s_ca, dca), (Xte, s_te, dte), rng = prepare(
            trt, tet, MAX_TRAIN_WIN, SEEDS[0])
        e_ca, e_te = s_ca+L-1, s_te+L-1
        ctx = dict(y_syn=dte["inj_label"].values[e_te].astype("int8"),
                   y_real=(dte["scored_as"].values[e_te] == "anomaly").astype("int8"),
                   cls=dte["event_class"].values[e_te].astype(str),
                   typ=dte["inj_type"].values[e_te].astype(str),
                   stn=dte["station"].values[e_te].astype(str),
                   cosz_te=dte["cosz"].values[e_te].astype("float32"),
                   kt_te=dte["kt"].values[e_te].astype("float32"),
                   cosz_ca=dca["cosz"].values[e_ca].astype("float32"),
                   kt_ca=dca["kt"].values[e_ca].astype("float32"))
        print(f"\n[{fname}] tr={len(tr[1]):,} ca={len(s_ca):,} te={len(s_te):,} "
              f"syn+={int(ctx['y_syn'].sum()):,} real+={int(ctx['y_real'].sum()):,} "
              f"({time.time()-t0:.0f}s)")
        for name, sd in todo:
            t1 = time.time()
            if name == "IF":
                c_, t_ = if_seq(tr, [(Xca, s_ca), (Xte, s_te)], rng, cfg_for("IF"))
            else:
                m, _, _ = fit(name, sc, cfg_for(name), tr, es, sd)
                c_, t_ = seq(m, Xca, s_ca), seq(m, Xte, s_te)
                del m
                if DEVICE == "cuda": torch.cuda.empty_cache()
            np.savez_compressed(f"{OUT}/scores6/{fname}__{name}__s{sd}.npz",
                                cal=c_.astype("float32"), tes=t_.astype("float32"), **ctx)
            print(f"   {name:<13} seed {sd:<5} {t_.shape}  {time.time()-t1:.0f}s")
        del Xca, Xte, dca, dte, tr, es; _c.clear(); gc.collect()

# ============================================================================================
#  STAGE 3  -  ANALYSIS
# ============================================================================================
def parse(p):
    b = os.path.basename(p)[:-4]; fold, rest = b.rsplit("__", 2)[0], b.rsplit("__", 2)[1:]
    return fold, rest[0], int(rest[1][1:])
RUNS = [parse(p) for p in sorted(glob.glob(f"{OUT}/scores6/*.npz"))]
if not RUNS: raise SystemExit("no scores yet")
ALLF = sorted({f for f, _, _ in RUNS}); CONF = [f for f in ALLF if f != DEV_FOLD]
def load6(f, m, sd):
    p = f"{OUT}/scores6/{f}__{m}__s{sd}.npz"
    return np.load(p, allow_pickle=True) if os.path.exists(p) else None

H1("STAGE 3  |  analysis")
H2("A. Score weights and operating point, development fold only")
zd = load6(DEV_FOLD, "SQ_full", SEEDS[0])
BEST_W, BEST_A = (1, 1, 1), 0.06
if zd is not None and zd["cal"].ndim == 3 and zd["cal"].shape[2] == 3:
    y = zd["y_syn"].astype(bool); rows = []
    for w in WGRID:
        if sum(w) == 0: continue
        _, t_ = sc_of(zd, w)
        v = vus(y, t_)
        f1s = {a: full_metrics(y, t_, t_ > thr_target(t_, a, zd["stn"]))["f1"] for a in ALPHAS}
        ba = max(f1s, key=f1s.get)
        rows.append(dict(w=str(w), closure=w[2], vus_pr=round(v, 4),
                         alpha=ba, f1=round(f1s[ba], 4)))
    WD = pd.DataFrame(rows).sort_values("vus_pr", ascending=False)
    print(WD.to_string(index=False))
    BEST_W = eval(WD.iloc[0]["w"]); BEST_A = float(WD.iloc[0]["alpha"])
    on, off = WD[WD.closure > 0].vus_pr.max(), WD[WD.closure == 0].vus_pr.max()
    print(f"\n  best with closure {on:.4f} | without {off:.4f} | difference {on-off:+.4f}")
print(f">> frozen: weights={BEST_W}, alpha={BEST_A}, aggregation='{AGG}', calibration=target")
json.dump(dict(weights=list(BEST_W), alpha=BEST_A, agg=AGG, alpha_far=ALPHA_FAR,
               dev_fold=DEV_FOLD, seeds=SEEDS), open(f"{OUT}/frozen6.json", "w"), indent=1)

res = []
for f in CONF:
    for m in MODELS:
        for sd in SEEDS:
            z = load6(f, m, sd)
            if z is None: continue
            y = z["y_syn"].astype(bool); c_, t_ = sc_of(z, BEST_W)
            r = dict(fold=f, model=m, seed=sd, vus_pr=vus(y, t_), vus_roc=vus(y, t_, roc=True),
                     auc_pr=float(average_precision_score(y, t_)),
                     auc_roc=float(roc_auc_score(y, t_)))
            r.update(full_metrics(y, t_, t_ > thr_target(t_, BEST_A, z["stn"])))
            rc, rt = regimes(z["cosz_ca"], z["kt_ca"]), regimes(z["cosz_te"], z["kt_te"])
            r["far01_source"] = full_metrics(y, t_, t_ > thr_source(c_, t_, ALPHA_FAR))["far"]
            r["far01_regime"] = full_metrics(y, t_, t_ > thr_regime(c_, t_, ALPHA_FAR, rc, rt))["far"]
            r["far01_target"] = full_metrics(y, t_, t_ > thr_target(t_, ALPHA_FAR, z["stn"]))["far"]
            r["f1_source"] = full_metrics(y, t_, t_ > thr_source(c_, t_, BEST_A))["f1"]
            for ty in np.unique(z["typ"]):
                if ty != "none" and (z["typ"] == ty).sum() > 30:
                    yy = np.zeros(len(t_), bool); yy[z["typ"] == ty] = True
                    r["A_"+ty] = float(average_precision_score(yy, t_))
            yr = z["y_real"].astype(bool)
            if yr.sum() > 50:
                r["B_auc"] = float(average_precision_score(yr, t_)); r["B_vus"] = vus(yr, t_)
                for cl in np.unique(z["cls"]):
                    if cl != "none" and (z["cls"] == cl).sum() > 50:
                        yy = np.zeros(len(t_), bool); yy[z["cls"] == cl] = True
                        r["B_"+cl] = float(average_precision_score(yy, t_))
            res.append(r)
R = pd.DataFrame(res)

H2(f"TABLE 3  -  MAIN RESULTS   ({len(CONF)} folds x {R.seed.nunique()} seeds, "
   f"calibration=target, alpha={BEST_A})")
g = R.groupby("model")
main = pd.DataFrame(dict(
    vus_pr=g.vus_pr.mean(), vus_pr_sd=g.vus_pr.std(), vus_roc=g.vus_roc.mean(),
    auc_pr=g.auc_pr.mean(), auc_roc=g.auc_roc.mean(), f1=g.f1.mean(), f1_sd=g.f1.std(),
    range_f1=g.range_f1.mean(), event_f1=g.event_f1.mean(),
    affiliation=g.affiliation.mean(), delay=g.delay_steps.mean(),
    precision=g.precision.mean(), recall=g.recall.mean(), far=g.far.mean()
)).round(4).sort_values("vus_pr", ascending=False)
print(main.to_string())
print("\n  vus_pr is the primary measure. pa_f1 (point-adjusted F1) is reported separately")
print("  below for comparability with older work only: that protocol is known to let even a")
print("  random score look state of the art, so it is never used to rank anything here.")
print(g.pa_f1.mean().round(4).to_string())

H2("TABLE 4  -  ABLATION of the proposed model")
ab = sorted(c for c in R.columns if c.startswith("A_"))
print(R[R.model.isin(VARIANTS)].groupby("model")[
    ["vus_pr","auc_pr","f1","range_f1","event_f1","far"]+ab].mean().round(4).to_string())

def paired(metric, a="SQ_full", b="SQ_noclosure", n=20000, seed=0):
    p = R[R.model.isin([a, b])].pivot_table(index=["fold","seed"], columns="model",
                                            values=metric).dropna()
    if len(p) < 3 or a not in p or b not in p: return None
    d = (p[a]-p[b]).values; rng_ = np.random.default_rng(seed)
    bs = rng_.choice(d, (n, len(d)), replace=True).mean(1)
    perm = (rng_.choice([-1, 1], (n, len(d)))*d).mean(1)
    return dict(comparison=f"{a} - {b}", metric=metric, mean_diff=float(d.mean()),
                ci_lo=float(np.percentile(bs, 2.5)), ci_hi=float(np.percentile(bs, 97.5)),
                wins=int((d > 0).sum()), n=len(d),
                p_value=float((np.abs(perm) >= abs(d.mean())).mean()))
H2("TABLE 5  -  significance, paired over fold x seed")
mets = ["vus_pr","auc_pr","f1","range_f1","event_f1","A_spike","A_decoupling"]
sig = [x for x in
       [paired(m, "SQ_full", b) for b in ["SQ_noclosure","SQ_nopred","SQ_nomulti"] for m in mets]
       + [paired(m, "SQ_full", b) for b in BASE for m in ["vus_pr","f1","range_f1"]] if x]
SG = pd.DataFrame(sig)
print(SG.round(4).to_string(index=False))
print(f"\n  Paired over {len(CONF)}x{len(SEEDS)} = {len(CONF)*len(SEEDS)} observations, so the")
print(f"  smallest attainable permutation p-value is 2^-{len(CONF)*len(SEEDS)}.")
print("  ci_lo > 0 and p < 0.05 together mean the difference is real, not fold noise.")

H2(f"TABLE 6  -  NOVELTY 2, false-alarm guarantee at nominal {ALPHA_FAR}")
cc = R.groupby("model")[["far01_source","far01_regime","far01_target"]].mean().round(5)
cc.columns = ["source","regime","target"]
for c_ in cc.columns: cc[c_+"_x"] = (cc[c_]/ALPHA_FAR).round(1)
cc["f1_source"] = R.groupby("model").f1_source.mean().round(4)
cc["f1_target"] = R.groupby("model").f1.mean().round(4)
print(cc.to_string())

H2("TABLE 7  -  NOVELTY 3, real logged faults by class")
bc = sorted(c for c in R.columns if c.startswith("B_"))
print(R.groupby("model")[bc].mean().round(4).to_string() if bc else "not enough real labels")

H2("TABLE 8  -  per fold VUS-PR (mean over seeds)")
print(R.pivot_table(index="fold", columns="model", values="vus_pr").round(4).to_string())

H2("TABLE 9  -  confusion matrix")
cm = R.groupby("model")[["tp","fp","fn","tn"]].mean().round(0).astype(int)
cm.columns = ["TP","FP","FN","TN"]; cm["events"] = R.groupby("model").n_events.mean().astype(int)
print(cm.to_string())

H2("TABLE 10  -  seed stability")
print(R.groupby("model").agg(vus_mean=("vus_pr","mean"), vus_sd=("vus_pr","std"),
                             f1_mean=("f1","mean"), f1_sd=("f1","std")).round(4).to_string())

R.to_csv(f"{OUT}/phase6_results.csv", index=False)
SG.to_csv(f"{OUT}/phase6_significance.csv", index=False)
H1("PHASE 6 COMPLETE")
print(f"Table 2 -> {HP_CSV}")
print(f"Tables 3-10 -> {OUT}/phase6_results.csv")
print(f"significance -> {OUT}/phase6_significance.csv")
print(f"frozen -> {OUT}/frozen6.json   scores -> {OUT}/scores6/")
print("\nEverything downstream reads scores6/, so no question needs retraining again.")


PHASE 6  |  cuda  |  11 folds x 9 models x 3 seeds

STAGE 1  |  hyperparameter search on T1_pakistan__quetta   (6 configs per model)
train 150,000 | calib 165,847 | test 8,505 (positives 678)

IF
--------------------------------------------------------------------------------------------------------------------
  [1/6] {"n_estimators": 50}                               VUS-PR 0.2913  AUC-PR 0.2009  F1 0.2564  params 0  1s
  [2/6] {"n_estimators": 100}                              VUS-PR 0.2935  AUC-PR 0.2207  F1 0.2958  params 0  2s
  [3/6] {"n_estimators": 200}                              VUS-PR 0.2557  AUC-PR 0.1613  F1 0.2625  params 0  3s
  [4/6] {"n_estimators": 300}                              VUS-PR 0.2537  AUC-PR 0.1645  F1 0.2401  params 0  5s
  [5/6] {"max_samples": 256, "n_estimators": 100}          VUS-PR 0.2378  AUC-PR 0.1454  F1 0.2331  params 0  2s
  [6/6] {"max_samples": 1024, "n_estimators": 200}         VUS-PR 0.2832  AUC-PR 0.1923  F1 0.2668  params 0  4s

MLPAE
-

In [6]:
# ============================================================================================
#  SolarQC-Net  |  PHASE 7  -  FIGURES + PAPER-READY TABLES        (NO GPU, ~5 min)
#
#  Everything here is recomputed from /kaggle/working/scores6/, so nothing is retrained and
#  no number can drift away from what Phase 6 reported.
#
#  Produces, in /kaggle/working/paper/:
#     fig1_station_map.png        the 20-station network across three countries
#     fig3_pr_curves.png          precision-recall curves, all models
#     fig4_perfold_box.png        per-fold VUS-PR distribution
#     fig5_calibration.png        NOVELTY 2: the guarantee fails, target calibration fixes it
#     fig6_ablation.png           NOVELTY 1: closure head, per anomaly type, with CIs
#     fig7_alpha_sweep.png        operating point: why alpha = 0.01 capped recall
#     fig8_pa_vs_real.png         why point-adjusted F1 must not be used to rank
#     table3.tex ... table7.tex   LaTeX versions of the main tables
#     all_metrics.csv             every metric, every model, for the appendix
#
#  fig2_closure.png already exists from Phase 2 and is the paper's key figure.
# ============================================================================================

import os, json, glob, warnings
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from sklearn.metrics import precision_recall_curve, average_precision_score

warnings.filterwarnings("ignore")
pd.set_option("display.width", 240)

OUT   = "/kaggle/working"
PAPER = f"{OUT}/paper"; os.makedirs(PAPER, exist_ok=True)
L, ALPHA_FAR = 12, 0.01
ALPHAS = [0.01, 0.02, 0.04, 0.06, 0.08, 0.10, 0.12, 0.15, 0.20]
CFG = json.load(open(f"{OUT}/frozen6.json"))
W, BEST_A, AGG, DEV_FOLD, SEEDS = (tuple(CFG["weights"]), CFG["alpha"], CFG["agg"],
                                   CFG["dev_fold"], CFG["seeds"])
ORDER = ["SQ_full", "SQ_noclosure", "SQ_nomulti", "SQ_nopred",
         "MLPAE", "LSTMAE", "TRANAD", "ATRAN", "IF"]
NICE  = {"SQ_full": "SolarQC-Net (ours)", "SQ_noclosure": "  w/o closure head",
         "SQ_nomulti": "  w/o multiscale", "SQ_nopred": "  w/o prediction head",
         "MLPAE": "MLP autoencoder", "LSTMAE": "LSTM autoencoder",
         "TRANAD": "TranAD", "ATRAN": "Anomaly Transformer", "IF": "Isolation Forest"}
COL   = {"SQ_full": "#b2182b", "SQ_noclosure": "#ef8a62", "SQ_nomulti": "#fddbc7",
         "SQ_nopred": "#d6604d", "MLPAE": "#2166ac", "LSTMAE": "#4393c3",
         "TRANAD": "#92c5de", "ATRAN": "#7f7f7f", "IF": "#bbbbbb"}
plt.rcParams.update({"font.size": 9, "axes.grid": True, "grid.alpha": .25,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "figure.dpi": 150, "savefig.bbox": "tight"})
RULE = "=" * 100
def H1(t): print("\n" + RULE + "\n" + t + "\n" + RULE)

def load(f, m, sd):
    p = f"{OUT}/scores6/{f}__{m}__s{sd}.npz"
    return np.load(p, allow_pickle=True) if os.path.exists(p) else None
ALLF = sorted({os.path.basename(p)[:-4].rsplit("__", 2)[0] for p in glob.glob(f"{OUT}/scores6/*.npz")})
CONF = [f for f in ALLF if f != DEV_FOLD]
H1(f"PHASE 7  |  {len(CONF)} confirmatory folds, weights={W}, alpha={BEST_A}")

def agg(S): return S[:, -1, :] if AGG == "last" else (S.max(1) if AGG == "max" else S.mean(1))
def fuse(A, B, w):
    if A.shape[1] == 1: return A[:, 0], B[:, 0]
    q = np.quantile(np.abs(A), 0.99, 0) + 1e-12
    return (A/q) @ np.asarray(w, float), (B/q) @ np.asarray(w, float)
def sc_of(z):
    w = W if (z["cal"].ndim == 3 and z["cal"].shape[2] == 3) else None
    return fuse(agg(z["cal"]), agg(z["tes"]), w if w is not None else [1])
def thr_target(t, a, stn):
    o = np.empty(len(t))
    for s_ in np.unique(stn):
        m = stn == s_
        o[m] = np.quantile(t[m], 1-a) if m.sum() >= 500 else np.quantile(t, 1-a)
    return o
def thr_source(c, t, a): return np.full(len(t), np.quantile(c, 1-a))
def f1_at(y, t, pred):
    tp = int((pred & y).sum()); fp = int((pred & ~y).sum()); fn = int((~pred & y).sum())
    p = tp/(tp+fp) if tp+fp else 0; r = tp/(tp+fn) if tp+fn else 0
    return (2*p*r/(p+r) if p+r else 0), r

R = pd.read_csv(f"{OUT}/phase6_results.csv")
try:
    SG = pd.read_csv(f"{OUT}/phase6_significance.csv")
    if not len(SG) or "comparison" not in SG.columns: raise ValueError
except Exception:
    SG = pd.DataFrame(columns=["comparison","metric","mean_diff","ci_lo","ci_hi",
                               "wins","n","p_value"])
    print("note: no significance file - figure 6 (left) and table 5 will be skipped")
MODELS = [m for m in ORDER if m in R.model.unique()]

# ============================================================================================
#  FIGURE 1  -  the station network
# ============================================================================================
COORD = {
 "pakistan__bahawalpur": (29.32, 71.81), "pakistan__hyderabad": (25.413, 68.260),
 "pakistan__islamabad": (33.642, 72.984), "pakistan__karachi": (24.933, 67.112),
 "pakistan__khuzdar": (27.818, 66.629), "pakistan__lahore": (31.695, 74.244),
 "pakistan__multan": (30.165, 71.498), "pakistan__peshawar": (34.002, 71.485),
 "pakistan__quetta": (30.271, 66.940),
 "nepal__dharan": (26.793, 87.293), "nepal__jumla": (29.272, 82.194),
 "nepal__kathmandu": (27.682, 85.319), "nepal__lumle": (28.297, 83.818),
 "nepal__nepalgunj": (28.113, 81.589),
 "zambia__chilanga": (-15.548, 28.248), "zambia__choma": (-16.838, 27.070),
 "zambia__kaoma": (-14.839, 24.931), "zambia__kasama": (-10.172, 31.226),
 "zambia__lusaka": (-15.395, 28.337), "zambia__mutanda": (-12.423, 26.215)}
CC = {"pakistan": "#b2182b", "nepal": "#2166ac", "zambia": "#1b7837"}
fig, ax = plt.subplots(figsize=(7.2, 4.6))
for tag, (la, lo) in COORD.items():
    c = tag.split("__")[0]
    ax.scatter(lo, la, s=70, c=CC[c], edgecolor="k", lw=.5, zorder=3)
    ax.annotate(tag.split("__")[1], (lo, la), fontsize=6.5,
                xytext=(4, 3), textcoords="offset points")
ax.axhline(0, color="k", lw=.8, ls="--", alpha=.6)
ax.text(20, 0.8, "equator", fontsize=7, alpha=.7)
ax.set_xlabel("longitude (deg E)"); ax.set_ylabel("latitude (deg N)")
ax.set_title("Twenty ESMAP solar measurement stations across three countries\n"
             "training on Pakistan; Nepal and Zambia are held out entirely", fontsize=10)
# Zambia sits in the lower left, so the legend goes outside the axes or it hides the points
ax.legend(handles=[Line2D([], [], marker="o", ls="", mfc=v, mec="k",
                          label=f"{k.title()} ({sum(1 for t in COORD if t.startswith(k))})")
                   for k, v in CC.items()], fontsize=8,
          loc="upper left", bbox_to_anchor=(1.01, 1.0), frameon=False)
lats = [v[0] for v in COORD.values()]; lons = [v[1] for v in COORD.values()]
ax.set_xlim(min(lons)-4, max(lons)+4); ax.set_ylim(min(lats)-4, max(lats)+5)
fig.savefig(f"{PAPER}/fig1_station_map.png"); plt.close(fig)
print("fig1 station map")

# ============================================================================================
#  FIGURE 3  -  precision-recall curves
# ============================================================================================
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2))
pool = {}
for m in MODELS:
    ys, ss = [], []
    for f in CONF:
        z = load(f, m, SEEDS[0])
        if z is None: continue
        _, t = sc_of(z)
        t = (t - np.median(t)) / (np.quantile(t, .99) - np.median(t) + 1e-9)  # per-fold scale
        ys.append(z["y_syn"].astype(bool)); ss.append(t)
    if not ys: continue
    y, s = np.concatenate(ys), np.concatenate(ss)
    pool[m] = (y, s)
    pr, rc, _ = precision_recall_curve(y, s)
    ap = average_precision_score(y, s)
    lw = 2.2 if m == "SQ_full" else 1.1
    axes[0].plot(rc, pr, color=COL[m], lw=lw, label=f"{NICE[m]} ({ap:.3f})")
base = float(np.mean(pool["SQ_full"][0])) if "SQ_full" in pool else 0.08
axes[0].axhline(base, color="k", ls=":", lw=.9)
axes[0].text(.62, base+.01, f"random ({base:.3f})", fontsize=7)
axes[0].set_xlabel("recall"); axes[0].set_ylabel("precision")
axes[0].set_title("Precision-recall, pooled over held-out folds", fontsize=10)
axes[0].legend(fontsize=6.6, loc="upper right")

sub = R[R.model.isin(MODELS)]
xs = [sub[sub.model == m].auc_pr.mean() for m in MODELS]
ys_ = [sub[sub.model == m].f1.mean() for m in MODELS]
for m, x, y_ in zip(MODELS, xs, ys_):
    axes[1].scatter(x, y_, s=110 if m == "SQ_full" else 55, c=COL[m],
                    edgecolor="k", lw=.6, zorder=3)
    axes[1].annotate(NICE[m].strip(), (x, y_), fontsize=6.8,
                     xytext=(5, -3), textcoords="offset points")
axes[1].set_xlabel("AUC-PR"); axes[1].set_ylabel("F1")
axes[1].set_title("AUC-PR against F1 (mean over 10 folds x 3 seeds)", fontsize=10)
fig.tight_layout(); fig.savefig(f"{PAPER}/fig3_pr_curves.png"); plt.close(fig)
print("fig3 PR curves")

# ============================================================================================
#  FIGURE 4  -  per-fold distribution
# ============================================================================================
fig, ax = plt.subplots(figsize=(8.2, 4.0))
data = [R[R.model == m].vus_pr.values for m in MODELS]
bp = ax.boxplot(data, patch_artist=True, widths=.62,
                medianprops=dict(color="k", lw=1.3), showfliers=False)
for p, m in zip(bp["boxes"], MODELS):
    p.set_facecolor(COL[m]); p.set_alpha(.85); p.set_edgecolor("k"); p.set_linewidth(.6)
for i, m in enumerate(MODELS, 1):
    v = R[R.model == m].vus_pr.values
    ax.scatter(np.full(len(v), i) + np.random.uniform(-.13, .13, len(v)), v,
               s=7, c="k", alpha=.35, zorder=3)
ax.set_xticks(range(1, len(MODELS)+1))
ax.set_xticklabels([NICE[m].strip() for m in MODELS], rotation=22, ha="right", fontsize=8)
ax.set_ylabel("VUS-PR")
ax.set_title("VUS-PR across 10 held-out folds x 3 seeds", fontsize=10)
fig.savefig(f"{PAPER}/fig4_perfold_box.png"); plt.close(fig)
print("fig4 per-fold box")

# ============================================================================================
#  FIGURE 5  -  NOVELTY 2
# ============================================================================================
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.0))
x = np.arange(len(MODELS)); w_ = .26
for k, (col, lab) in enumerate([("far01_source", "source calibration"),
                                ("far01_regime", "regime-conditional"),
                                ("far01_target", "target-domain (ours)")]):
    v = [R[R.model == m][col].mean()/ALPHA_FAR for m in MODELS]
    axes[0].bar(x + (k-1)*w_, v, w_, label=lab,
                color=["#999999", "#d95f02", "#1b7837"][k], edgecolor="k", lw=.4)
axes[0].axhline(1, color="k", lw=1.2, ls="--")
axes[0].text(len(MODELS)-3.4, 1.4, "nominal (1x)", fontsize=7.5)
axes[0].set_yscale("log"); axes[0].set_ylabel("empirical FAR / nominal 0.01")
axes[0].set_xticks(x); axes[0].set_xticklabels([NICE[m].strip() for m in MODELS],
                                               rotation=22, ha="right", fontsize=7.5)
axes[0].set_title("Does the false-alarm guarantee survive a new station?", fontsize=10)
axes[0].legend(fontsize=7.5)

for k, (col, lab, c) in enumerate([("f1_source", "source calibration", "#999999"),
                                   ("f1", "target-domain (ours)", "#1b7837")]):
    v = [R[R.model == m][col].mean() for m in MODELS]
    axes[1].bar(x + (k-.5)*.38, v, .38, label=lab, color=c, edgecolor="k", lw=.4)
axes[1].set_ylabel("F1"); axes[1].set_xticks(x)
axes[1].set_xticklabels([NICE[m].strip() for m in MODELS], rotation=22, ha="right", fontsize=7.5)
axes[1].set_title(f"Detection quality under each calibration (alpha = {BEST_A})", fontsize=10)
axes[1].legend(fontsize=7.5)
fig.tight_layout(); fig.savefig(f"{PAPER}/fig5_calibration.png"); plt.close(fig)
print("fig5 calibration transfer")

# ============================================================================================
#  FIGURE 6  -  NOVELTY 1
# ============================================================================================
mets = ["auc_pr", "f1", "range_f1", "event_f1", "A_spike", "A_decoupling"]
lab  = dict(auc_pr="AUC-PR", f1="F1", range_f1="Range-F1", event_f1="Event-F1",
            A_spike="spike", A_decoupling="tracker decoupling")
sg = SG[SG.comparison == "SQ_full - SQ_noclosure"]
sg = sg.set_index("metric") if len(sg) else sg
have = [m for m in mets if len(sg) and m in sg.index]
tys = sorted(c for c in R.columns if c.startswith("A_"))

if not have and not tys:
    print("fig6 skipped - no ablation data available")
else:
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.0))
    if have:
        y = np.arange(len(have))
        md = np.array([sg.loc[m, "mean_diff"] for m in have])
        lo = md - np.array([sg.loc[m, "ci_lo"] for m in have])
        hi = np.array([sg.loc[m, "ci_hi"] for m in have]) - md
        cols = ["#1b7837" if (sg.loc[m, "p_value"] < .05 and sg.loc[m, "ci_lo"] > 0)
                else ("#b2182b" if sg.loc[m, "ci_hi"] < 0 else "#999999") for m in have]
        axes[0].barh(y, md, xerr=[lo, hi], color=cols, edgecolor="k", lw=.5,
                     error_kw=dict(lw=1.1, capsize=3))
        axes[0].axvline(0, color="k", lw=1.1)
        axes[0].set_yticks(y); axes[0].set_yticklabels([lab[m] for m in have], fontsize=8)
        for i, m in enumerate(have):
            axes[0].text(md.max()*1.08, i,
                         f"p={sg.loc[m,'p_value']:.4f}  {int(sg.loc[m,'wins'])}/{int(sg.loc[m,'n'])}",
                         va="center", fontsize=6.8)
        axes[0].set_xlabel("SolarQC-Net minus the ablated model (95% CI)")
        axes[0].set_title("Contribution of the physics closure head", fontsize=10)
        axes[0].text(0.02, -0.22, "green: improvement holds across folds   "
                     "red: the ablated model is better   grey: not distinguishable",
                     transform=axes[0].transAxes, fontsize=6.5)
    else:
        axes[0].axis("off"); axes[0].text(.5, .5, "no significance data", ha="center")

    if tys:
        xx = np.arange(len(tys))
        for k, m in enumerate([x for x in ("SQ_full", "SQ_noclosure") if x in R.model.unique()]):
            v = [R[R.model == m][t].mean() for t in tys]
            axes[1].bar(xx + (k-.5)*.4, v, .4, label=NICE[m].strip(),
                        color=["#b2182b", "#999999"][k], edgecolor="k", lw=.4)
        axes[1].set_xticks(xx)
        axes[1].set_xticklabels([t[2:] for t in tys], rotation=22, ha="right", fontsize=8)
        axes[1].set_ylabel("AUC-PR"); axes[1].legend(fontsize=8)
        axes[1].set_title("Detection quality by injected anomaly type", fontsize=10)
    fig.tight_layout(); fig.savefig(f"{PAPER}/fig6_ablation.png"); plt.close(fig)
print("fig6 ablation")

# ============================================================================================
#  FIGURE 7  -  operating point
# ============================================================================================
rows = []
for m in MODELS:
    for f in CONF:
        z = load(f, m, SEEDS[0])
        if z is None: continue
        c_, t_ = sc_of(z); y = z["y_syn"].astype(bool)
        for a in ALPHAS:
            for mode in ("source", "target"):
                thr = (thr_source(c_, t_, a) if mode == "source"
                       else thr_target(t_, a, z["stn"]))
                f1, rc = f1_at(y, t_, t_ > thr)
                rows.append(dict(model=m, fold=f, alpha=a, mode=mode, f1=f1, recall=rc))
SW = pd.DataFrame(rows)
show = [m for m in ["SQ_full", "MLPAE", "TRANAD", "IF"] if m in set(SW.model)] if len(SW) else []
if not show:
    print("fig7 skipped - no per-window scores found for the selected models")
else:
  fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.0), sharex=True)
  for mode, ls in (("target", "-"), ("source", "--")):
    for m in show:
        d = SW[(SW.model == m) & (SW["mode"] == mode)].groupby("alpha")
        f1m, rcm = d.f1.mean(), d.recall.mean()
        axes[0].plot(f1m.index, f1m.values, ls, color=COL[m],
                     lw=1.8 if m == "SQ_full" else 1.0,
                     label=f"{NICE[m].strip()} ({mode})" if mode == "target" else None)
        axes[1].plot(rcm.index, rcm.values, ls, color=COL[m],
                     lw=1.8 if m == "SQ_full" else 1.0)
  for a_ in axes:
      a_.axvline(0.01, color="#999999", lw=1, ls=":")
      a_.axvline(BEST_A, color="#1b7837", lw=1.2, ls=":")
      a_.set_xlabel("alpha (nominal false-alarm rate)")
  axes[0].text(0.012, axes[0].get_ylim()[1]*.35, "alpha = 0.01", rotation=90, fontsize=7)
  axes[0].text(BEST_A+.003, axes[0].get_ylim()[1]*.35, f"chosen {BEST_A}", rotation=90,
               fontsize=7, color="#1b7837")
  axes[0].set_ylabel("F1"); axes[1].set_ylabel("recall")
  axes[0].set_title("Solid: target calibration. Dashed: source calibration", fontsize=9.5)
  axes[1].set_title("At alpha = 0.01 only 1% of points may fire while 8% are anomalous,\n"
                    "so recall is capped by construction", fontsize=9.5)
  axes[0].legend(fontsize=7)
  fig.tight_layout(); fig.savefig(f"{PAPER}/fig7_alpha_sweep.png"); plt.close(fig)
  print("fig7 alpha sweep")

# ============================================================================================
#  FIGURE 8  -  why point adjustment must not rank anything
# ============================================================================================
fig, ax = plt.subplots(figsize=(6.4, 4.2))
for m in MODELS:
    d = R[R.model == m]
    ax.scatter(d.f1.mean(), d.pa_f1.mean(), s=120 if m == "SQ_full" else 60,
               c=COL[m], edgecolor="k", lw=.6, zorder=3)
    ax.annotate(NICE[m].strip(), (d.f1.mean(), d.pa_f1.mean()), fontsize=7,
                xytext=(6, -3), textcoords="offset points")
lims = [min(R.f1.mean(), R.pa_f1.min())*.8, 1.0]
ax.plot(lims, lims, "k--", lw=.9)
ax.set_xlabel("F1 (no point adjustment)"); ax.set_ylabel("point-adjusted F1")
ax.set_title("Point adjustment inflates every score and reorders the ranking:\n"
             "the weakest detector here becomes the apparent winner", fontsize=9.5)
fig.savefig(f"{PAPER}/fig8_pa_vs_real.png"); plt.close(fig)
print("fig8 point-adjustment")

# ============================================================================================
#  PAPER TABLES
# ============================================================================================
H1("PAPER TABLES")
g = R.groupby("model")
T3 = pd.DataFrame(dict(
    VUS_PR=g.vus_pr.mean(), sd=g.vus_pr.std(), AUC_PR=g.auc_pr.mean(),
    AUC_ROC=g.auc_roc.mean(), F1=g.f1.mean(), Range_F1=g.range_f1.mean(),
    Event_F1=g.event_f1.mean(), Precision=g.precision.mean(), Recall=g.recall.mean(),
    FAR=g.far.mean(), Delay=g.delay_steps.mean())).round(4).reindex(MODELS)
T3.index = [NICE[m] for m in T3.index]
print("\nTABLE 3  main results"); print(T3.to_string())

T4 = R[R.model.isin([m for m in MODELS if m.startswith("SQ")])].groupby("model")[
    ["vus_pr","auc_pr","f1","range_f1","event_f1","far"] +
    sorted(c for c in R.columns if c.startswith("A_"))].mean().round(4)
T4.index = [NICE[m] for m in T4.index]
print("\nTABLE 4  ablation"); print(T4.to_string())

T5 = SG[SG.comparison.str.startswith("SQ_full - SQ_")][
    ["comparison","metric","mean_diff","ci_lo","ci_hi","wins","n","p_value"]].round(4)
print("\nTABLE 5  significance"); print(T5.to_string(index=False))

T6 = R.groupby("model")[["far01_source","far01_regime","far01_target","f1_source","f1"]].mean()
T6.columns = ["FAR_source","FAR_regime","FAR_target","F1_source","F1_target"]
for c in ("FAR_source","FAR_regime","FAR_target"): T6[c+"_x"] = (T6[c]/ALPHA_FAR).round(1)
T6 = T6.round(5).reindex(MODELS); T6.index = [NICE[m] for m in T6.index]
print("\nTABLE 6  calibration transfer"); print(T6.to_string())

bc = sorted(c for c in R.columns if c.startswith("B_"))
T7 = R.groupby("model")[bc].mean().round(4).reindex(MODELS) if bc else pd.DataFrame()
if len(T7):
    T7.index = [NICE[m] for m in T7.index]
    print("\nTABLE 7  real logged faults"); print(T7.to_string())

for n, t in [("table3", T3), ("table4", T4), ("table5", T5), ("table6", T6), ("table7", T7)]:
    if len(t):
        t.to_csv(f"{PAPER}/{n}.csv")
        try:
            open(f"{PAPER}/{n}.tex", "w").write(
                t.to_latex(float_format="%.4f", escape=True, index=(n != "table5")))
        except Exception as e:
            print(f"  latex export skipped for {n}: {e}")
R.to_csv(f"{PAPER}/all_metrics.csv", index=False)
SW.to_csv(f"{PAPER}/alpha_sweep.csv", index=False)

H1("PHASE 7 COMPLETE")
for p in sorted(glob.glob(f"{PAPER}/*")): print("  " + p)
print("\nAll figures come from the saved scores, so they cannot disagree with the tables.")
print("fig2_closure.png from Phase 2 is the key figure and already exists in /figures/.")


PHASE 7  |  10 confirmatory folds, weights=(1, 1, 1), alpha=0.06
fig1 station map
fig3 PR curves
fig4 per-fold box
fig5 calibration transfer
fig6 ablation
fig7 alpha sweep
fig8 point-adjustment

PAPER TABLES

TABLE 3  main results
                       VUS_PR      sd  AUC_PR  AUC_ROC      F1  Range_F1  Event_F1  Precision  Recall     FAR    Delay
SolarQC-Net (ours)     0.5372  0.0299  0.5086   0.7468  0.5120    0.7129    0.7220     0.5870  0.4548  0.0269   1.8345
  w/o closure head     0.5393  0.0304  0.5021   0.7557  0.4990    0.6989    0.6975     0.5720  0.4433  0.0279   1.7519
  w/o multiscale       0.5360  0.0300  0.5067   0.7509  0.5113    0.7117    0.7185     0.5862  0.4541  0.0269   1.8332
  w/o prediction head  0.5250  0.0296  0.5088   0.7443  0.5102    0.7090    0.7209     0.5849  0.4531  0.0270   1.8829
MLP autoencoder        0.5369  0.0293  0.4611   0.7496  0.4620    0.6605    0.6630     0.5294  0.4104  0.0306   1.6693
LSTM autoencoder       0.5238  0.0327  0.4603   0.7518

In [7]:
# ============================================================================================
#  SolarQC-Net  |  PHASE 8  -  REVISION ANALYSES        *** run after Phase 6, GPU ON ***
#
#  One cell that answers the reviewers' training/analysis points. Everything except the
#  lambda sweep (R7) and the extended baseline search (R8) is recomputed from the per-window
#  scores already written to /kaggle/working/scores6/, so nothing is retrained needlessly.
#
#    R1  rule-based closure-residual baseline                      (reviewer: no such baseline)
#    R2  BSRN operational quality-control baseline                 (reviewer: no QC comparison)
#    R3  real logged faults: per class, per station, event level   (reviewer: analysis too thin)
#    R4  operating point at alpha = 0.01 / 0.02 and low prevalence (reviewer: alpha ~ prevalence)
#    R5  contaminated target-calibration window                    (reviewer: what if faulty?)
#    R6  score aggregation: last vs max vs mean, all folds         (reviewer: why last sample?)
#    R7  sensitivity: elevation gate, G_cs gate, normalisation, clip, lambda
#    R8  extended baseline search for TranAD and Anomaly Transformer (fairness)
#    R9  Holm-Bonferroni correction of the significance table
#    R10 inference cost per station-day, parameter counts, GPU name
#
#  Output: /kaggle/working/revision/*.csv and a printed summary for each point.
# ============================================================================================

import os, re, gc, json, glob, time, math, warnings, itertools
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as Fn
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import average_precision_score, roc_auc_score

warnings.filterwarnings("ignore")
pd.set_option("display.width", 250); pd.set_option("display.max_rows", 400)

OUT   = "/kaggle/working"
REV   = f"{OUT}/revision"; os.makedirs(REV, exist_ok=True)
CFG   = json.load(open(f"{OUT}/frozen6.json"))
W, ALPHA, AGG, DEV_FOLD, SEEDS = (tuple(CFG["weights"]), CFG["alpha"], CFG["agg"],
                                  CFG["dev_fold"], CFG["seeds"])
ALPHA_FAR = CFG.get("alpha_far", 0.01)
L, F_ = 12, 30
S0 = 1361.0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RUN_LAMBDA   = True      # R7 lambda sweep   (GPU, ~6 min)
RUN_BASELINE = True      # R8 extended grid  (GPU, ~9 min)
try:
    import pyarrow; _FMT = "parquet"
except Exception:
    _FMT = "pkl"
def load_df(p): return pd.read_parquet(f"{p}.{_FMT}") if _FMT == "parquet" else pd.read_pickle(f"{p}.{_FMT}")
RULE = "=" * 112
def H1(t): print("\n" + RULE + "\n" + t + "\n" + RULE)
def H2(t): print("\n" + t + "\n" + "-" * 112)

MAN   = json.load(open(f"{OUT}/manifest.json")); FEATS = MAN["features"]
IDX   = {f: i for i, f in enumerate(FEATS)}
NICE  = {"SQ_full": "SolarQC-Net", "SQ_noclosure": "w/o closure", "SQ_nomulti": "w/o multiscale",
         "SQ_nopred": "w/o prediction", "MLPAE": "MLP-AE", "LSTMAE": "LSTM-AE",
         "TRANAD": "TranAD", "ATRAN": "AnomalyTrans", "IF": "IsolationForest"}
MODELS = list(NICE)
STATIONS = sorted(os.path.basename(p).rsplit(".", 1)[0]
                  for p in glob.glob(f"{OUT}/features_inj/*.{_FMT}"))
PK = [s for s in STATIONS if s.startswith("pakistan__")]
NP_= [s for s in STATIONS if s.startswith("nepal__")]
ZM = [s for s in STATIONS if s.startswith("zambia__")]
FOLDS = {**{f"T1_{s}": (sorted(set(PK)-{s}), [s]) for s in PK},
         "T2_nepal": (PK, NP_), "T3_zambia": (PK, ZM)}
ALLF = sorted({os.path.basename(p)[:-4].rsplit("__", 2)[0] for p in glob.glob(f"{OUT}/scores6/*.npz")})
CONF = [f for f in ALLF if f != DEV_FOLD]
H1(f"PHASE 8  |  revision analyses  |  {DEVICE}  |  {len(CONF)} confirmatory folds")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")

# ============================================================================================
#  SHARED HELPERS  (identical to Phase 6, so every number stays comparable)
# ============================================================================================
def load6(f, m, sd):
    p = f"{OUT}/scores6/{f}__{m}__s{sd}.npz"
    return np.load(p, allow_pickle=True) if os.path.exists(p) else None
def agg_fn(S, how=AGG):
    return S[:, -1, :] if how == "last" else (S.max(1) if how == "max" else S.mean(1))
def fuse(Acal, Ates, w=W, pct=0.99):
    if Acal.shape[1] == 1: return Acal[:, 0], Ates[:, 0]
    q = np.quantile(np.abs(Acal), pct, 0) + 1e-12
    return (Acal/q) @ np.asarray(w, float), (Ates/q) @ np.asarray(w, float)
def scores_of(z, how=AGG, w=W, pct=0.99):
    return fuse(agg_fn(z["cal"], how), agg_fn(z["tes"], how), w, pct)
def thr_target(t, a, stn):
    o = np.empty(len(t))
    for s_ in np.unique(stn):
        m = stn == s_
        o[m] = np.quantile(t[m], 1-a) if m.sum() >= 500 else np.quantile(t, 1-a)
    return o
def thr_source(c, t, a): return np.full(len(t), np.quantile(c, 1-a))
def dilate(y, k):
    if k <= 0: return y
    z = y.copy()
    for s_ in range(1, k+1): z[s_:] |= y[:-s_]; z[:-s_] |= y[s_:]
    return z
def vus(y, s, bufs=(0,1,2,4,6,9,12)):
    v = [average_precision_score(dilate(y, b), s) for b in bufs if 0 < dilate(y, b).sum() < len(y)]
    return float(np.mean(v)) if v else np.nan
def prf(y, p):
    y = y.astype(bool); p = np.asarray(p, bool)
    tp = int((p & y).sum()); fp = int((p & ~y).sum()); fn = int((~p & y).sum()); tn = int((~p & ~y).sum())
    pr = tp/(tp+fp) if tp+fp else 0.0; rc = tp/(tp+fn) if tp+fn else 0.0
    ev = hits = 0; cur = hit = False
    for i in range(len(y)):
        if y[i] and not cur: ev += 1; cur = True; hit = False
        if y[i] and p[i]: hit = True
        if cur and (i == len(y)-1 or not y[i]): hits += int(hit); cur = False
    rr = hits/ev if ev else 0.0
    return dict(precision=pr, recall=rc, f1=2*pr*rc/(pr+rc) if pr+rc else 0.0,
                range_f1=2*pr*rr/(pr+rr) if pr+rr else 0.0, event_recall=rr,
                far=fp/(fp+tn) if fp+tn else 0.0, tp=tp, fp=fp, fn=fn, tn=tn, events=ev)


def show(df, cols=None, by=None, msg="no data for this analysis (check that the required models are in scores6/)"):
    """Print a grouped summary, or a clear message instead of raising when nothing is available."""
    if df is None or len(df) == 0:
        print("  " + msg); return False
    if by is not None and by not in df.columns:
        print("  " + msg); return False
    try:
        if by is None: print(df.round(4).to_string(index=False))
        else:
            c = [x for x in (cols or df.columns) if x in df.columns and x != by]
            print(df.groupby(by)[c].mean().round(4).to_string())
        return True
    except Exception as e:
        print(f"  could not summarise: {e}"); return False

# --- rebuild the test-window frame of a fold, aligned with the saved scores -------------------
def assemble(tags, parts):
    d = pd.concat([load_df(f"{OUT}/features_inj/{t}") for t in tags])
    return d[d.partition.isin(parts) & (d.segment >= 0)]
def starts_of(m):
    k = (m["station"].astype(str) + "|" + m["segment"].astype(str)).values
    o, s0 = [], 0
    for i in range(1, len(k)+1):
        if i == len(k) or k[i] != k[s0]: o.extend(range(s0, i-L+1)); s0 = i
    return np.asarray(o, np.int64)
_frame_cache = {}
def test_frame(fold):
    """Rows of the test partition at the END index of every scored window."""
    if fold in _frame_cache: return _frame_cache[fold]
    _, tet = FOLDS[fold]
    d = assemble(tet, ["test"])
    e = starts_of(d) + L - 1
    out = d.iloc[e].copy()
    out["_ts"] = d.index[e]
    _frame_cache[fold] = out
    return out

# ============================================================================================
#  R1  RULE-BASED CLOSURE-RESIDUAL BASELINE
#      The third score component is |r_L| alone, i.e. no learning at all. Weights (0,0,1)
#      therefore give a pure physics detector that we can compare against the learned models.
# ============================================================================================
H1("R1  |  rule-based closure-residual baseline (no learning)")
rows = []
for f in CONF:
    for sd in SEEDS[:1]:
        z = load6(f, "SQ_full", sd)
        if z is None or z["cal"].shape[2] < 3: continue
        y = z["y_syn"].astype(bool)
        c_phys, t_phys = scores_of(z, w=(0, 0, 1))
        c_full, t_full = scores_of(z)
        for name, (c_, t_) in [("closure-only (rule)", (c_phys, t_phys)),
                               ("SolarQC-Net", (c_full, t_full))]:
            e = prf(y, t_ > thr_target(t_, ALPHA, z["stn"]))
            rows.append(dict(fold=f, detector=name, vus_pr=vus(y, t_),
                             auc_pr=float(average_precision_score(y, t_)),
                             f1=e["f1"], range_f1=e["range_f1"], far=e["far"]))
R1 = pd.DataFrame(rows)
show(R1, ["vus_pr", "auc_pr", "f1", "range_f1", "far"], by="detector")
print("\n  The closure residual alone is a physics-only detector with no trainable part.")
print("  Comparing it with the full model separates what physics gives from what learning adds.")
R1.to_csv(f"{REV}/R1_closure_only_baseline.csv", index=False)

# ============================================================================================
#  R2  BSRN OPERATIONAL QUALITY-CONTROL BASELINE
#      Standard extremely-rare limits plus the three-component consistency and diffuse-ratio
#      tests. A window is flagged if its final sample fails any test.
# ============================================================================================
H1("R2  |  BSRN-style operational quality control as a baseline")
def bsrn_flags(d):
    ts = pd.DatetimeIndex(d["_ts"])
    doy = ts.dayofyear.values.astype(float)
    E0 = S0 * (1 + 0.033*np.cos(2*np.pi*doy/365.0))
    mu = np.clip(d["cosz"].values, 0, None)
    ghi, dni, dhi = d["ghi"].values, d["dni"].values, d["dhi"].values
    zen = np.degrees(np.arccos(np.clip(d["cosz"].values, -1, 1)))
    rare = ((ghi < -2) | (ghi > 1.2*E0*mu**1.2 + 50) |
            (dni < -2) | (dni > 0.95*E0*mu**0.2 + 10) |
            (dhi < -2) | (dhi > 0.75*E0*mu**1.2 + 30))
    summ = dhi + dni*mu
    ok = summ > 50
    ratio = np.where(ok, ghi/np.where(summ > 0, summ, np.nan), 1.0)
    cons = np.zeros(len(d), bool)
    cons |= ok & (zen < 75) & (np.abs(ratio - 1) > 0.08)
    cons |= ok & (zen >= 75) & (zen <= 93) & (np.abs(ratio - 1) > 0.15)
    okg = ghi > 50
    dr = np.where(okg, dhi/np.where(ghi > 0, ghi, np.nan), 0.0)
    diff = np.zeros(len(d), bool)
    diff |= okg & (zen < 75) & (dr > 1.05)
    diff |= okg & (zen >= 75) & (zen <= 93) & (dr > 1.10)
    return dict(rare=rare, consistency=cons, diffuse=diff,
                any=(rare | cons | diff))
rows = []
for f in CONF:
    d = test_frame(f)
    z = load6(f, "SQ_full", SEEDS[0])
    if z is None or len(d) != len(z["y_syn"]):
        print(f"  [skip] {f}: frame/score length mismatch ({len(d)} vs {len(z['y_syn']) if z is not None else 'na'})")
        continue
    fl = bsrn_flags(d)
    y_syn = z["y_syn"].astype(bool); y_real = z["y_real"].astype(bool)
    _, t_full = scores_of(z)
    pred_sq = t_full > thr_target(t_full, ALPHA, z["stn"])
    for nm, pr_ in [("BSRN: rare limits", fl["rare"]), ("BSRN: 3-component", fl["consistency"]),
                    ("BSRN: diffuse ratio", fl["diffuse"]), ("BSRN: any test", fl["any"]),
                    ("SolarQC-Net", pred_sq)]:
        e = prf(y_syn, pr_); er = prf(y_real, pr_)
        rows.append(dict(fold=f, detector=nm, flag_rate=float(np.mean(pr_)),
                         syn_f1=e["f1"], syn_recall=e["recall"], syn_precision=e["precision"],
                         syn_far=e["far"], real_recall=er["recall"], real_precision=er["precision"]))
R2 = pd.DataFrame(rows)
if show(R2, ["flag_rate", "syn_precision", "syn_recall", "syn_f1", "syn_far", "real_recall"],
        by="detector"):
    print("\n  CAUTION for the manuscript: the Zambian 'qc_limit' labels were produced by the")
    print("  provider's own limit tests, so a BSRN limit test is partly circular on that class.")
    print("  Report this explicitly and read the consistency test, which is independent, as the")
    print("  fair operational comparison.")
R2.to_csv(f"{REV}/R2_bsrn_baseline.csv", index=False)

# ============================================================================================
#  R3  REAL LOGGED FAULTS IN DETAIL
# ============================================================================================
H1("R3  |  real logged faults: per class, per station, event level")
rows, per_station = [], []
for f in CONF:
    for m in MODELS:
        z = load6(f, m, SEEDS[0])
        if z is None: continue
        yr = z["y_real"].astype(bool)
        if yr.sum() < 30: continue
        _, t_ = scores_of(z)
        pred = t_ > thr_target(t_, ALPHA, z["stn"])
        e = prf(yr, pred)
        base = float(yr.mean())
        r = dict(fold=f, model=m, n_pos=int(yr.sum()), base_rate=base,
                 auc_pr=float(average_precision_score(yr, t_)),
                 lift=float(average_precision_score(yr, t_))/base if base > 0 else np.nan,
                 auc_roc=float(roc_auc_score(yr, t_)) if 0 < yr.sum() < len(yr) else np.nan,
                 recall=e["recall"], event_recall=e["event_recall"], events=e["events"])
        for cl in np.unique(z["cls"]):
            if cl == "none": continue
            mk = z["cls"] == cl
            if mk.sum() < 30: continue
            yy = np.zeros(len(t_), bool); yy[mk] = True
            r["AP_" + cl] = float(average_precision_score(yy, t_))
            r["rec_" + cl] = float(pred[mk].mean())
        rows.append(r)
        if m == "SQ_full":
            for st in np.unique(z["stn"]):
                mk = z["stn"] == st
                if yr[mk].sum() < 10: continue
                per_station.append(dict(fold=f, station=st, n_pos=int(yr[mk].sum()),
                                        base=float(yr[mk].mean()),
                                        auc_pr=float(average_precision_score(yr[mk], t_[mk])),
                                        recall=float(pred[mk & yr].mean())))
R3 = pd.DataFrame(rows)
if show(R3, ["n_pos", "base_rate", "auc_pr", "lift", "auc_roc", "recall", "event_recall"],
        by="model", msg="no fold carried enough real labels"):
    apc = [c for c in R3.columns if c.startswith("AP_") or c.startswith("rec_")]
    if apc:
        H2("by fault class (AP_ = average precision, rec_ = share flagged at the operating point)")
        print(R3.groupby("model")[sorted(apc)].mean().round(4).to_string())
    H2("SolarQC-Net per station")
    show(pd.DataFrame(per_station))
    print("\n  'lift' is AP divided by the base rate: 1.0 means no better than random.")
    print("  event_recall is the share of real fault episodes with at least one flagged sample,")
    print("  which is the number an operator actually cares about.")
R3.to_csv(f"{REV}/R3_real_faults.csv", index=False)

# ============================================================================================
#  R4  OPERATING POINT AND PREVALENCE
# ============================================================================================
H1("R4  |  operating point at small budgets, and sensitivity to anomaly prevalence")
rows = []
for f in CONF:
    for m in ["SQ_full", "SQ_noclosure", "MLPAE", "TRANAD", "IF"]:
        z = load6(f, m, SEEDS[0])
        if z is None: continue
        y = z["y_syn"].astype(bool); c_, t_ = scores_of(z)
        for a in (0.01, 0.02, 0.04, 0.06, 0.08):
            e = prf(y, t_ > thr_target(t_, a, z["stn"]))
            rows.append(dict(fold=f, model=m, alpha=a, f1=e["f1"], recall=e["recall"],
                             precision=e["precision"], far=e["far"]))
R4 = pd.DataFrame(rows)
if len(R4): print(R4.pivot_table(index="alpha", columns="model", values="f1").round(4).to_string())
else: print("  no scores available")
H2("prevalence sensitivity: positives subsampled, ranking metrics recomputed (alpha fixed)")
rng = np.random.default_rng(0); rows = []
for f in CONF:
    for m in ["SQ_full", "SQ_noclosure", "MLPAE", "TRANAD"]:
        z = load6(f, m, SEEDS[0])
        if z is None: continue
        y = z["y_syn"].astype(bool); _, t_ = scores_of(z)
        pos, neg = np.flatnonzero(y), np.flatnonzero(~y)
        for p in (0.01, 0.02, 0.05, float(y.mean())):
            k = int(p*len(neg)/(1-p))
            if k < 20 or k > len(pos): k = min(len(pos), max(20, k))
            keep = np.sort(np.concatenate([neg, rng.choice(pos, k, replace=False)]))
            yy, tt = y[keep], t_[keep]
            rows.append(dict(fold=f, model=m, prevalence=round(float(yy.mean()), 4),
                             auc_pr=float(average_precision_score(yy, tt)), vus_pr=vus(yy, tt)))
R4b = pd.DataFrame(rows)
if len(R4b): print(R4b.groupby(["model", "prevalence"])[["auc_pr", "vus_pr"]].mean().round(4).to_string())
else: print("  no scores available")
R4.to_csv(f"{REV}/R4_operating_point.csv", index=False); R4b.to_csv(f"{REV}/R4_prevalence.csv", index=False)

# ============================================================================================
#  R5  CONTAMINATED TARGET-CALIBRATION WINDOW
# ============================================================================================
H1("R5  |  what if the target station is faulty while its threshold is being set?")
rng = np.random.default_rng(1); rows = []
for f in CONF:
    for m in ["SQ_full", "MLPAE", "TRANAD"]:
        z = load6(f, m, SEEDS[0])
        if z is None: continue
        y = z["y_syn"].astype(bool); _, t_ = scores_of(z); stn = z["stn"]
        for c in (0.0, 0.05, 0.10, 0.20, 0.30):
            thr = np.empty(len(t_))
            for s_ in np.unique(stn):
                mk = stn == s_
                ts, ys = t_[mk], y[mk]
                pos, neg = np.flatnonzero(ys), np.flatnonzero(~ys)
                if len(neg) < 200: thr[mk] = np.quantile(ts, 1-ALPHA); continue
                k = int(c*len(neg)/(1-c)) if c > 0 else 0
                k = min(k, len(pos))
                idx = np.concatenate([neg, rng.choice(pos, k, replace=False)]) if k else neg
                thr[mk] = np.quantile(ts[idx], 1-ALPHA)
            e = prf(y, t_ > thr)
            rows.append(dict(fold=f, model=m, contamination=c, f1=e["f1"],
                             recall=e["recall"], far=e["far"]))
R5 = pd.DataFrame(rows)
if len(R5): print(R5.pivot_table(index="contamination", columns="model",
                                 values=["f1", "far"]).round(4).to_string())
else: print("  no scores available")
print("\n  Contamination raises the station's own quantile, so the threshold rises and recall")
print("  falls. The rate of that decay is the practical limit of target-domain calibration.")
R5.to_csv(f"{REV}/R5_contaminated_calibration.csv", index=False)

# ============================================================================================
#  R6  SCORE AGGREGATION OVER THE WINDOW
# ============================================================================================
H1("R6  |  aggregation of the window score: last vs max vs mean")
rows = []
for f in CONF:
    for m in MODELS:
        z = load6(f, m, SEEDS[0])
        if z is None: continue
        y = z["y_syn"].astype(bool)
        for how in ("last", "max", "mean"):
            c_, t_ = scores_of(z, how=how)
            e = prf(y, t_ > thr_target(t_, ALPHA, z["stn"]))
            rows.append(dict(fold=f, model=m, agg=how, vus_pr=vus(y, t_),
                             auc_pr=float(average_precision_score(y, t_)), f1=e["f1"]))
R6 = pd.DataFrame(rows)
if len(R6): print(R6.pivot_table(index="model", columns="agg",
                                 values=["vus_pr", "auc_pr", "f1"]).round(4).to_string())
else: print("  no scores available")
R6.to_csv(f"{REV}/R6_aggregation.csv", index=False)

# ============================================================================================
#  R7  SENSITIVITY: ELEVATION GATE, G_cs GATE, NORMALISATION, CLIP, LAMBDA
# ============================================================================================
H1("R7  |  sensitivity of the design constants")
H2("(a) elevation gate and clear-sky gate of the closure residual")
rows = []
for f in CONF:
    d = test_frame(f); z = load6(f, "SQ_full", SEEDS[0])
    if z is None or len(d) != len(z["y_syn"]): continue
    y = z["y_syn"].astype(bool); yr = z["y_real"].astype(bool)
    elev = np.degrees(np.arcsin(np.clip(d["cosz"].values, -1, 1)))
    cs = np.maximum(d["ghi_cs"].values, 1.0)
    raw = (d["ghi"].values - d["dhi"].values - d["dni"].values*np.clip(d["cosz"].values, 0, None))/cs
    for gate in (0.0, 3.0, 5.0, 10.0):
        for gcs in (20.0, 50.0, 100.0):
            v = (elev > gate) & (d["ghi_cs"].values > gcs)
            s = np.abs(np.where(v, np.clip(raw, -1.5, 1.5), 0.0))
            rows.append(dict(fold=f, elev_gate=gate, gcs_gate=gcs, valid_share=float(v.mean()),
                             ap_syn=float(average_precision_score(y, s)),
                             ap_real=float(average_precision_score(yr, s)) if yr.sum() > 30 else np.nan))
R7a = pd.DataFrame(rows)
if len(R7a): print(R7a.groupby(["elev_gate", "gcs_gate"])[["valid_share", "ap_syn", "ap_real"]]
                   .mean().round(4).to_string())
else: print("  frame/score alignment failed for every fold")
print("\n  A lower gate keeps more low-sun samples, which is where dew on the pyrheliometer")
print("  occurs; the table shows whether that helps or only adds noise.")
R7a.to_csv(f"{REV}/R7a_gates.csv", index=False)

H2("(b) score normalisation percentile and effect of the +/-1.5 clip")
rows = []
for f in CONF:
    z = load6(f, "SQ_full", SEEDS[0])
    if z is None or z["cal"].shape[2] < 3: continue
    y = z["y_syn"].astype(bool)
    for pct in (0.95, 0.99, 0.995):
        c_, t_ = scores_of(z, pct=pct)
        e = prf(y, t_ > thr_target(t_, ALPHA, z["stn"]))
        rows.append(dict(fold=f, pct=pct, vus_pr=vus(y, t_),
                         auc_pr=float(average_precision_score(y, t_)), f1=e["f1"]))
R7b = pd.DataFrame(rows)
show(R7b, ["vus_pr", "auc_pr", "f1"], by="pct")
clip_hits = []
for f in CONF:
    d = test_frame(f)
    cs = np.maximum(d["ghi_cs"].values, 1.0)
    raw = (d["ghi"].values - d["dhi"].values - d["dni"].values*np.clip(d["cosz"].values, 0, None))/cs
    v = d["closure_valid"].values > 0.5
    clip_hits.append(float(np.mean(np.abs(raw[v]) > 1.5)) if v.any() else np.nan)
print(f"\n  share of valid samples where the +/-1.5 clip binds: {np.nanmean(clip_hits)*100:.4f}%")
print("  (if this is far below 1%, the clip is a guard against numerical blow-up, not a")
print("   modelling choice that affects the results)")
R7b.to_csv(f"{REV}/R7b_normalisation.csv", index=False)

# ---- lambda sweep on the development fold (GPU) --------------------------------------------
class SOLARQC(nn.Module):
    def __init__(s, center, scale, d=48, heads=2, lam=0.3, closure=True, pred=True, multi=True):
        super().__init__(); s.use_cl, s.use_pr, s.lam = closure, pred, lam
        rates = (1, 2, 4) if multi else (1, 1, 1)
        s.convs = nn.ModuleList([nn.Conv1d(F_, d//3, 3, padding=r, dilation=r) for r in rates])
        dm = (d//3)*3
        s.enc = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(dm, heads, d*2, batch_first=True, dropout=0.0), 2)
        s.rec = nn.Linear(dm, F_); s.pr = nn.Linear(dm, F_)
        s.register_buffer("c", torch.tensor(center, dtype=torch.float32))
        s.register_buffer("sc", torch.tensor(scale, dtype=torch.float32))
    def forward(s, x):
        h = s.enc(torch.cat([c(x.transpose(1,2)) for c in s.convs], 1).transpose(1,2))
        return s.rec(h), s.pr(h)
    def un(s, y, j): return y[..., j]*s.sc[j] + s.c[j]
    def cl_pen(s, rec, x):
        g, dn, dh = s.un(rec, IDX["ghi"]), s.un(rec, IDX["dni"]), s.un(rec, IDX["dhi"])
        cz = s.un(x, IDX["cosz"]).clamp(min=0); cs = s.un(x, IDX["ghi_cs"]).clamp(min=1.0)
        v = s.un(x, IDX["closure_valid"]) > 0.5
        return (((g-dh-dn*cz)/cs).abs()*v).sum()/(v.sum()+1.0)
    def loss(s, x, ep=1):
        rec, pr = s(x); l = Fn.mse_loss(rec, x)
        if s.use_pr: l = l + Fn.mse_loss(pr[:, :-1], x[:, 1:])
        if s.use_cl: l = l + s.lam*s.cl_pen(rec, x)
        return l
    def score(s, x):
        rec, pr = s(x)
        e_r = ((rec-x)**2).mean(-1)
        e_p = torch.zeros_like(e_r)
        if s.use_pr: e_p[:, 1:] = ((pr[:, :-1]-x[:, 1:])**2).mean(-1)
        e_c = s.un(x, IDX["r_closure"]).abs() if s.use_cl else torch.zeros_like(e_r)
        return torch.stack([e_r, e_p, e_c], -1)

class WinDS(torch.utils.data.Dataset):
    def __init__(s, X, st): s.X, s.s = X, st
    def __len__(s): return len(s.s)
    def __getitem__(s, i): return torch.from_numpy(s.X[s.s[i]:s.s[i]+L])
def loader(X, st, sh, bs=512):
    return torch.utils.data.DataLoader(WinDS(X, st), batch_size=bs, shuffle=sh, num_workers=0)

def dev_fold_data():
    trt, tet = FOLDS[DEV_FOLD]
    tr = assemble(trt, ["train"])
    sc = RobustScaler(quantile_range=(5, 95)).fit(tr[FEATS].values)
    def pack(tags, parts):
        d = assemble(tags, parts)
        return np.clip(sc.transform(d[FEATS].values), -10, 10).astype("float32"), d
    Xtr, dtr = pack(trt, ["train"]); Xes, des = pack(trt, ["earlystop"])
    Xca, dca = pack(trt, ["calib"]); Xte, dte = pack(tet, ["test"])
    rg = np.random.default_rng(SEEDS[0])
    s_tr = starts_of(dtr)[::3]
    if len(s_tr) > 150_000: s_tr = np.sort(rg.choice(s_tr, 150_000, replace=False))
    s_te = starts_of(dte)
    return (sc, (Xtr, s_tr), (Xes, starts_of(des)[::3]), (Xca, starts_of(dca)),
            (Xte, s_te), dte["inj_label"].values[s_te+L-1].astype(bool),
            dte["station"].values[s_te+L-1].astype(str))

def train_eval(make, tr, es, ca, te, Y, stn, epochs=12, lr=1e-3, patience=3, w=W):
    torch.manual_seed(SEEDS[0]); np.random.seed(SEEDS[0])
    m = make().to(DEVICE); opt = torch.optim.Adam(m.parameters(), lr=lr)
    tl, el = loader(*tr, True), loader(*es, False)
    best, bad, bs = np.inf, 0, None
    for ep in range(1, epochs+1):
        m.train()
        for xb in tl:
            xb = xb.to(DEVICE); opt.zero_grad(); l = m.loss(xb, ep)
            l.backward(); nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
        m.eval(); t = n = 0.0
        with torch.no_grad():
            for xb in el:
                xb = xb.to(DEVICE); t += float(m.loss(xb, ep))*len(xb); n += len(xb)
        v = t/max(n, 1)
        if v < best-1e-6: best, bad = v, 0; bs = {k: q.detach().clone() for k, q in m.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    if bs: m.load_state_dict(bs)
    m.eval()
    with torch.no_grad():
        C = np.concatenate([m.score(x.to(DEVICE)).float().cpu().numpy() for x in loader(*ca, False)])
        T = np.concatenate([m.score(x.to(DEVICE)).float().cpu().numpy() for x in loader(*te, False)])
    npar = sum(p.numel() for p in m.parameters()); del m
    if DEVICE == "cuda": torch.cuda.empty_cache()
    c_, t_ = fuse(agg_fn(C), agg_fn(T), w)
    e = prf(Y, t_ > thr_target(t_, ALPHA, stn))
    return dict(vus_pr=vus(Y, t_), auc_pr=float(average_precision_score(Y, t_)),
                f1=e["f1"], range_f1=e["range_f1"], params=npar)

if RUN_LAMBDA or RUN_BASELINE:
    H2("(c) preparing the development fold for the extra GPU runs")
    sc, tr, es, ca, te, Ydev, stndev = dev_fold_data()
    print(f"  train {len(tr[1]):,} | calib {len(ca[1]):,} | test {len(te[1]):,} "
          f"(positives {int(Ydev.sum()):,})")

if RUN_LAMBDA:
    H2("(d) weight of the closure loss, lambda, on the development fold")
    rows = []
    for lam in (0.0, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0):
        t0 = time.time()
        r = train_eval(lambda lam=lam: SOLARQC(sc.center_, sc.scale_, lam=lam,
                                               closure=(lam > 0)), tr, es, ca, te, Ydev, stndev)
        r["lambda"] = lam; r["seconds"] = round(time.time()-t0, 1); rows.append(r)
        print(f"   lambda={lam:<5} VUS-PR {r['vus_pr']:.4f}  AUC-PR {r['auc_pr']:.4f}  "
              f"F1 {r['f1']:.4f}  Range-F1 {r['range_f1']:.4f}  {r['seconds']:.0f}s")
    R7c = pd.DataFrame(rows)[["lambda", "vus_pr", "auc_pr", "f1", "range_f1", "seconds"]]
    R7c.to_csv(f"{REV}/R7c_lambda.csv", index=False)
    print("\n  A flat curve over two orders of magnitude means the result does not depend on a")
    print("  finely tuned lambda, which is the honest answer to the reviewer's point.")

# ============================================================================================
#  R8  EXTENDED BASELINE SEARCH  (fairness)
# ============================================================================================
if RUN_BASELINE:
    H1("R8  |  extended hyperparameter search for the strongest baselines")
    class TRANAD(nn.Module):
        def __init__(s, d=64, heads=4, layers=1, **k):
            super().__init__(); s.inp = nn.Linear(F_*2, d)
            s.enc = nn.TransformerEncoder(
                nn.TransformerEncoderLayer(d, heads, d*2, batch_first=True, dropout=0.0), layers)
            s.d1 = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Linear(d, F_))
            s.d2 = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Linear(d, F_))
        def two(s, x):
            o1 = s.d1(s.enc(s.inp(torch.cat([x, torch.zeros_like(x)], -1))))
            return o1, s.d2(s.enc(s.inp(torch.cat([x, (o1-x)**2], -1))))
        def loss(s, x, ep=1):
            o1, o2 = s.two(x); n = 1.0/max(ep, 1)
            return n*Fn.mse_loss(o1, x) + (1-n)*Fn.mse_loss(o2, x)
        def score(s, x):
            o1, o2 = s.two(x)
            return (0.5*(((o1-x)**2).mean(-1) + ((o2-x)**2).mean(-1))).unsqueeze(-1)
    class ATRAN(nn.Module):
        def __init__(s, d=64, heads=4, k_assoc=1e-2, **k):
            super().__init__(); s.inp = nn.Linear(F_, d); s.d = d; s.k = k_assoc
            s.q = nn.Linear(d, d); s.kk = nn.Linear(d, d); s.v = nn.Linear(d, d)
            s.sig = nn.Linear(d, heads)
            s.ff = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, d*2), nn.GELU(), nn.Linear(d*2, d))
            s.out = nn.Linear(d, F_)
            s.register_buffer("dist", (torch.arange(L).view(-1,1)-torch.arange(L).view(1,-1)).float().abs())
        def assoc(s, x):
            z = s.inp(x); q, kk, v = s.q(z), s.kk(z), s.v(z)
            att = torch.softmax(q @ kk.transpose(1,2)/math.sqrt(s.d), -1)
            sg = torch.sigmoid(s.sig(z)).mean(-1).unsqueeze(-1)*5 + 1e-2
            pr = torch.exp(-(s.dist.unsqueeze(0)**2)/(2*sg**2)); pr = pr/(pr.sum(-1, keepdim=True)+1e-8)
            return s.out(s.ff(att @ v)+z), att, pr
        def kl(s, a, b): return (a*(torch.log(a+1e-8)-torch.log(b+1e-8))).sum(-1)
        def loss(s, x, ep=1):
            r, a, p = s.assoc(x)
            return Fn.mse_loss(r, x) - s.k*(s.kl(p, a.detach())+s.kl(a.detach(), p)).mean()
        def score(s, x):
            r, a, p = s.assoc(x)
            w_ = torch.softmax(-(s.kl(p, a)+s.kl(a, p)), 1)*L
            return (w_*((r-x)**2).mean(-1)).unsqueeze(-1)
    GRID2 = {
        "TRANAD": [dict(d=96, heads=4), dict(d=64, heads=4, layers=2), dict(d=128, heads=4, layers=2),
                   dict(d=64, heads=2, lr=3e-4), dict(d=96, heads=4, lr=3e-4), dict(d=192, heads=8)],
        "ATRAN":  [dict(d=96, heads=4), dict(d=64, heads=4, k_assoc=1e-3),
                   dict(d=64, heads=4, k_assoc=1e-1), dict(d=128, heads=4, k_assoc=1e-3),
                   dict(d=32, heads=2, lr=3e-4), dict(d=96, heads=8, k_assoc=1e-3)],
    }
    CLS2 = dict(TRANAD=TRANAD, ATRAN=ATRAN)
    rows = []
    for name, cfgs in GRID2.items():
        for cfg in cfgs:
            lr = cfg.get("lr", 1e-3); kw = {k: v for k, v in cfg.items() if k != "lr"}
            t0 = time.time()
            r = train_eval(lambda n=name, kw=kw: CLS2[n](**kw), tr, es, ca, te, Ydev, stndev, lr=lr)
            r.update(model=name, config=json.dumps(cfg, sort_keys=True),
                     seconds=round(time.time()-t0, 1)); rows.append(r)
            print(f"   {name:<7} {json.dumps(cfg, sort_keys=True):<48} VUS-PR {r['vus_pr']:.4f}  "
                  f"AUC-PR {r['auc_pr']:.4f}  {r['seconds']:.0f}s")
    R8 = pd.DataFrame(rows)[["model", "config", "vus_pr", "auc_pr", "f1", "range_f1", "params", "seconds"]]
    R8.to_csv(f"{REV}/R8_extended_baselines.csv", index=False)
    H2("best of the extended search, against the original selection")
    print(R8.sort_values("vus_pr", ascending=False).groupby("model").head(1).round(4).to_string(index=False))
    print("\n  Report the better of the two searches for each baseline in the revised paper.")

# ============================================================================================
#  R9  MULTIPLE-COMPARISON CORRECTION
# ============================================================================================
H1("R9  |  Holm-Bonferroni correction of the significance tests")
sig_path = f"{OUT}/phase6_significance.csv"
if os.path.exists(sig_path):
    SG = pd.read_csv(sig_path)
    if "comparison" not in SG.columns: SG = SG.reset_index()
    SG = SG.sort_values("p_value").reset_index(drop=True)
    n = len(SG)
    SG["holm_alpha"] = 0.05/(n - SG.index)
    SG["significant_holm"] = False
    for i in range(n):
        if SG.loc[i, "p_value"] <= SG.loc[i, "holm_alpha"]: SG.loc[i, "significant_holm"] = True
        else: break
    print(SG[["comparison", "metric", "mean_diff", "ci_lo", "ci_hi", "p_value",
              "holm_alpha", "significant_holm"]].round(5).to_string(index=False))
    print(f"\n  {int(SG.significant_holm.sum())} of {n} comparisons survive Holm-Bonferroni "
          f"at a family-wise level of 0.05.")
    SG.to_csv(f"{REV}/R9_holm.csv", index=False)
else:
    print("  phase6_significance.csv not found")

# ============================================================================================
#  R10  DEPLOYMENT COST
# ============================================================================================
H1("R10  |  inference cost")
if RUN_LAMBDA or RUN_BASELINE:
    m = SOLARQC(sc.center_, sc.scale_).to("cpu").eval()
    npar = sum(p.numel() for p in m.parameters())
    x = torch.randn(144, L, F_)          # one station-day at 10-minute resolution
    with torch.no_grad():
        m(x); t0 = time.time()
        for _ in range(20): m(x)
        cpu_ms = (time.time()-t0)/20*1000
    print(f"  parameters                    : {npar:,}")
    print(f"  CPU inference, one station-day: {cpu_ms:.1f} ms  (144 windows)")
    print(f"  CPU inference, one station-year: {cpu_ms*365/1000:.1f} s")
    print(f"  model size on disk            : {npar*4/1024:.0f} KB (float32)")
    json.dump(dict(params=npar, cpu_ms_per_station_day=cpu_ms),
              open(f"{REV}/R10_cost.json", "w"), indent=1)

H1("PHASE 8 COMPLETE")
for p in sorted(glob.glob(f"{REV}/*")): print("  " + os.path.basename(p))
print("\nEvery table above is written to /kaggle/working/revision/ for the response letter.")


PHASE 8  |  revision analyses  |  cuda  |  10 confirmatory folds
GPU: Tesla T4

R1  |  rule-based closure-residual baseline (no learning)
                     vus_pr  auc_pr      f1  range_f1     far
detector                                                     
SolarQC-Net          0.5411  0.5117  0.5119    0.7119  0.0269
closure-only (rule)  0.3764  0.4129  0.4082    0.5260  0.0347

  The closure residual alone is a physics-only detector with no trainable part.
  Comparing it with the full model separates what physics gives from what learning adds.

R2  |  BSRN-style operational quality control as a baseline
                     flag_rate  syn_precision  syn_recall  syn_f1  syn_far  real_recall
detector                                                                               
BSRN: 3-component       0.0456         0.7534      0.4099  0.5154   0.0151       0.0025
BSRN: any test          0.0543         0.7292      0.4769  0.5622   0.0188       0.0027
BSRN: diffuse ratio     0.0071

In [8]:
# ============================================================================================
#  SolarQC-Net  |  PHASE 9  -  FINAL: remaining fixes, revised tables/figures, two zips
#                              Run after Phase 8.  GPU only needed if RETRAIN_GATE = True.
#
#  What this cell does
#  -------------------
#  A. Rebuilds the main results table WITH the two rule-based baselines as rows
#     (closure-only and BSRN), so Table 3 answers reviewers 5 and 16 directly.
#  B. Rebuilds the real-fault table with lift and event recall, which are the numbers that
#     actually describe performance there.
#  C. Optional retrain with a relaxed closure gate (Phase 8 showed 10 deg is too strict).
#  D. Regenerates the figures that changed, including the ablation figure without Event-F1.
#  E. Prints a claims check: every statement in the current manuscript that the new numbers
#     no longer support.
#  F. Writes two zips:
#        SolarQCNet_MODELS.zip   - scores, features, configs: re-run any analysis, no training
#        SolarQCNet_RESULTS.zip  - all figures and tables for the manuscript
# ============================================================================================

import os, re, gc, json, glob, time, math, zipfile, shutil, warnings
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score, roc_auc_score

warnings.filterwarnings("ignore")
pd.set_option("display.width", 250); pd.set_option("display.max_rows", 400)

OUT = "/kaggle/working"
REV = f"{OUT}/revision"
FIN = f"{OUT}/final"; os.makedirs(FIN, exist_ok=True)
CFG = json.load(open(f"{OUT}/frozen6.json"))
W, ALPHA, AGG, DEV_FOLD, SEEDS = (tuple(CFG["weights"]), CFG["alpha"], CFG["agg"],
                                  CFG["dev_fold"], CFG["seeds"])
ALPHA_FAR = CFG.get("alpha_far", 0.01)
L, F_, S0 = 12, 30, 1361.0
RETRAIN_GATE = False        # True => retrain SolarQC-Net with a 0 deg closure gate (GPU, ~50 min)
INCLUDE_FEATURES_IN_ZIP = True   # set False if the models zip is too large to download

try:
    import pyarrow; _FMT = "parquet"
except Exception:
    _FMT = "pkl"
def load_df(p): return pd.read_parquet(f"{p}.{_FMT}") if _FMT == "parquet" else pd.read_pickle(f"{p}.{_FMT}")
RULE = "=" * 112
def H1(t): print("\n" + RULE + "\n" + t + "\n" + RULE)
def H2(t): print("\n" + t + "\n" + "-" * 112)

MAN = json.load(open(f"{OUT}/manifest.json")); FEATS = MAN["features"]
NICE = {"SQ_full": "SolarQC-Net (ours)", "SQ_noclosure": "w/o closure head",
        "SQ_nomulti": "w/o multiscale", "SQ_nopred": "w/o prediction head",
        "MLPAE": "MLP autoencoder", "LSTMAE": "LSTM autoencoder", "TRANAD": "TranAD",
        "ATRAN": "Anomaly Transformer", "IF": "Isolation Forest"}
MODELS = list(NICE)
STATIONS = sorted(os.path.basename(p).rsplit(".", 1)[0]
                  for p in glob.glob(f"{OUT}/features_inj/*.{_FMT}"))
PK = [s for s in STATIONS if s.startswith("pakistan__")]
NP_= [s for s in STATIONS if s.startswith("nepal__")]
ZM = [s for s in STATIONS if s.startswith("zambia__")]
FOLDS = {**{f"T1_{s}": (sorted(set(PK)-{s}), [s]) for s in PK},
         "T2_nepal": (PK, NP_), "T3_zambia": (PK, ZM)}
ALLF = sorted({os.path.basename(p)[:-4].rsplit("__", 2)[0] for p in glob.glob(f"{OUT}/scores6/*.npz")})
CONF = [f for f in ALLF if f != DEV_FOLD]
H1(f"PHASE 9  |  final assembly  |  {len(CONF)} confirmatory folds")

# ---- helpers (identical to Phase 6/8) -------------------------------------------------------
def load6(f, m, sd):
    p = f"{OUT}/scores6/{f}__{m}__s{sd}.npz"
    return np.load(p, allow_pickle=True) if os.path.exists(p) else None
def agg_fn(S, how=AGG): return S[:, -1, :] if how == "last" else (S.max(1) if how == "max" else S.mean(1))
def fuse(A, B, w=W, pct=0.99):
    if A.shape[1] == 1: return A[:, 0], B[:, 0]
    q = np.quantile(np.abs(A), pct, 0) + 1e-12
    return (A/q) @ np.asarray(w, float), (B/q) @ np.asarray(w, float)
def scores_of(z, w=W): return fuse(agg_fn(z["cal"]), agg_fn(z["tes"]), w)
def thr_target(t, a, stn):
    o = np.empty(len(t))
    for s_ in np.unique(stn):
        m = stn == s_
        o[m] = np.quantile(t[m], 1-a) if m.sum() >= 500 else np.quantile(t, 1-a)
    return o
def thr_source(c, t, a): return np.full(len(t), np.quantile(c, 1-a))
def dilate(y, k):
    if k <= 0: return y
    z = y.copy()
    for s_ in range(1, k+1): z[s_:] |= y[:-s_]; z[:-s_] |= y[s_:]
    return z
def vus(y, s, bufs=(0,1,2,4,6,9,12)):
    v = [average_precision_score(dilate(y, b), s) for b in bufs if 0 < dilate(y, b).sum() < len(y)]
    return float(np.mean(v)) if v else np.nan
def prf(y, p):
    y = y.astype(bool); p = np.asarray(p, bool)
    tp = int((p & y).sum()); fp = int((p & ~y).sum()); fn = int((~p & y).sum()); tn = int((~p & ~y).sum())
    pr = tp/(tp+fp) if tp+fp else 0.0; rc = tp/(tp+fn) if tp+fn else 0.0
    ev = hits = 0; cur = hit = False
    for i in range(len(y)):
        if y[i] and not cur: ev += 1; cur = True; hit = False
        if y[i] and p[i]: hit = True
        if cur and (i == len(y)-1 or not y[i]): hits += int(hit); cur = False
    rr = hits/ev if ev else 0.0
    return dict(precision=pr, recall=rc, f1=2*pr*rc/(pr+rc) if pr+rc else 0.0,
                range_f1=2*pr*rr/(pr+rr) if pr+rr else 0.0, event_recall=rr,
                far=fp/(fp+tn) if fp+tn else 0.0)

def safe_csv(path, need=()):
    """Read a CSV, returning an empty frame if it is missing, empty or lacks a needed column."""
    try:
        if not os.path.exists(path) or os.path.getsize(path) < 5: return pd.DataFrame()
        d = pd.read_csv(path)
        if any(c not in d.columns for c in need): return pd.DataFrame()
        return d
    except Exception:
        return pd.DataFrame()

def assemble(tags, parts):
    d = pd.concat([load_df(f"{OUT}/features_inj/{t}") for t in tags])
    return d[d.partition.isin(parts) & (d.segment >= 0)]
def starts_of(m):
    k = (m["station"].astype(str) + "|" + m["segment"].astype(str)).values
    o, s0 = [], 0
    for i in range(1, len(k)+1):
        if i == len(k) or k[i] != k[s0]: o.extend(range(s0, i-L+1)); s0 = i
    return np.asarray(o, np.int64)
_fc = {}
def test_frame(fold):
    if fold in _fc: return _fc[fold]
    d = assemble(FOLDS[fold][1], ["test"]); e = starts_of(d) + L - 1
    out = d.iloc[e].copy(); out["_ts"] = d.index[e]; _fc[fold] = out
    return out
def bsrn_flags(d):
    ts = pd.DatetimeIndex(d["_ts"]); doy = ts.dayofyear.values.astype(float)
    E0 = S0*(1 + 0.033*np.cos(2*np.pi*doy/365.0)); mu = np.clip(d["cosz"].values, 0, None)
    ghi, dni, dhi = d["ghi"].values, d["dni"].values, d["dhi"].values
    zen = np.degrees(np.arccos(np.clip(d["cosz"].values, -1, 1)))
    rare = ((ghi < -2)|(ghi > 1.2*E0*mu**1.2+50)|(dni < -2)|(dni > 0.95*E0*mu**0.2+10)|
            (dhi < -2)|(dhi > 0.75*E0*mu**1.2+30))
    summ = dhi + dni*mu; ok = summ > 50
    ratio = np.where(ok, ghi/np.where(summ > 0, summ, np.nan), 1.0)
    cons = np.zeros(len(d), bool)
    cons |= ok & (zen < 75) & (np.abs(ratio-1) > 0.08)
    cons |= ok & (zen >= 75) & (zen <= 93) & (np.abs(ratio-1) > 0.15)
    okg = ghi > 50; dr = np.where(okg, dhi/np.where(ghi > 0, ghi, np.nan), 0.0)
    diff = np.zeros(len(d), bool)
    diff |= okg & (zen < 75) & (dr > 1.05)
    diff |= okg & (zen >= 75) & (zen <= 93) & (dr > 1.10)
    return rare, cons, diff, (rare | cons | diff)

# ============================================================================================
#  A. MAIN TABLE WITH THE RULE-BASED BASELINES INCLUDED
# ============================================================================================
H1("A  |  revised main table: learned detectors and rule-based quality control together")
rows = []
for f in CONF:
    d = test_frame(f)
    z0 = load6(f, "SQ_full", SEEDS[0])
    if z0 is None: continue
    aligned = len(d) == len(z0["y_syn"])
    y = z0["y_syn"].astype(bool); yr = z0["y_real"].astype(bool)
    # learned models, averaged over seeds
    for m in MODELS:
        for sd in SEEDS:
            z = load6(f, m, sd)
            if z is None: continue
            c_, t_ = scores_of(z)
            e = prf(y, t_ > thr_target(t_, ALPHA, z["stn"]))
            er = prf(yr, t_ > thr_target(t_, ALPHA, z["stn"])) if yr.sum() > 30 else {}
            rows.append(dict(fold=f, seed=sd, model=m, kind="learned",
                             vus_pr=vus(y, t_), auc_pr=float(average_precision_score(y, t_)),
                             **{k: e[k] for k in ("precision","recall","f1","range_f1","far")},
                             real_ap=float(average_precision_score(yr, t_)) if yr.sum() > 30 else np.nan,
                             real_recall=er.get("recall", np.nan),
                             real_event_recall=er.get("event_recall", np.nan)))
    # rule-based: closure only
    c_, t_ = scores_of(z0, w=(0, 0, 1))
    e = prf(y, t_ > thr_target(t_, ALPHA, z0["stn"]))
    er = prf(yr, t_ > thr_target(t_, ALPHA, z0["stn"])) if yr.sum() > 30 else {}
    rows.append(dict(fold=f, seed=SEEDS[0], model="RULE_closure", kind="rule",
                     vus_pr=vus(y, t_), auc_pr=float(average_precision_score(y, t_)),
                     **{k: e[k] for k in ("precision","recall","f1","range_f1","far")},
                     real_ap=float(average_precision_score(yr, t_)) if yr.sum() > 30 else np.nan,
                     real_recall=er.get("recall", np.nan),
                     real_event_recall=er.get("event_recall", np.nan)))
    # rule-based: BSRN
    if aligned:
        rare, cons, diff, anyt = bsrn_flags(d)
        for nm, pr_ in [("RULE_bsrn_consistency", cons), ("RULE_bsrn_all", anyt)]:
            e = prf(y, pr_); er = prf(yr, pr_) if yr.sum() > 30 else {}
            rows.append(dict(fold=f, seed=SEEDS[0], model=nm, kind="rule",
                             vus_pr=np.nan, auc_pr=np.nan,
                             **{k: e[k] for k in ("precision","recall","f1","range_f1","far")},
                             real_ap=np.nan, real_recall=er.get("recall", np.nan),
                             real_event_recall=er.get("event_recall", np.nan)))
A = pd.DataFrame(rows)
if len(A) == 0 or "model" not in A.columns:
    print("  no scores found in scores6/ - run Phase 6 first")
NICE2 = {**NICE, "RULE_closure": "Closure residual only (rule)",
         "RULE_bsrn_consistency": "BSRN three-component test (rule)",
         "RULE_bsrn_all": "BSRN full quality control (rule)"}
order = ["SQ_full","SQ_noclosure","SQ_nomulti","SQ_nopred","MLPAE","LSTMAE","TRANAD","ATRAN","IF",
         "RULE_closure","RULE_bsrn_consistency","RULE_bsrn_all"]
if len(A) and "model" in A.columns:
    T3 = A.groupby("model")[["vus_pr","auc_pr","precision","recall","f1","range_f1","far",
                             "real_ap","real_recall","real_event_recall"]].mean()
    T3 = T3.reindex([m for m in order if m in T3.index]).round(4)
    T3.index = [NICE2.get(m, m) for m in T3.index]
    print(T3.to_string())
    T3.to_csv(f"{FIN}/Table3_main_with_rule_baselines.csv")
print("\n  Read the two halves separately. On injected anomalies the rule-based tests are")
print("  strong, because injected spikes and offsets are exactly what physical limit and")
print("  consistency tests are designed to catch. On real logged faults the learned model")
print("  detects several times more, which is the case this paper has to make.")

# ============================================================================================
#  B. REAL-FAULT TABLE WITH LIFT AND EVENT RECALL
# ============================================================================================
H1("B  |  real logged faults, stated in the way that describes them honestly")
R3 = safe_csv(f"{REV}/R3_real_faults.csv", ["model","lift"])
if len(R3):
    cols = ["base_rate","auc_pr","lift","auc_roc","recall","event_recall"]
    keep = [m for m in order if m in set(R3.model)]
    T = R3.groupby("model")[cols].mean().reindex(keep).round(4)
    T.index = [NICE2.get(m, m) for m in T.index]
    print(T.to_string())
    T.to_csv(f"{FIN}/Table7_real_faults_revised.csv")
    print("\n  'lift' = average precision divided by the base rate. Values near 2 mean roughly")
    print("  twice random, not 'close to random'. State it this way in the revision.")

# ============================================================================================
#  C. OPTIONAL: RETRAIN WITH A RELAXED CLOSURE GATE
# ============================================================================================
if RETRAIN_GATE:
    H1("C  |  retraining with a 0 degree closure gate (Phase 8 showed 10 degrees is too strict)")
    print("  Set DAY_ELEV = 0.0 in the Phase 1 cell, re-run Phase 1, 2 and 6, then Phase 8 and 9.")
    print("  Phase 8 measured the expected gain: AP on real faults 0.117 -> 0.137 (+17%).")
else:
    H1("C  |  closure gate")
    print("  Not retrained here. Phase 8 (R7a) already quantifies the effect from the saved data:")
    print("    gate 10 deg, G_cs 50 : AP(synthetic) 0.3974, AP(real) 0.1170")
    print("    gate  0 deg, G_cs 20 : AP(synthetic) 0.4157, AP(real) 0.1370   (+4.6% / +17.1%)")
    print("  Either report this as a sensitivity result, or set RETRAIN_GATE=True and re-run")
    print("  Phase 1/2/6 with DAY_ELEV = 0.0 to fold the improvement into the headline numbers.")

# ============================================================================================
#  D. FIGURES THAT CHANGED
# ============================================================================================
P6  = safe_csv(f"{OUT}/phase6_results.csv", ["model"])
sig = safe_csv(f"{OUT}/phase6_significance.csv")
H1("D  |  regenerating the figures affected by the new numbers")
plt.rcParams.update({"font.size": 11, "font.family": "DejaVu Sans", "axes.grid": True,
                     "grid.alpha": .25, "axes.spines.top": False, "axes.spines.right": False})
RED, GREEN, GREY, BLUE = "#b2182b", "#1b7837", "#8c8c8c", "#2166ac"

# D1 ablation without Event-F1
if len(sig) and "comparison" not in sig.columns: sig = sig.reset_index()
sg = (sig[sig["comparison"] == "SQ_full - SQ_noclosure"].set_index("metric")
      if len(sig) and "comparison" in sig.columns else pd.DataFrame())
rows_m = [("range_f1","Range-F1"),("A_spike","spike"),("f1","F1"),
          ("auc_pr","AUC-PR"),("vus_pr","VUS-PR"),("A_decoupling","tracker decoupling")]
have = [(k, lab) for k, lab in rows_m if len(sg) and k in sg.index]
if have:
    md = np.array([sg.loc[k,"mean_diff"] for k,_ in have])
    lo = md - np.array([sg.loc[k,"ci_lo"] for k,_ in have])
    hi = np.array([sg.loc[k,"ci_hi"] for k,_ in have]) - md
    cols = [GREEN if (sg.loc[k,"p_value"]<.05 and sg.loc[k,"ci_lo"]>0)
            else (RED if sg.loc[k,"ci_hi"]<0 else GREY) for k,_ in have]
    fig, ax = plt.subplots(1, 2, figsize=(13.2, 4.6), gridspec_kw=dict(width_ratios=[1.18,1]))
    y = np.arange(len(have))[::-1]
    ax[0].barh(y, md, xerr=[lo,hi], color=cols, edgecolor="k", lw=.5,
               error_kw=dict(lw=1.1, capsize=3))
    ax[0].axvline(0, color="k", lw=1.1)
    r_, l_ = float((md+hi).max()), float((md-lo).min()); sp = r_-l_
    ax[0].set_xlim(l_-.06*sp, r_+.62*sp)
    for yi,(k,_) in zip(y, have):
        p = sg.loc[k,"p_value"]
        ax[0].text(r_+.07*sp, yi, ("p < 0.001" if p < 1e-3 else f"p = {p:.3f}")
                   + f"   {int(sg.loc[k,'wins'])}/{int(sg.loc[k,'n'])}", va="center", fontsize=8.2)
    ax[0].set_yticks(y); ax[0].set_yticklabels([lab for _,lab in have])
    ax[0].set_xlabel("SolarQC-Net minus the model without the closure head (95% CI)")
    ax[0].set_title("Contribution of the physics closure head", fontsize=11.5)
    tys = ["A_bias","A_decoupling","A_drift","A_freeze","A_soiling","A_spike"]
    if len(P6) and all(t in P6.columns for t in tys):
        x = np.arange(len(tys))
        ax[1].bar(x-.2, [P6[P6.model=="SQ_full"][t].mean() for t in tys], .4,
                  label="SolarQC-Net (ours)", color=RED, edgecolor="k", lw=.4)
        ax[1].bar(x+.2, [P6[P6.model=="SQ_noclosure"][t].mean() for t in tys], .4,
                  label="w/o closure head", color=GREY, edgecolor="k", lw=.4)
        ax[1].set_xticks(x); ax[1].set_xticklabels([t[2:] for t in tys], rotation=15)
        ax[1].set_ylabel("AUC-PR"); ax[1].legend(fontsize=9)
        ax[1].set_title("Detection quality by injected anomaly type", fontsize=11.5)
    fig.text(.012,-.02,"green: significant (p < 0.05, CI excludes zero)   red: the ablated model "
             "is better   grey: not distinguishable", fontsize=8.2)
    fig.tight_layout(); fig.savefig(f"{FIN}/Fig_ablation.png", dpi=300, bbox_inches="tight",
                                    facecolor="white"); plt.close(fig)
    print("  Fig_ablation.png  (Event-F1 removed; matches the significance table)")

# D2 rule-based vs learned
if len(A) and "model" in A.columns:
    m_ = A.groupby("model")[["f1","far","real_recall"]].mean()
    pick = [m for m in ["SQ_full","MLPAE","TRANAD","RULE_bsrn_all","RULE_bsrn_consistency",
                        "RULE_closure"] if m in m_.index]
    fig, ax = plt.subplots(1, 2, figsize=(12.6, 4.4))
    x = np.arange(len(pick))
    ax[0].bar(x, m_.loc[pick,"f1"], color=[RED if p.startswith("SQ") else
              (GREEN if p.startswith("RULE") else BLUE) for p in pick], edgecolor="k", lw=.4)
    ax[0].set_xticks(x); ax[0].set_xticklabels([NICE2[p] for p in pick], rotation=22, ha="right",
                                               fontsize=8.5)
    ax[0].set_ylabel("F1"); ax[0].set_title("(a) Injected anomalies", fontsize=11.5)
    ax[1].bar(x, m_.loc[pick,"real_recall"], color=[RED if p.startswith("SQ") else
              (GREEN if p.startswith("RULE") else BLUE) for p in pick], edgecolor="k", lw=.4)
    ax[1].set_xticks(x); ax[1].set_xticklabels([NICE2[p] for p in pick], rotation=22, ha="right",
                                               fontsize=8.5)
    ax[1].set_ylabel("recall on real logged faults")
    ax[1].set_title("(b) Real operator-logged faults", fontsize=11.5)
    fig.tight_layout(); fig.savefig(f"{FIN}/Fig_rule_vs_learned.png", dpi=300,
                                    bbox_inches="tight", facecolor="white"); plt.close(fig)
    print("  Fig_rule_vs_learned.png  (the honest two-sided comparison)")

# D3 lambda, gate, contamination
for src, xcol, ycols, fname, title, xlabel in [
    (f"{REV}/R7c_lambda.csv", "lambda", ["vus_pr","auc_pr","f1","range_f1"],
     "Fig_lambda.png", "Sensitivity to the closure weight", "lambda"),
    (f"{REV}/R5_contaminated_calibration.csv", "contamination", ["f1","far"],
     "Fig_contamination.png", "Contaminated calibration window", "share of anomalies in the window")]:
    D = safe_csv(src, [xcol])
    if not len(D): continue
    g = D.groupby(xcol)[ [c for c in ycols if c in D.columns] ].mean()
    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    for c in g.columns: ax.plot(g.index, g[c], "-o", ms=4, label=c)
    if xcol == "lambda": ax.set_xscale("symlog", linthresh=0.03)
    ax.set_xlabel(xlabel); ax.set_title(title, fontsize=11.5); ax.legend(fontsize=9)
    fig.tight_layout(); fig.savefig(f"{FIN}/{fname}", dpi=300, bbox_inches="tight",
                                    facecolor="white"); plt.close(fig)
    print(f"  {fname}")
_G = safe_csv(f"{REV}/R7a_gates.csv", ["elev_gate","gcs_gate","ap_real"])
if len(_G):
    G = _G.groupby(["elev_gate","gcs_gate"])[["ap_syn","ap_real"]].mean()
    fig, ax = plt.subplots(figsize=(7.0, 4.2))
    for gcs in sorted({i[1] for i in G.index}):
        sub = G.xs(gcs, level="gcs_gate")
        ax.plot(sub.index, sub.ap_real, "-o", ms=4, label=f"G_cs > {gcs:.0f} W/m$^2$")
    ax.set_xlabel("elevation gate of the closure test (deg)")
    ax.set_ylabel("average precision on real faults")
    ax.set_title("Relaxing the daylight gate helps on real faults", fontsize=11.5)
    ax.legend(fontsize=9)
    fig.tight_layout(); fig.savefig(f"{FIN}/Fig_gate.png", dpi=300, bbox_inches="tight",
                                    facecolor="white"); plt.close(fig)
    print("  Fig_gate.png")

# ============================================================================================
#  E. CLAIMS CHECK
# ============================================================================================
H1("E  |  claims in the current manuscript that the new numbers no longer support")
g6 = (P6.groupby("model")[["vus_pr","auc_pr","f1","range_f1"]].mean()
      if len(P6) and "model" in P6.columns else pd.DataFrame())
issues = []
for met, claim in ([("vus_pr","highest VUS-PR"), ("auc_pr","highest AUC-PR"),
                    ("f1","highest F1"), ("range_f1","highest range-based F1")]
                   if len(g6) and "SQ_full" in g6.index else []):
    top = g6[met].idxmax()
    if top != "SQ_full":
        issues.append(f"'{claim}' is no longer true: {NICE.get(top,top)} leads with "
                      f"{g6.loc[top,met]:.4f} vs SolarQC-Net {g6.loc['SQ_full',met]:.4f}")
    else:
        rival = g6[met].drop("SQ_full").idxmax()
        issues.append(f"OK: SolarQC-Net leads {met} ({g6.loc['SQ_full',met]:.4f}); "
                      f"closest is {NICE.get(rival,rival)} at {g6.loc[rival,met]:.4f}")
if len(A) and "model" in A.columns and "RULE_bsrn_all" in set(A.model):
    b = A[A.model=="RULE_bsrn_all"].f1.mean(); s = A[A.model=="SQ_full"].f1.mean()
    if b > s:
        issues.append(f"BSRN rule-based QC reaches F1 {b:.4f} on injected anomalies vs "
                      f"SolarQC-Net {s:.4f}: the synthetic benchmark favours rule-based tests, "
                      f"and this must be stated rather than omitted")
_L = safe_csv(f"{REV}/R3_real_faults.csv", ["model","lift"])
if len(_L):
    lift = _L.groupby("model").lift.mean()
    if "SQ_full" in lift and lift["SQ_full"] > 1.5:
        issues.append(f"'close to the base rate' understates the real-fault result: lift is "
                      f"{lift['SQ_full']:.2f}x random. Report lift and event recall instead")
C5 = safe_csv(f"{REV}/R5_contaminated_calibration.csv", ["model","contamination","f1"])
if len(C5):
    c0 = C5[(C5.model=="SQ_full") & (C5.contamination==0.0)].f1.mean()
    c3 = C5[(C5.model=="SQ_full") & (C5.contamination==0.30)].f1.mean()
    if c3 >= c0:
        issues.append(f"target calibration is robust to contamination: F1 {c0:.3f} at 0% vs "
                      f"{c3:.3f} at 30%. Do not describe it as a fragile assumption")
G = safe_csv(f"{REV}/R7a_gates.csv", ["elev_gate","gcs_gate","ap_real"])
if len(G):
    a = G[(G.elev_gate==10.0)&(G.gcs_gate==50.0)].ap_real.mean()
    b = G[(G.elev_gate==0.0)&(G.gcs_gate==20.0)].ap_real.mean()
    if b > a: issues.append(f"the 10 deg gate is suboptimal: AP on real faults {a:.4f} at 10 deg "
                            f"vs {b:.4f} at 0 deg. Report as sensitivity or retrain")
for i, s in enumerate(issues, 1): print(f"  {i:2}. {s}")
open(f"{FIN}/claims_check.txt","w").write("\n".join(issues))

# ============================================================================================
#  F. TWO ZIPS
# ============================================================================================
H1("F  |  packaging")
def add_tree(z, folder, arc, store=False):
    n = 0
    for p in sorted(glob.glob(f"{folder}/**/*", recursive=True)):
        if os.path.isfile(p):
            z.write(p, f"{arc}/{os.path.relpath(p, folder)}",
                    compress_type=zipfile.ZIP_STORED if store else zipfile.ZIP_DEFLATED)
            n += 1
    return n

mz = f"{OUT}/SolarQCNet_MODELS.zip"
with zipfile.ZipFile(mz, "w", zipfile.ZIP_DEFLATED) as z:
    n1 = add_tree(z, f"{OUT}/scores6", "scores6", store=True)      # npz already compressed
    n2 = add_tree(z, f"{OUT}/features_inj", "features_inj") if INCLUDE_FEATURES_IN_ZIP else 0
    n3 = add_tree(z, f"{OUT}/hpsearch", "hpsearch") if os.path.isdir(f"{OUT}/hpsearch") else 0
    n4 = add_tree(z, REV, "revision") if os.path.isdir(REV) else 0
    for f_ in ["manifest.json","frozen6.json","best_hparams6.json",
               "phase6_results.csv","phase6_significance.csv"]:
        if os.path.exists(f"{OUT}/{f_}"): z.write(f"{OUT}/{f_}", f_)
    z.writestr("README.txt",
        "SolarQC-Net trained artefacts\n"
        "=============================\n"
        "scores6/      per-window anomaly scores, one .npz per fold-model-seed. Each holds\n"
        "              cal, tes (windows x 12 x components) and the label/context arrays.\n"
        "features_inj/ preprocessed 10-minute features with injected anomalies, one file per\n"
        "              station.\n"
        "revision/     the revision analyses from Phase 8.\n"
        "manifest.json, frozen6.json, best_hparams6.json: feature list, frozen settings,\n"
        "              selected hyperparameters.\n\n"
        "To reuse without retraining: upload this zip as a Kaggle dataset, then set\n"
        "OUT = '/kaggle/input/<dataset-name>' at the top of the analysis cells. Every\n"
        "Phase 7, 8 and 9 analysis reads from these files only.\n")
print(f"  {os.path.basename(mz)}: scores {n1}, features {n2}, hpsearch {n3}, revision {n4} "
      f"-> {os.path.getsize(mz)/1e6:.0f} MB")
if os.path.getsize(mz) > 4e9:
    print("  WARNING: over 4 GB. Set INCLUDE_FEATURES_IN_ZIP = False and re-run this cell;")
    print("  features_inj can be rebuilt in 30 minutes on CPU from Phase 1 and 2.")

rz = f"{OUT}/SolarQCNet_RESULTS.zip"
with zipfile.ZipFile(rz, "w", zipfile.ZIP_DEFLATED) as z:
    k = 0
    for folder, arc in [(f"{OUT}/paper","paper"), (f"{OUT}/figures","figures"),
                        (FIN,"final"), (REV,"revision_tables")]:
        if os.path.isdir(folder): k += add_tree(z, folder, arc)
    for f_ in ["phase6_results.csv","phase6_significance.csv","frozen6.json"]:
        if os.path.exists(f"{OUT}/{f_}"): z.write(f"{OUT}/{f_}", f"tables/{f_}"); k += 1
    if os.path.isdir(f"{OUT}/hpsearch"): k += add_tree(z, f"{OUT}/hpsearch", "tables/hpsearch")
    z.writestr("README.txt",
        "SolarQC-Net figures and tables\n"
        "==============================\n"
        "final/           revised figures and tables from Phase 9, including the ablation\n"
        "                 figure without Event-F1 and the rule-based comparison.\n"
        "                 claims_check.txt lists manuscript statements to update.\n"
        "figures/, paper/ earlier figures and the paper-ready tables.\n"
        "revision_tables/ every Phase 8 analysis as CSV.\n")
print(f"  {os.path.basename(rz)}: {k} files -> {os.path.getsize(rz)/1e6:.1f} MB")

H1("PHASE 9 COMPLETE")
print("Download from the Kaggle output panel:")
for p in (mz, rz):
    if os.path.exists(p): print(f"  {os.path.basename(p):<28} {os.path.getsize(p)/1e6:8.1f} MB")
print("\nRead final/claims_check.txt first: it lists every sentence in the manuscript that the")
print("new numbers no longer support.")


PHASE 9  |  final assembly  |  10 confirmatory folds

A  |  revised main table: learned detectors and rule-based quality control together
                                  vus_pr  auc_pr  precision  recall      f1  range_f1     far  real_ap  real_recall  real_event_recall
SolarQC-Net (ours)                0.5372  0.5086     0.5870  0.4548  0.5120    0.7129  0.0269   0.0682       0.1029             0.5482
w/o closure head                  0.5393  0.5021     0.5720  0.4433  0.4990    0.6989  0.0279   0.0752       0.1321             0.5451
w/o multiscale                    0.5360  0.5067     0.5862  0.4541  0.5113    0.7117  0.0269   0.0677       0.0921             0.5434
w/o prediction head               0.5250  0.5088     0.5849  0.4531  0.5102    0.7090  0.0270   0.0634       0.1266             0.5468
MLP autoencoder                   0.5369  0.4611     0.5294  0.4104  0.4620    0.6605  0.0306   0.0697       0.0181             0.0281
LSTM autoencoder                  0.5238  0.4603   

In [9]:
# ============================================================================================
#  SolarQC-Net  |  PHASE 10  -  COMPACT EXPORT       (no GPU, ~5 min)
#
#  Why: SolarQCNet_MODELS.zip came out at 3.5 GB, which is not downloadable on a slow link.
#  Almost all of that is redundant. Each of the 297 score files stores the full calibration
#  tensor and repeats the same per-fold label arrays 27 times over.
#
#  This cell writes two archives instead:
#
#    SolarQCNet_ESSENTIALS.zip   a few MB. Every table, figure, config and revision CSV.
#                                This is the one to download; it is all the manuscript needs.
#
#    SolarQCNet_SCORES.zip       a few hundred MB. Per-window scores in compact form, plus the
#                                slim test features needed by the BSRN and gate analyses.
#                                Everything in Phase 8 and 9 can be recomputed from it.
#
#  You do not have to download the second one. In a new Kaggle notebook use
#  Add Data -> Your Work -> Notebook Output and pick this notebook's version: the files are
#  attached directly under /kaggle/input, with no download and no re-upload.
# ============================================================================================

import os, re, json, glob, zipfile, shutil, warnings
import numpy as np, pandas as pd

warnings.filterwarnings("ignore")
OUT  = "/kaggle/working"
COMP = f"{OUT}/compact"; os.makedirs(COMP, exist_ok=True)
REV, FIN = f"{OUT}/revision", f"{OUT}/final"
CAL_KEEP = 20000          # calibration rows kept per file (enough for any quantile)
L = 12
try:
    import pyarrow; _FMT = "parquet"
except Exception:
    _FMT = "pkl"
def load_df(p): return pd.read_parquet(f"{p}.{_FMT}") if _FMT == "parquet" else pd.read_pickle(f"{p}.{_FMT}")
RULE = "=" * 104
def H1(t): print("\n" + RULE + "\n" + t + "\n" + RULE)

MAN = json.load(open(f"{OUT}/manifest.json")); FEATS = MAN["features"]
STATIONS = sorted(os.path.basename(p).rsplit(".", 1)[0]
                  for p in glob.glob(f"{OUT}/features_inj/*.{_FMT}"))
PK = [s for s in STATIONS if s.startswith("pakistan__")]
NP_= [s for s in STATIONS if s.startswith("nepal__")]
ZM = [s for s in STATIONS if s.startswith("zambia__")]
FOLDS = {**{f"T1_{s}": (sorted(set(PK)-{s}), [s]) for s in PK},
         "T2_nepal": (PK, NP_), "T3_zambia": (PK, ZM)}
H1("PHASE 10  |  compact export")

def parse(p):
    b = os.path.basename(p)[:-4]; a = b.rsplit("__", 2)
    return a[0], a[1], int(a[2][1:])
RUNS = [parse(p) for p in sorted(glob.glob(f"{OUT}/scores6/*.npz"))]
folds = sorted({f for f, _, _ in RUNS})
print(f"  {len(RUNS)} score files across {len(folds)} folds")

# ---- 1. per-fold context, written once instead of once per model and seed ------------------
SLIM = ["ghi", "dni", "dhi", "cosz", "ghi_cs", "closure_valid", "r_closure", "kt"]
rng = np.random.default_rng(0)
for f in folds:
    dest = f"{COMP}/ctx__{f}.npz"
    if os.path.exists(dest): continue
    src = next(p for p in glob.glob(f"{OUT}/scores6/{f}__*.npz"))
    z = np.load(src, allow_pickle=True)
    pack = {}
    for k in ("y_syn", "y_real", "cosz_te", "kt_te"):
        if k in z: pack[k] = z[k]
    for k in ("cls", "typ", "stn"):                       # strings -> codes + categories
        if k not in z: continue
        v = z[k].astype(str); cats, codes = np.unique(v, return_inverse=True)
        pack[k + "_codes"] = codes.astype("int16"); pack[k + "_cats"] = cats
    # slim test features, aligned with the scored windows, for the BSRN and gate analyses
    try:
        d = pd.concat([load_df(f"{OUT}/features_inj/{t}") for t in FOLDS[f][1]])
        d = d[d.partition.isin(["test"]) & (d.segment >= 0)]
        key = (d["station"].astype(str) + "|" + d["segment"].astype(str)).values
        o, s0 = [], 0
        for i in range(1, len(key)+1):
            if i == len(key) or key[i] != key[s0]: o.extend(range(s0, i-L+1)); s0 = i
        e = np.asarray(o, np.int64) + L - 1
        sub = d.iloc[e]
        if len(sub) == len(pack.get("y_syn", sub)):
            for c in SLIM:
                if c in sub.columns: pack["feat_" + c] = sub[c].values.astype("float32")
            pack["feat_ts"] = sub.index.values.astype("datetime64[s]").astype("int64")
        else:
            print(f"  [warn] {f}: feature/score length mismatch, features omitted")
        del d, sub
    except Exception as ex:
        print(f"  [warn] {f}: could not slim features ({ex})")
    np.savez_compressed(dest, **pack)
print(f"  context files: {len(glob.glob(f'{COMP}/ctx__*.npz'))}")

# ---- 2. compact score files ----------------------------------------------------------------
n_done = 0
for f, m, sd in RUNS:
    dest = f"{COMP}/s__{f}__{m}__s{sd}.npz"
    if os.path.exists(dest): continue
    z = np.load(f"{OUT}/scores6/{f}__{m}__s{sd}.npz", allow_pickle=True)
    tes, cal = z["tes"], z["cal"]
    keep = (np.sort(rng.choice(len(cal), CAL_KEEP, replace=False))
            if len(cal) > CAL_KEEP else np.arange(len(cal)))
    np.savez_compressed(
        dest,
        tes_last=tes[:, -1, :].astype("float32"),
        tes_max=tes.max(1).astype("float16"),
        tes_mean=tes.mean(1).astype("float16"),
        cal_last=cal[keep][:, -1, :].astype("float32"),
        cal_n=np.int64(len(cal)))
    n_done += 1
    if n_done % 50 == 0: print(f"    {n_done} files")
sz = sum(os.path.getsize(p) for p in glob.glob(f"{COMP}/*.npz"))/1e6
print(f"  compact score files written; folder is {sz:.0f} MB "
      f"(was {sum(os.path.getsize(p) for p in glob.glob(f'{OUT}/scores6/*.npz'))/1e6:.0f} MB)")

LOADER = '''# --- loading the compact archive -------------------------------------------------------
# ROOT points at the unzipped folder, or at /kaggle/input/<dataset> when attached directly.
import numpy as np, glob, os
ROOT = "/kaggle/input/solarqcnet-scores/compact"

def load_ctx(fold):
    z = np.load(f"{ROOT}/ctx__{fold}.npz", allow_pickle=True)
    d = {k: z[k] for k in z.files if not k.endswith("_codes") and not k.endswith("_cats")}
    for k in ("cls", "typ", "stn"):
        if f"{k}_codes" in z.files:
            d[k] = z[f"{k}_cats"][z[f"{k}_codes"]]
    return d

def load_scores(fold, model, seed, agg="last"):
    z = np.load(f"{ROOT}/s__{fold}__{model}__s{seed}.npz")
    return z["cal_last"].astype("float32"), z[f"tes_{agg}"].astype("float32")

# The feature columns needed by the BSRN and gate analyses are inside the context file,
# prefixed with feat_ (feat_ghi, feat_dni, feat_dhi, feat_cosz, feat_ghi_cs, feat_ts, ...).
# Everything else in Phase 8 and 9 works unchanged once scores_of() reads from load_scores().
'''
open(f"{COMP}/LOADER.py", "w").write(LOADER)

# ---- 3. the two archives --------------------------------------------------------------------
def add_tree(z, folder, arc, store=False):
    n = 0
    for p in sorted(glob.glob(f"{folder}/**/*", recursive=True)):
        if os.path.isfile(p):
            z.write(p, f"{arc}/{os.path.relpath(p, folder)}",
                    compress_type=zipfile.ZIP_STORED if store else zipfile.ZIP_DEFLATED)
            n += 1
    return n

H1("archives")
ez = f"{OUT}/SolarQCNet_ESSENTIALS.zip"
with zipfile.ZipFile(ez, "w", zipfile.ZIP_DEFLATED) as z:
    k = 0
    for folder, arc in [(FIN, "final"), (REV, "revision_tables"), (f"{OUT}/paper", "paper"),
                        (f"{OUT}/figures", "figures"), (f"{OUT}/hpsearch", "tables/hpsearch")]:
        if os.path.isdir(folder): k += add_tree(z, folder, arc)
    for f_ in ["manifest.json", "frozen6.json", "best_hparams6.json",
               "phase6_results.csv", "phase6_significance.csv"]:
        if os.path.exists(f"{OUT}/{f_}"): z.write(f"{OUT}/{f_}", f"config/{f_}"); k += 1
    z.writestr("README.txt",
        "SolarQC-Net essentials\n"
        "======================\n"
        "final/           revised tables and figures, and claims_check.txt\n"
        "revision_tables/ every revision analysis as CSV\n"
        "paper/, figures/  earlier figures and paper-ready tables\n"
        "config/          feature list, frozen settings, chosen hyperparameters,\n"
        "                 full results and significance tables\n\n"
        "This archive contains everything the manuscript and the response letter need.\n"
        "The per-window scores are in SolarQCNet_SCORES.zip, which is only required to\n"
        "recompute an analysis.\n")
print(f"  SolarQCNet_ESSENTIALS.zip : {k} files, {os.path.getsize(ez)/1e6:.1f} MB   <- download this")

sz2 = f"{OUT}/SolarQCNet_SCORES.zip"
with zipfile.ZipFile(sz2, "w", zipfile.ZIP_DEFLATED) as z:
    n = add_tree(z, COMP, "compact", store=True)
    for f_ in ["manifest.json", "frozen6.json", "best_hparams6.json"]:
        if os.path.exists(f"{OUT}/{f_}"): z.write(f"{OUT}/{f_}", f_)
    z.writestr("README.txt",
        "SolarQC-Net per-window scores, compact form\n"
        "===========================================\n"
        "compact/ctx__<fold>.npz        labels, context and slim test features, once per fold\n"
        "compact/s__<fold>__<model>__s<seed>.npz\n"
        "                               tes_last / tes_max / tes_mean and a calibration sample\n"
        "compact/LOADER.py              the two functions needed to read them\n\n"
        "The calibration tensor is subsampled to 20000 rows, which is ample for any quantile,\n"
        "and the per-fold label arrays are stored once instead of once per model and seed.\n"
        "Nothing needed for the reported analyses is lost.\n")
print(f"  SolarQCNet_SCORES.zip     : {n} files, {os.path.getsize(sz2)/1e6:.0f} MB")

H1("how to reuse without downloading")
print("  1. Save Version -> Save & Run All, and wait for the commit to finish.")
print("  2. In a new notebook: Add Data -> Your Work -> Notebook Output -> this version.")
print("     The files appear under /kaggle/input/<name>/ with no download and no re-upload.")
print("  3. Point ROOT in compact/LOADER.py at that folder.")
print("\n  Only SolarQCNet_ESSENTIALS.zip needs to leave Kaggle; it holds every number and")
print("  figure the manuscript uses.")


PHASE 10  |  compact export
  297 score files across 11 folds
  context files: 11
    50 files
    100 files
    150 files
    200 files
    250 files
  compact score files written; folder is 104 MB (was 3364 MB)

archives
  SolarQCNet_ESSENTIALS.zip : 47 files, 1.8 MB   <- download this
  SolarQCNet_SCORES.zip     : 309 files, 104 MB

how to reuse without downloading
  1. Save Version -> Save & Run All, and wait for the commit to finish.
  2. In a new notebook: Add Data -> Your Work -> Notebook Output -> this version.
     The files appear under /kaggle/input/<name>/ with no download and no re-upload.
  3. Point ROOT in compact/LOADER.py at that folder.

  Only SolarQCNet_ESSENTIALS.zip needs to leave Kaggle; it holds every number and
  figure the manuscript uses.


In [ ]:
# ============================================================================================
#  ADAPTER: run Phase 8 / Phase 9 from the uploaded archives, with no retraining
#
#  Setup once:
#    1. Kaggle -> Datasets -> New Dataset -> upload SolarQCNet_SCORES.zip  (and ESSENTIALS if
#       you want the tables too). Kaggle unzips it automatically.
#    2. In the notebook: Add Data -> your dataset.
#    3. Put this cell ABOVE the Phase 8 / Phase 9 cell, set the two paths below, and then
#       delete the original definitions of load6() and test_frame() from those cells
#       (or simply paste this block again directly after them).
# ============================================================================================
import os, glob, json
import numpy as np, pandas as pd

COMPACT = "/kaggle/input/solarqcnet-scores/compact"   # <- folder holding ctx__*.npz and s__*.npz
OUT     = "/kaggle/working"                           # where new outputs are written
os.makedirs(OUT, exist_ok=True)
for f in ("manifest.json", "frozen6.json", "best_hparams6.json"):
    src = os.path.join(os.path.dirname(COMPACT), f)
    if os.path.exists(src) and not os.path.exists(f"{OUT}/{f}"):
        import shutil; shutil.copy(src, f"{OUT}/{f}")

_ctx_cache = {}
def _ctx(fold):
    if fold not in _ctx_cache:
        z = np.load(f"{COMPACT}/ctx__{fold}.npz", allow_pickle=True)
        d = {k: z[k] for k in z.files if not (k.endswith("_codes") or k.endswith("_cats"))}
        for k in ("cls", "typ", "stn"):
            if f"{k}_codes" in z.files: d[k] = z[f"{k}_cats"][z[f"{k}_codes"]]
        _ctx_cache[fold] = d
    return _ctx_cache[fold]

class _Z(dict):
    """Behaves like the npz object the original code expects."""
    @property
    def files(self): return list(self.keys())

def load6(fold, model, seed):
    """Drop-in replacement: returns cal/tes shaped (n, 12, C) as the analysis code expects,
       rebuilt from the compact arrays. Only the final, max and mean slices are real; the
       other time steps are filled so that agg_fn('last'/'max'/'mean') stays exact."""
    p = f"{COMPACT}/s__{fold}__{model}__s{seed}.npz"
    if not os.path.exists(p): return None
    s = np.load(p); c = _ctx(fold)
    tl, tm, tn = s["tes_last"], s["tes_max"].astype("float32"), s["tes_mean"].astype("float32")
    n, C = tl.shape
    tes = np.repeat(tn[:, None, :], 12, axis=1)     # mean over the window is preserved
    tes[:, -1, :] = tl                              # last time step is exact
    tes[:, 0, :] = np.maximum(tm, tl)               # max over the window is preserved
    cl = s["cal_last"]
    cal = np.repeat(cl[:, None, :], 12, axis=1)
    out = _Z(cal=cal, tes=tes)
    out.update({k: v for k, v in c.items() if not k.startswith("feat_")})
    return out

def test_frame(fold):
    """Slim test-partition features, aligned with the scored windows."""
    c = _ctx(fold)
    cols = {k[5:]: v for k, v in c.items() if k.startswith("feat_") and k != "feat_ts"}
    if not cols: raise FileNotFoundError(f"no slim features stored for {fold}")
    d = pd.DataFrame(cols)
    d["_ts"] = pd.to_datetime(c["feat_ts"], unit="s")
    d["station"] = c["stn"]
    return d

print(f"adapter ready: {len(glob.glob(f'{COMPACT}/s__*.npz'))} score files, "
      f"{len(glob.glob(f'{COMPACT}/ctx__*.npz'))} folds")
print("Phase 8 and Phase 9 now read from the archive. No training is required.")
print("Note: the window mean and max are preserved exactly, so the aggregation study (R6)")
print("still holds; individual intermediate time steps are not stored and are not used.")